## TESIS:
### **"VALOR PREDICTIVO INCREMENTAL DE LAS VARIABLES CLIMÁTICAS PARA ANTICIPAR, A UN MES, CAÍDAS INTERANUALES REALES DE LA RECAUDACIÓN DE TRIBUTOS INTERNOS POR JURISDICCIÓN FISCAL EN EL PERÚ, 2005-2025"**

#### **PRESENTADO POR:**
#### - Bach. FABIÁN ALEXANDER HERRERA AGUAYO
#### - Bach. BERNABÉ GONZÁLES GARCÍA

### **MAESTRÍA EN CIENCIA DE DATOS**
### **UNIVERSIDAD CONTINENTAL**


## Paso 01 — Módulo de auditoría y trazabilidad

Define las herramientas comunes de **auditoría y trazabilidad** que usarán todas las celdas del notebook:

- Crea una carpeta única de ejecución (`00_AUDITORIA/<RUN_ID>/`) donde se almacena todo lo relativo a esta corrida.
- Prepara los archivos de registro: `ejecucion.log`, `hashes.json`, `errores.json`, `manifiesto.json` y la carpeta `flags/`.
- Define las funciones compartidas `log()`, `sha256_file()`, `sha256_dataframe()`, `registrar_hash()`, `crear_flag()`, `registrar_error()`, `auditar()`, `resumen_entorno()` y `escribir_manifiesto()`.
- Inicializa la semilla y la hora de arranque.

**No procesa datos.** Solo deja lista la infraestructura de auditoría que garantiza que cada paso posterior quede **fechado, hasheado y verificable**.

In [1]:
# ==============================================================================
# 01: MÓDULO DE AUDITORÍA Y TRAZABILIDAD
# ==============================================================================
# Define las utilidades que todas las celdas usarán para documenta su ejecución: flags, logs JSON, hashes, resumen por celda.
# No ejecuta ningún proceso de datos.

import os, sys, time, json, hashlib, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from functools import wraps

# Configuración global
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
WORK_ROOT = Path(os.environ.get("TESIS_WORK_ROOT", "/content"))
BASE_DIR = WORK_ROOT / "datos_tesis" / RUN_ID
OUTPUT_DIR = WORK_ROOT / "resultados_tesis" / RUN_ID
BASE_DIR.mkdir(parents=True, exist_ok=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

# Variables que deben fijarse antes de importar bibliotecas numéricas.
for _k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS"]:
    os.environ[_k] = "1"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
HASHSEED_AL_INICIO = os.environ.get("PYTHONHASHSEED", "NO_FIJADO_AL_INICIO")
AUDIT_DIR = OUTPUT_DIR / "00_AUDITORIA" / RUN_ID
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Archivos de auditoría
LOG_FILE      = AUDIT_DIR / "ejecucion.log"          # Log legible línea por línea
FLAGS_DIR     = AUDIT_DIR / "flags"                  # Flags por paso
HASHES_FILE   = AUDIT_DIR / "hashes.json"            # Hashes de cada artefacto
ERRORS_FILE   = AUDIT_DIR / "errores.json"           # Errores (si hay)
MANIFEST_FILE = AUDIT_DIR / "manifiesto.json"        # Manifiesto global
FLAGS_DIR.mkdir(exist_ok=True)

# Estado en memoria
_HASHES = {}
_ERRORS = []
_START_TIME = time.time()

def _ahora():
    return datetime.now(timezone.utc).isoformat()

def log(mensaje, nivel="INFO", celda=None):
    # Escribe una línea al log con timestamp, nivel y celda de origen.
    prefijo = f"[{_ahora()}] [{nivel}]"
    if celda:
        prefijo += f" [{celda}]"
    linea = f"{prefijo} {mensaje}"
    print(linea)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(linea + "\n")

def sha256_file(path):
    # Hash SHA-256 de un archivo.
    if not path or not Path(path).exists():
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dataframe(df):
    # Hash SHA-256 canónico de un DataFrame (orden de columnas y filas).
    import pandas as pd
    contenido = df.to_csv(index=False, lineterminator='\n',
                          float_format='%.17g').encode('utf-8')
    return hashlib.sha256(contenido).hexdigest()

def registrar_hash(nombre, path=None, df=None, extra=None):
    # Registra el hash de un archivo o DataFrame con metadatos.
    entry = {
        "nombre": nombre,
        "timestamp": _ahora(),
        "tipo": "dataframe" if df is not None else "archivo",
        "sha256": sha256_dataframe(df) if df is not None else sha256_file(path),
    }
    if path is not None:
        entry["path"] = str(path)
        entry["bytes"] = Path(path).stat().st_size if Path(path).exists() else 0
    if df is not None:
        entry["filas"] = len(df)
        entry["columnas"] = len(df.columns)
    if extra:
        entry.update(extra)
    _HASHES[nombre] = entry
    log(f"HASH registrado: {nombre} = {(entry['sha256'] or 'N/A')[:16]}...", celda="HASH")
    with open(HASHES_FILE, "w", encoding="utf-8") as f:
        json.dump(_HASHES, f, ensure_ascii=False, indent=2)
    return entry["sha256"]

def crear_flag(paso, estado="OK", metadata=None):
    # Crea un flag .flag con nombre y contenido JSON. Sirve para que el jurado vea: 'el paso X terminó en estado Y'.
    flag_path = FLAGS_DIR / f"{paso}.{estado}.flag"
    contenido = {
        "paso": paso,
        "estado": estado,
        "timestamp": _ahora(),
        "run_id": RUN_ID,
        "metadata": metadata or {},
    }
    flag_path.write_text(json.dumps(contenido, ensure_ascii=False, indent=2),
                         encoding="utf-8")
    log(f"FLAG creado: {flag_path.name}", celda="FLAG")
    return flag_path

def registrar_error(paso, exception, contexto=None):
    # Registra un error con traceback completo en JSON.
    import traceback
    entry = {
        "paso": paso,
        "timestamp": _ahora(),
        "tipo": type(exception).__name__,
        "mensaje": str(exception),
        "traceback": traceback.format_exc(),
        "contexto": contexto or {},
    }
    _ERRORS.append(entry)
    with open(ERRORS_FILE, "w", encoding="utf-8") as f:
        json.dump(_ERRORS, f, ensure_ascii=False, indent=2)
    log(f"ERROR en {paso}: {entry['tipo']} — {entry['mensaje'][:100]}",
        nivel="ERROR", celda=paso)

def auditar(celda_nombre, guardar_output=None):
    # Decorador para auditar una función completa:
    # - Log al inicio y fin
    # - Tiempo de ejecución
    # - Flag OK o ERROR
    # - Hash del output (si se indica)

    def decorador(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            t0 = time.time()
            log(f"INICIO {func.__name__}", celda=celda_nombre)
            try:
                resultado = func(*args, **kwargs)
                duracion = time.time() - t0
                metadata = {"duracion_seg": round(duracion, 2)}
                if guardar_output:
                    out_name, out_val = guardar_output(resultado)
                    if hasattr(out_val, "to_csv"):  # DataFrame
                        registrar_hash(out_name, df=out_val)
                    else:
                        registrar_hash(out_name, path=out_val)
                crear_flag(celda_nombre, estado="OK", metadata=metadata)
                log(f"FIN {func.__name__} — {duracion:.2f}s", celda=celda_nombre)
                return resultado
            except Exception as e:
                registrar_error(celda_nombre, e)
                crear_flag(celda_nombre, estado="ERROR")
                raise
        return wrapper
    return decorador

def resumen_entorno():
    #Captura versiones y plataforma para el manifiesto.
    paquetes = ['numpy','pandas','scikit-learn','xgboost','tensorflow',
                'geopandas','xarray','statsmodels','matplotlib','openpyxl']
    versiones = {}
    for p in paquetes:
        try:
            versiones[p] = importlib.metadata.version(p)
        except Exception:
            versiones[p] = "NO_DISPONIBLE"
    return {
        "python": sys.version,
        "plataforma": platform.platform(),
        "run_id": RUN_ID,
        "inicio_utc": _ahora(),
        "paquetes": versiones,
        "cpu": platform.processor(), "n_cpu": os.cpu_count(),
        "hashseed_inicio": HASHSEED_AL_INICIO,
        "hilos": {k: os.environ.get(k) for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]},
        "gpu": str(sys.modules["tensorflow"].config.list_physical_devices("GPU")) if "tensorflow" in sys.modules else "NO_CARGADO",
    }

def escribir_manifiesto(secciones_extra=None):
    #Escribe el manifiesto final con entorno, hashes, flags, errores y log.
    entorno = resumen_entorno()
    flags = sorted([f.name for f in FLAGS_DIR.glob("*.flag")])
    manifiesto = {
        "run_id": RUN_ID,
        "entorno": entorno,
        "hashes_artefactos": _HASHES,
        "flags": flags,
        "errores": _ERRORS,
        "log_sha256": sha256_file(LOG_FILE),
        "duracion_total_seg": round(time.time() - _START_TIME, 2),
        "secciones_extra": secciones_extra or {},
    }
    MANIFEST_FILE.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2),
                              encoding="utf-8")
     # El manifiesto toma el hash del log ya cerrado; no modificarlo aquí.
    return manifiesto

# Log de arranque
log(f"RUN_ID = {RUN_ID}", celda="INIT")
log(f"AUDIT_DIR = {AUDIT_DIR}", celda="INIT")
log(f"Entorno capturado", celda="INIT")

[2026-10-30T00:29:00.118006+00:00] [INFO] [INIT] RUN_ID = 20261030T002900_112193Z
[2026-10-30T00:29:00.118760+00:00] [INFO] [INIT] AUDIT_DIR = /content/resultados_tesis/20261030T002900_112193Z/00_AUDITORIA/20261030T002900_112193Z
[2026-10-30T00:29:00.118969+00:00] [INFO] [INIT] Entorno capturado


## Paso 02 — Entorno Colab y carga del ZIP

Ejecuta con CPU o GPU. Se conserva el entorno numérico preinstalado de Colab y se instalan solo bibliotecas ausentes; las versiones efectivas se exportan. Si modificaste paquetes antes de abrir este notebook y aparecen errores de importación, elimina el entorno anterior y comienza desde el Paso 01.

El ZIP se identifica por contenido (SHA-256), por lo que `DataOrigen.zip`, `DataOrigen(1).zip` y `DataOrigen(2).zip` son válidos si contienen la fuente. Si no está en `/content`, aparecerá el selector para subirlo. El ZIP de resultados no se acepta como fuente.

Las rutas explícitas opcionales son `TESIS_ZIP_FUENTES`, `TESIS_ZIP_REFERENCIA` y `TESIS_NOTEBOOK`. Cada corrida usa una carpeta exclusiva. La búsqueda completa conserva 447 candidatos y puede tardar; el Paso 08 muestra el avance individual.


In [3]:
# ==============================================================================
# 02: INSTALACIÓN, CARGA AUTOMÁTICA Y CONFIGURACIÓN GLOBAL
# ==============================================================================
try:
  import subprocess, sys, importlib.util
  from packaging.version import Version
  # Compatibilidad Colab: conservar su conjunto numérico preinstalado.
  # Instalar únicamente dependencias ausentes; no sustituir IPython/NumPy/TensorFlow.
  # El entorno efectivo queda congelado y se verifica mediante importación real.
  REQUERIDAS = {
      'numpy': ('numpy','2.3.5'), 'pandas': ('pandas','2.2.3'),
      'scikit-learn': ('sklearn','1.8.0'), 'tensorflow': ('tensorflow','2.21.0'),
      'geopandas': ('geopandas','1.2.0'), 'eccodes': ('eccodes','2.50.0'),
      'openpyxl': ('openpyxl','3.1.5'), 'xgboost': ('xgboost','3.4.1'),
      'statsmodels': ('statsmodels','0.15.0'), 'nbformat': ('nbformat','5.11.1'),
      'matplotlib': ('matplotlib','3.10.8'), 'seaborn': ('seaborn','0.13.2'),
      'joblib': ('joblib','1.5.3'), 'threadpoolctl': ('threadpoolctl','3.6.0'),
      'pyproj': ('pyproj','3.8.0'), 'shapely': ('shapely','2.2.0'),
      'pyogrio': ('pyogrio','0.13.0'), 'ipython': ('IPython','9.17.1')}
  presentes = {}
  faltantes = []
  for paquete,(modulo,version_respaldo) in REQUERIDAS.items():
      try: presentes[paquete] = importlib.metadata.version(paquete)
      except importlib.metadata.PackageNotFoundError:
          faltantes.append(f'{paquete}=={version_respaldo}')

  # Las restricciones impiden que instalar un lector GRIB actualice el núcleo de Colab.
  restricciones = OUTPUT_DIR / 'Restricciones_Entorno_Inicial.txt'
  restricciones.write_text('\n'.join(f'{p}=={v}' for p,v in presentes.items())+'\n')
  if faltantes:
      if os.environ.get('TESIS_OMITIR_INSTALACION') == '1':
          raise RuntimeError(f'Dependencias ausentes: {faltantes}')
      subprocess.run([sys.executable,'-m','pip','install','-q','-c',str(restricciones),*faltantes],check=True)
      importlib.invalidate_caches()
  for paquete,(modulo,_) in REQUERIDAS.items():
      try: importlib.import_module(modulo)
      except Exception as exc:
          raise RuntimeError(f'No se pudo importar {paquete}. Si el entorno fue modificado antes, '
              'desconectar y eliminar el entorno de ejecución y ejecutar desde Paso 01.') from exc

  # Contratos mínimos de API usados por el notebook; no certifican versiones futuras.
  for paquete,minimo in [('numpy','1.26'),('pandas','2.2'),('scikit-learn','1.5'),
                         ('tensorflow','2.19'),('matplotlib','3.8')]:
      if Version(importlib.metadata.version(paquete)) < Version(minimo):
          raise RuntimeError(f'{paquete} requiere >= {minimo}; seleccione un entorno Colab reciente o 2026.07.')
  LIST_DEPENDENCIAS = [f'{p}=={importlib.metadata.version(p)}' for p in REQUERIDAS]
  (OUTPUT_DIR / 'requirements_list.txt').write_text('\n'.join(LIST_DEPENDENCIAS)+'\n')
  (OUTPUT_DIR / 'requirements_entorno_completo.txt').write_text(
      subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
  print('Entorno efectivo registrado; no se sustituyeron bibliotecas ya instaladas.')

  import os
  import zipfile
  import warnings
  import unicodedata
  import hashlib
  import random
  from pathlib import Path
  from IPython.display import display, Markdown
  import numpy as np
  import pandas as pd
  import geopandas as gpd
  import matplotlib.pyplot as plt
  import seaborn as sns

  # Mantener visibles las advertencias que puedan afectar al análisis.
  warnings.simplefilter("default")

  # Ocultar únicamente el aviso de obsolescencia generado por Jupyter.
  warnings.filterwarnings(
      "ignore",
      message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
      category=DeprecationWarning,
      module=r"jupyter_client\.session",
  )

  # Semillas de reproducibilidad
  SEED_MODELADO = 20260918
  SEED_BOOTSTRAP = 20260919

  # PYTHONHASHSEED no se modifica retroactivamente.
  # Registrar la observación en auditoría sin mostrarla en pantalla.
  if HASHSEED_AL_INICIO != str(SEED_MODELADO):
      with open(LOG_FILE, "a", encoding="utf-8") as archivo_log:
          archivo_log.write(
              "[PASO 02][REPRODUCIBILIDAD] "
              f"PYTHONHASHSEED registrado al inicio: {HASHSEED_AL_INICIO}. "
              f"Valor esperado: {SEED_MODELADO}. "
              "La semilla del hash del proceso actual no se modifica "
              "retroactivamente.\n"
          )

  random.seed(SEED_MODELADO)
  np.random.seed(SEED_MODELADO)

  # Rutas explícitas, extracción aislada y protección contra traversal.
  HASH_ZIP_FUENTES = 'ed5a55d08b6f9105827285d292492d1d7f825fb193025bfceec93025f10a78b2'
  RUTA_ZIP_FUENTES = os.environ.get('TESIS_ZIP_FUENTES','').strip()
  ZIP_REFERENCIA = Path(os.environ.get('TESIS_ZIP_REFERENCIA', '/content/Resultados_Tesis_Final_20261028T043209_299661Z.zip'))
  REG_NOTEBOOK = Path(os.environ.get('TESIS_NOTEBOOK','/content/Tesis.ipynb'))

  def seleccionar_zip_fuentes(ruta_explicita=''):
      if ruta_explicita:
          ruta = Path(ruta_explicita).expanduser()
          if not ruta.is_file(): raise FileNotFoundError(f'Ruta indicada inexistente: {ruta}')
          if sha256_file(ruta) != HASH_ZIP_FUENTES: raise ValueError('El ZIP indicado no coincide con la fuente auditada')
          return ruta
      candidatos = [p for p in WORK_ROOT.glob('*.zip') if p.is_file() and sha256_file(p)==HASH_ZIP_FUENTES]
      if len(candidatos)==1:
          print(f'Fuente identificada por SHA-256: {candidatos[0].name}')
          return candidatos[0]
      if len(candidatos)>1:
          print('Hay varias copias idénticas:', [str(p) for p in candidatos])
          return seleccionar_zip_fuentes(input('Escriba la ruta exacta de la copia que desea usar: ').strip())
      try: from google.colab import files
      except ImportError:
          raise FileNotFoundError('Defina TESIS_ZIP_FUENTES con la ruta del ZIP de fuentes')
      print('Seleccione DataOrigen.zip (también se aceptan sus copias con sufijos).')
      subidos = files.upload()
      candidatos = [Path(nombre).resolve() for nombre in subidos
                    if Path(nombre).is_file() and sha256_file(Path(nombre))==HASH_ZIP_FUENTES]
      if len(candidatos)!=1:
          raise ValueError('Seleccione exactamente un ZIP de fuentes con el hash auditado. No el ZIP de resultados.')
      return candidatos[0]
  ZIP_FUENTES = seleccionar_zip_fuentes(RUTA_ZIP_FUENTES)
  zip_path = ZIP_FUENTES
  if zip_path.resolve() == ZIP_REFERENCIA.resolve():
      raise ValueError("Fuentes y resultados de referencia deben ser archivos diferentes")
  def extraer_zip_seguro(ruta, destino):
      with zipfile.ZipFile(ruta) as z:
          for miembro in z.infolist():
              p = (destino / miembro.filename).resolve()
              if not p.is_relative_to(destino.resolve()):
                  raise ValueError(f"Ruta insegura: {miembro.filename}")
          nombres = [Path(x).name.lower() for x in z.namelist()]
          if "cuadroa13.xlsx" not in nombres or "predicciones_prueba_12_modelos.csv" in nombres:
              raise ValueError("El ZIP no cumple el contrato de fuentes")
          z.extractall(destino)
  if sha256_file(zip_path) != 'ed5a55d08b6f9105827285d292492d1d7f825fb193025bfceec93025f10a78b2':
      raise ValueError('ZIP de fuentes distinto del auditado: conciliar versión antes de ejecutar')
  extraer_zip_seguro(zip_path, BASE_DIR)

  # Utilidades
  def buscar_archivo(nombre_parcial, directorio=BASE_DIR):
      objetivo = nombre_parcial.lower()
      candidatos = sorted(r for r in directorio.rglob("*") if r.is_file()
                          and objetivo in r.name.lower() and not r.name.startswith("~$"))
      if len(candidatos) > 1:
          raise ValueError(f"Entrada ambigua: {nombre_parcial}: {candidatos}")
      return candidatos[0] if candidatos else None

  def calcular_hash_archivo(filepath):
      if not filepath or not Path(filepath).exists(): return "N/A"
      hasher = hashlib.sha256()
      with open(filepath, 'rb') as f:
          for chunk in iter(lambda: f.read(65536), b""):
              hasher.update(chunk)
      return hasher.hexdigest()

  def calcular_hash_dataframe(df):
      return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values).hexdigest()

  def normalizar_texto(texto):
      if pd.isna(texto): return ""
      texto = str(texto).strip().upper()
      return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

  def parsear_fecha_bcrp(val):
      if pd.isna(val): return pd.NaT
      if isinstance(val, (pd.Timestamp, np.datetime64)): return pd.to_datetime(val).replace(day=1)
      s = str(val).strip().lower()
      if not s or 'total' in s or 'fuente' in s: return pd.NaT
      digits = ''.join(filter(str.isdigit, s))
      if len(digits) == 6 and 1990 <= int(digits[:4]) <= 2030:
          return pd.Timestamp(year=int(digits[:4]), month=int(digits[4:6]), day=1)
      for k, m in {'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,'jul':7,'ago':8,'set':9,'sep':9,'oct':10,'nov':11,'dic':12}.items():
          if k in s:
              if len(digits) == 2: return pd.Timestamp(year=2000+int(digits), month=m, day=1)
              if len(digits) == 4: return pd.Timestamp(year=int(digits), month=m, day=1)
      try: return pd.to_datetime(s).replace(day=1)
      except: return pd.NaT

  JURISDICCIONES_26 = [
      "AMAZONAS", "ANCASH", "APURIMAC", "AREQUIPA", "AYACUCHO", "CAJAMARCA",
      "CALLAO", "CUSCO", "HUANCAVELICA", "HUANUCO", "ICA", "JUNIN",
      "LA LIBERTAD", "LAMBAYEQUE", "LIMA METROPOLITANA", "LIMA PROVINCIAS",
      "LORETO", "MADRE DE DIOS", "MOQUEGUA", "PASCO", "PIURA", "PUNO",
      "SAN MARTIN", "TACNA", "TUMBES", "UCAYALI"
  ]

  def homologar_nombre_jurisdiccion(nombre):
      nom = normalizar_texto(nombre)
      mapeo = {
          'PROV. CONST. DEL CALLAO': 'CALLAO', 'PROVINCIA CONSTITUCIONAL DEL CALLAO': 'CALLAO',
          'CALLAO': 'CALLAO', 'LIMA METROPOLITANA': 'LIMA METROPOLITANA', 'LIMA PROVINCIAS': 'LIMA PROVINCIAS',
          'OTROS DE LIMA': 'LIMA PROVINCIAS', 'RESTO DE LIMA': 'LIMA PROVINCIAS',
          'ANCASH': 'ANCASH', 'APURIMAC': 'APURIMAC', 'HUANUCO': 'HUANUCO', 'JUNIN': 'JUNIN', 'SAN MARTIN': 'SAN MARTIN'
      }
      return mapeo.get(nom, nom)

  print("\n[VERIFICACIÓN DE ARCHIVOS]")
  for req in ["CuadroA13", "TablaCalendario", "TablaIPC", "TablaInflacion", "TablaPBI", "TipoCambio", "ICEN", "ONI", "PROVINCIA.gpkg", "precipitacion_era5land.grib", "temperatura_era5land.grib"]:
      encontrado = buscar_archivo(req)
      if encontrado is None: raise FileNotFoundError(f"Fuente requerida: {req}")
      print(f" - {req:<30}: {'OK -> ' + encontrado.name if encontrado else 'NO ENCONTRADO'}")

  # Registrar el ZIP una sola vez, al cierre de la carga
  registrar_hash("DataOrigen.zip", path=zip_path)
  crear_flag("Celda02_Carga_ZIP", "OK", {"zip": str(zip_path)})

except Exception as e:
    registrar_error("Celda02_Carga_ZIP", e)
    raise

# Fechas reconstruidas a fin de mes, con zona horaria explícita.
def corte_mes(mes):
    d = pd.Timestamp(mes).tz_localize(None)
    return (d.to_period('M').to_timestamp() + pd.offsets.MonthEnd(0)
            + pd.Timedelta(hours=23,minutes=59,seconds=59)).tz_localize('America/Lima')

def disponibilidad_etiqueta(objetivo):
    # El objetivo necesita A13 e IPC del objetivo y del mismo mes un año antes.
    m = pd.Timestamp(objetivo)
    return max(corte_mes(m+pd.DateOffset(months=l)) for l in [2,1,-10,-11])

CORTE_TRAIN = corte_mes('2019-12-01')
CORTE_SELECCION = corte_mes('2022-12-01')

def mascara_admisible(df, operacion):
    bloques, corte = {'train':(['ENTRENAMIENTO'],CORTE_TRAIN),
        'valid':(['VALIDACION'],CORTE_SELECCION),
        'reajuste':(['ENTRENAMIENTO','VALIDACION'],CORTE_SELECCION)}[operacion]
    return df.bloque_temporal.isin(bloques) & (df.fecha_disponible_Y <= corte)

def validar_fuente(df, fecha, valores, grupos, nombre):
    if df.duplicated(grupos+[fecha]).any(): raise ValueError(f'{nombre}: claves duplicadas')
    if not np.isfinite(df[valores].to_numpy(float)).all(): raise ValueError(f'{nombre}: valores no finitos')
    for _, sub in df.groupby(grupos[0]) if grupos else [('serie',df)]:
        meses = pd.DatetimeIndex(sub[fecha].sort_values())
        if not meses.equals(pd.date_range('2005-01-01','2025-12-01',freq='MS')):
            raise ValueError(f'{nombre}: continuidad/cobertura mensual incompleta')

def validar_panel(df):
    claves=['jurisdiccion','origen_t','objetivo_t1']
    if df.duplicated(claves).any(): raise ValueError('Panel: claves duplicadas')
    if set(df.jurisdiccion)!=set(JURISDICCIONES_26): raise ValueError('Panel: catálogo territorial incorrecto')
    if df.Y_caida.isna().any() or not df.Y_caida.isin([0,1]).all(): raise ValueError('Etiqueta desconocida/no binaria')
    pred=[c for c in df if c.startswith(('recaud_real_t_','var_real_t_','mes_obj_','ctrl_','f1_','f2_'))]+['ind_renta_anual']
    if len(pred)!=47 or not np.isfinite(df[pred].to_numpy(float)).all(): raise ValueError('Predictores incompletos o no finitos')
    for j,g in df.groupby('jurisdiccion'):
        if not pd.DatetimeIndex(g.objetivo_t1.sort_values()).equals(pd.date_range('2007-03-01','2025-12-01',freq='MS')):
            raise ValueError(f'Panel incompleto: {j}')

def auditar_muestras(df):
    resumen=[]
    for op,bloques,corte,publicado in [('train',['ENTRENAMIENTO'],CORTE_TRAIN,4004),
            ('valid',['VALIDACION'],CORTE_SELECCION,936),
            ('reajuste',['ENTRENAMIENTO','VALIDACION'],CORTE_SELECCION,4940)]:
        d=df.loc[df.bloque_temporal.isin(bloques),['jurisdiccion','origen_t','objetivo_t1','fecha_disponible_Y']].copy()
        d['incluida']=d.fecha_disponible_Y <= corte;d['corte']=corte
        d['motivo']=np.where(d.incluida,'Disponible','A13 m+2 posterior al corte')
        d.to_csv(OUTPUT_DIR / f'Muestra_{op}_Incluidas_Excluidas.csv',index=False)
        resumen.append({'operacion':op,'publicado':publicado,'nominal':len(d),'efectivo':int(d.incluida.sum()),'corte':corte})
    pd.DataFrame(resumen).to_csv(OUTPUT_DIR / 'Auditoria_Muestras.csv',index=False)
    d=pd.DataFrame(resumen);d['referencia']='Tesis 4.2.2,4.3,5.1.1; auditoría C04'
    d.to_csv(OUTPUT_DIR / 'Discrepancias_Documentales_V8.csv',index=False)

def exportar_disponibilidad(df):
    registros=[]
    # Cada dependencia de historia (nivel y variación), clima y controles queda trazada.
    for origen in sorted(df.origen_t.unique()):
        origen=pd.Timestamp(origen);corte=corte_mes(origen)
        reglas=[('PBI',-2,2),('inflacion',-1,1),('TC',-1,1),('ONI',-4,4),('ICEN',-3,3)]
        reglas += [('ERA5',-k,3) for k in [3,4,5]]
        reglas += [(fuente,-k-extra,lag) for k in range(2,14) for extra in [0,12]
                   for fuente,lag in [('A13',2),('IPC',1)]]
        for fuente,desfase,lag in reglas:
            mes=origen+pd.DateOffset(months=desfase);disp=corte_mes(mes+pd.DateOffset(months=lag))
            if disp>corte: raise ValueError('Fuente futura detectada')
            registros.append({'fuente':fuente,'mes_observado':mes,'origen':origen,
                'objetivo':origen+pd.DateOffset(months=1),'disponible':disp,'corte':corte,'tipo':'Reconstruida-Seudorreal'})
    pd.DataFrame(registros).to_csv(OUTPUT_DIR / 'Matriz_Disponibilidad.csv',index=False)

def media_soporte(x,w):
    x=np.asarray(x,float);w=np.asarray(w,float)
    if not np.isfinite(x).all() or not np.isfinite(w).all() or (w<0).any() or w.sum()<=0:
        raise ValueError('Dato ausente dentro del soporte congelado o pesos inválidos')
    return float(np.dot(x,w)/w.sum())

def leer_grib_mensual(path, variable):
    import eccodes as ec
    arrays=[];meses=[];metadatos=[];lons=lats=None
    with open(path,'rb') as f:
        while True:
            gid=ec.codes_grib_new_from_file(f)
            if gid is None: break
            try:
                meta={k:ec.codes_get(gid,k) for k in ['shortName','units','stream','stepType','step','dataDate','Ni','Nj','class','bitmapPresent']}
                if meta['shortName']!=variable or meta['class']!='l5' or meta['bitmapPresent']!=1:
                    raise ValueError(f'Producto GRIB no reconocido: {meta}')
                mes=pd.to_datetime(str(meta['dataDate']),format='%Y%m%d').to_period('M').to_timestamp()
                if not pd.Timestamp('2005-01-01')<=mes<=pd.Timestamp('2025-12-01'): continue
                shape=(int(meta['Nj']),int(meta['Ni']))
                a=ec.codes_get_array(gid,'values').reshape(shape).astype(float)
                bitmap=ec.codes_get_array(gid,'bitmap').reshape(shape).astype(bool);a[~bitmap]=np.nan
                lon=ec.codes_get_array(gid,'longitudes').reshape(shape);lat=ec.codes_get_array(gid,'latitudes').reshape(shape)
                lon=(lon+180)%360-180
                if lons is None: lons=lon[0];lats=lat[:,0]
                elif not (np.array_equal(lons,lon[0]) and np.array_equal(lats,lat[:,0])): raise ValueError('GRIB cambia grilla')
                if variable=='2t' and meta['units']=='K' and meta['stream']=='moda':
                    a=a-273.15;meta['conversion']='K - 273.15 => Celsius'
                elif variable=='tp' and meta['units']=='m' and meta['stream']=='mnth' and meta['stepType']=='avgas' and int(meta['step'])==24:
                    # Media mensual de acumulaciones diarias de 24 h: una sola conversión.
                    a=a*1000*mes.days_in_month;meta['conversion']='m por acumulacion diaria media * 1000 * dias => mm/mes'
                else: raise ValueError(f'Unidad/acumulación sin regla documentada: {meta}')
                arrays.append(a);meses.append(mes);metadatos.append(meta)
            finally: ec.codes_release(gid)
    idx=np.argsort(meses);fechas=pd.DatetimeIndex([meses[i] for i in idx])
    if not fechas.equals(pd.date_range('2005-01-01','2025-12-01',freq='MS')): raise ValueError('GRIB sin continuidad mensual')
    return np.stack(arrays)[idx],fechas,lons,lats,metadatos

def brier_macro(df,p):
    y=df['Y_caida'].to_numpy(float);p=np.asarray(p,float).reshape(-1)
    if len(y)!=len(p) or not np.isfinite(p).all() or not np.isin(y,[0,1]).all() or ((p<0)|(p>1)).any():
        raise ValueError('Brier: etiquetas o probabilidades inválidas')
    return float(pd.Series((y-p)**2).groupby(df.jurisdiccion.to_numpy()).mean().mean())

def indices_mbb(T,block=6,n_boot=2000,seed=20260919):
    if not 1<=block<=T: raise ValueError('Bloque incompatible')
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block+1,size=(n_boot,int(np.ceil(T/block))))
    return (starts[:,:,None]+np.arange(block)).reshape(n_boot,-1)[:,:T]

def validar_predicciones(df):
    if df.duplicated(['jurisdiccion','objetivo_t1']).any(): raise ValueError('Predicciones duplicadas')
    if set(df.jurisdiccion)!=set(JURISDICCIONES_26): raise ValueError('Prueba: catálogo incorrecto')
    esperados=pd.date_range('2023-01-01','2025-12-01',freq='MS')
    for _,g in df.groupby('jurisdiccion'):
        if not pd.DatetimeIndex(pd.to_datetime(g.objetivo_t1).sort_values()).equals(esperados): raise ValueError('Prueba no común/completa')
    cols=[f'prob_{a}_{f}' for a in ['EN','RF','XGB','LSTM'] for f in ['F0','F1','F2']]
    for c in cols:brier_macro(df,df[c])

def verificar_contrastes(pred,c):
    for escenario,grp in c.groupby('Escenario'):
        sub=pred if escenario=='Panel 26 Jurisdicciones' else pred.loc[pred.jurisdiccion!='LIMA METROPOLITANA']
        directos={}
        for alg in ['EN','RF','XGB','LSTM']:
            bs=[brier_macro(sub,sub[f'prob_{alg}_{f}']) for f in ['F0','F1','F2']]
            directos[alg]=[bs[1]-bs[0],bs[2]-bs[1],bs[2]-bs[0]]
        directos['RESUMEN MULTIMODELO']=np.mean(list(directos.values()),axis=0)
        for alg,d in directos.items():
            g=grp.loc[grp.Familia_Algoritmica.eq(alg)].set_index('Contraste')
            v=[g.loc[n,'Delta_Brier'] for n in ['Aporte Territorial (F1-F0)','Aporte ENOS (F2-F1)','Aporte Total (F2-F0)']]
            np.testing.assert_allclose(v,d,atol=1e-12,rtol=0)
            np.testing.assert_allclose(v[2],v[0]+v[1],atol=1e-12,rtol=0)

def exportar_diagnosticos(pred):
    from sklearn.metrics import average_precision_score,roc_auc_score,precision_recall_curve,auc
    import statsmodels.api as sm
    filas=[]
    for a in ['EN','RF','XGB','LSTM']:
        for f in ['F0','F1','F2']:
            for nivel,clave,d in [('pooled','toda_prueba',pred)]+[('anio',str(k),d) for k,d in pred.groupby(pred.objetivo_t1.dt.year)]+[('jurisdiccion',k,d) for k,d in pred.groupby('jurisdiccion')]:
                y=d.Y_caida.to_numpy();p=d[f'prob_{a}_{f}'].to_numpy()
                row={'model_id':f'{a}_{f}_reajuste','conjunto':'PRUEBA','nivel':nivel,'clave':clave,'n':len(d),
                    'inicio':d.objetivo_t1.min(),'fin':d.objetivo_t1.max(),'Brier_Macro':brier_macro(d,p),
                    'hash_predicciones':sha256_dataframe(d[['jurisdiccion','objetivo_t1','Y_caida',f'prob_{a}_{f}']])}
                if len(np.unique(y))<2: row['motivo']='Una sola clase: AP/ROC/calibracion no comparables'
                else:
                    row['AP']=average_precision_score(y,p);row['ROC_AUC']=roc_auc_score(y,p)
                    precision,recall,_=precision_recall_curve(y,p);row['PR_AUC_Trapezoidal_Historico']=auc(recall,precision)
                    try:
                        # Clip numérico únicamente para el logit diagnóstico; no cambia p evaluada.
                        q=np.clip(p,1e-12,1-1e-12)
                        fit=sm.GLM(y,sm.add_constant(np.log(q/(1-q)),has_constant='add'),family=sm.families.Binomial()).fit()
                        if not fit.converged or not np.isfinite(fit.params).all(): raise ValueError('Calibración no converge')
                        row['Intercepto_Calibracion'],row['Pendiente_Calibracion']=fit.params
                    except Exception as exc:row['motivo_calibracion']=str(exc)
                filas.append(row)
    pd.DataFrame(filas).to_csv(OUTPUT_DIR / 'Diagnosticos_AP_ROC_Calibracion.csv',index=False)

def definir_caida(actual,anterior):
    actual=np.asarray(actual,float);anterior=np.asarray(anterior,float)
    if ((actual<=0)&np.isfinite(actual)).any() or ((anterior<=0)&np.isfinite(anterior)).any():
        raise ValueError('Recaudación no positiva')
    return np.where(np.isfinite(actual)&np.isfinite(anterior),(actual<anterior).astype(float),np.nan)

def pruebas_nucleo():
    # Casos artificiales mínimos: validan funciones, no son fuentes del experimento.
    d=pd.DataFrame({'jurisdiccion':['A','A','B'],'Y_caida':[1,0,1]})
    np.testing.assert_allclose(brier_macro(d,[.8,.3,.4]),((.04+.09)/2+.36)/2,rtol=0,atol=1e-12)
    serie=np.arange(36,dtype=float)**2;idx=indices_mbb(36)
    if abs(serie.mean()-serie[idx].mean())<1:raise AssertionError('Fixture no distingue centro bootstrap')
    if not (disponibilidad_etiqueta('2022-10-01')<=CORTE_SELECCION<disponibilidad_etiqueta('2022-11-01')):raise AssertionError('Corte m+2')
    y=definir_caida([2,np.nan,3],[3,1,np.nan]);assert y[0]==1 and np.isnan(y[1:]).all()
    np.testing.assert_allclose(media_soporte([20,30],[.2,.3]),26,atol=1e-12)
    try:media_soporte([20,np.nan],[.2,.3])
    except ValueError:pass
    else:raise AssertionError('Faltante no bloqueado')
    assert (.2+.3)/1.0==.5 # cobertura física, diferente de pesos normalizados
    cfg=json.loads(json.dumps({'max_features':.5,'max_depth':None,'n_estimators':500}))
    assert isinstance(cfg['max_features'],float) and cfg['max_depth'] is None and isinstance(cfg['n_estimators'],int)
    resultado={'estado':'OK','nivel':'unitario','pruebas':['Brier territorial y macro','centro bootstrap distinto',
        'corte m+2','objetivo desconocido','soporte faltante','cobertura','tipos JSON']}
    (OUTPUT_DIR/'Pruebas_Nucleo.json').write_text(json.dumps(resultado,indent=2))
    return resultado

pruebas_nucleo()

Entorno efectivo registrado; no se sustituyeron bibliotecas ya instaladas.
Seleccione DataOrigen.zip (también se aceptan sus copias con sufijos).


Saving DataOrigen.zip to DataOrigen.zip

[VERIFICACIÓN DE ARCHIVOS]
 - CuadroA13                     : OK -> CuadroA13.xlsx
 - TablaCalendario               : OK -> TablaCalendario.xlsx
 - TablaIPC                      : OK -> TablaIPC.csv
 - TablaInflacion                : OK -> TablaInflacion.csv
 - TablaPBI                      : OK -> TablaPBI.csv
 - TipoCambio                    : OK -> TipoCambio.csv
 - ICEN                          : OK -> ICEN.txt
 - ONI                           : OK -> ONI.xlsx
 - PROVINCIA.gpkg                : OK -> PROVINCIA.gpkg
 - precipitacion_era5land.grib   : OK -> precipitacion_era5land.grib
 - temperatura_era5land.grib     : OK -> temperatura_era5land.grib
[2026-10-30T00:35:06.918366+00:00] [INFO] [HASH] HASH registrado: DataOrigen.zip = ed5a55d08b6f9105...
[2026-10-30T00:35:06.918845+00:00] [INFO] [FLAG] FLAG creado: Celda02_Carga_ZIP.OK.flag


{'estado': 'OK',
 'nivel': 'unitario',
 'pruebas': ['Brier territorial y macro',
  'centro bootstrap distinto',
  'corte m+2',
  'objetivo desconocido',
  'soporte faltante',
  'cobertura',
  'tipos JSON']}

## Paso 03 — Carga de fuentes F0, F1 y F2

Carga en memoria todas las fuentes de información necesarias para construir el panel:

- **F0 (tributario y macro):** Cuadro A13 de SUNAT, calendario tributario, IPC de Lima, PBI, inflación y tipo de cambio. Se homologan los nombres de las 26 jurisdicciones y se deflacta el IPC con base diciembre 2021.
- **F1 (clima territorial):** se leen los archivos GRIB de precipitación y temperatura (ERA5-Land), se intersectan las celdas de la grilla con las jurisdicciones del `PROVINCIA.gpkg`, se calculan los pesos espaciales (`INTER_PESOS`) y se obtienen las series mensuales de precipitación y temperatura por jurisdicción.
- **F2 (índices ENOS):** se leen `ONI.xlsx` e `ICEN.txt` y se construyen sus series mensuales.

Al final se imprimen los hashes SHA-256 de cada archivo fuente y se crea el flag `Celda03_Carga_Fuentes.OK.flag`.

**Control V8:** Bitmap terrestre congelado, áreas equivalentes y unidades por metadatos GRIB. Calendario desde vencimientos; publicaciones faltantes identificadas como reconstrucción.


In [4]:
# ==============================================================================
# 03: CARGA DE FUENTES Y GENERACIÓN DE HASHES (REPRODUCIBILIDAD)
# ==============================================================================
# Carga las fuentes F0 (tributario + macro BCRP), F1 (clima territorial) y F2
# (índices ENOS). El IPC, la inflación, el PBI y el tipo de cambio vienen con
# formato BCRP de 2 filas de cabecera y fechas tipo "Ene91"; se parsean con un
# único loader robusto común.
# La inflación mensual se toma DIRECTAMENTE de TablaInflacion.csv (serie oficial
# PN01271PM), no se deriva por pct_change() del IPC.

# Ocultar únicamente el aviso de obsolescencia generado por Jupyter.
import warnings

warnings.filterwarnings(
    action="ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

try:
    import re
    import numpy as np
    import pandas as pd

    print("\n--- CARGANDO F0: TRIBUTARIO Y MACRO ---")

    # =========================================================================
    # Loader robusto para CSV del BCRP
    # =========================================================================
    _ABREV_MESES_BCRP = {
        'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
        'jul': 7, 'ago': 8, 'sep': 9, 'set': 9, 'oct': 10,
        'nov': 11, 'dic': 12,
    }

    def _parsear_mes_bcrp(texto):
        """'Ene91' → Timestamp(1991-01-01). 'Dic21' → 2021-12-01. Otros → NaT."""
        s = str(texto).strip().lower()
        m = re.fullmatch(r'([a-z]{3})\.?(\d{2})', s)
        if not m:
            return pd.NaT
        mes_abrev, anio_2d = m.group(1), int(m.group(2))
        if mes_abrev not in _ABREV_MESES_BCRP:
            return pd.NaT
        anio = 1900 + anio_2d if anio_2d >= 50 else 2000 + anio_2d
        return pd.Timestamp(year=anio, month=_ABREV_MESES_BCRP[mes_abrev], day=1)

    def _leer_csv_bcrp(path, col_valor):
        """Lee un CSV BCRP con 2 filas de cabecera y lo devuelve limpio:
           DataFrame con columnas ['fecha', col_valor]."""
        raw = None
        for enc in ['utf-8-sig', 'utf-8', 'latin1', 'cp1252']:
            try:
                raw = pd.read_csv(path, sep=',', header=None,
                                  encoding=enc, engine='python',
                                  skipinitialspace=True)
                break
            except Exception:
                continue
        if raw is None:
            raise ValueError(f"No se pudo leer {path}")

        registros = []
        for _, fila in raw.iterrows():
            fecha = _parsear_mes_bcrp(fila.iloc[0])
            if pd.isna(fecha):
                continue
            try:
                valor = float(str(fila.iloc[1]).replace(',', '.').strip())
            except (ValueError, TypeError):
                continue
            if not np.isfinite(valor):
                continue
            registros.append({'fecha': fecha, col_valor: valor})

        df = pd.DataFrame(registros).sort_values('fecha').reset_index(drop=True)
        df = df[
            (df['fecha'] >= '2005-01-01') &
            (df['fecha'] <= '2025-12-01')
        ].reset_index(drop=True)
        return df

    # =========================================================================
    # Carga tributaria (CuadroA13 y Calendario)
    # =========================================================================
    def cargar_serie_bcrp(filepath, col_names):
        """Wrapper legacy."""
        df = _leer_csv_bcrp(filepath, col_names[0])
        for extra in col_names[1:]:
            df[extra] = np.nan
        return df

    def cargar_cuadro_a13(filepath):
        raw = pd.read_excel(filepath, sheet_name='cdrA13', header=None)
        fila_anio, fila_mes = None, None
        for i in range(min(15, len(raw))):
            if len([v for v in raw.iloc[i].dropna().astype(str)
                    if v.strip().isdigit()
                    and 2005 <= int(v.strip()) <= 2030]) >= 2:
                fila_anio = i
                break
        for i in range(fila_anio + 1, min(fila_anio + 5, len(raw))):
            if len([v for v in raw.iloc[i].dropna().astype(str)
                    if v.strip()[:3].lower() in
                    ['ene','feb','mar','abr','may','jun','jul',
                     'ago','set','sep','oct','nov','dic','tot']]) >= 6:
                fila_mes = i
                break

        anios_raw, meses_raw = raw.iloc[fila_anio], raw.iloc[fila_mes]
        mapa_fechas = {}
        anio_actual = None
        for j in range(2, raw.shape[1]):
            if pd.notna(anios_raw.iloc[j]):
                try:
                    a_int = int(anios_raw.iloc[j])
                    if 2005 <= a_int <= 2030:
                        anio_actual = a_int
                except Exception:
                    pass
            if pd.notna(meses_raw.iloc[j]):
                m_str = str(meses_raw.iloc[j]).strip().lower()
                if m_str.startswith('tot'):
                    continue
                mes_num = {'ene':1,'feb':2,'mar':3,'abr':4,'may':5,'jun':6,
                           'jul':7,'ago':8,'set':9,'sep':9,'oct':10,
                           'nov':11,'dic':12}.get(m_str[:3])
                if mes_num and anio_actual:
                    mapa_fechas[j] = pd.Timestamp(year=anio_actual,
                                                  month=mes_num, day=1)

        registros = []
        for i in range(fila_mes + 1, len(raw)):
            fila = raw.iloc[i]
            jur = fila.iloc[0] if pd.notna(fila.iloc[0]) else fila.iloc[1]
            if pd.isna(jur):
                continue
            jur_str = str(jur).strip()
            if jur_str.lower() == 'total' or jur_str.upper() == 'LIMA':
                continue
            jur_norm = homologar_nombre_jurisdiccion(jur_str)
            if jur_norm not in JURISDICCIONES_26:
                continue
            for j_col, fecha in mapa_fechas.items():
                if pd.notna(fila.iloc[j_col]):
                    registros.append({
                        'jurisdiccion': jur_norm,
                        'fecha': fecha,
                        'recaud_nominal': float(fila.iloc[j_col]),
                    })

        df_a13 = pd.DataFrame(registros)
        if df_a13.duplicated(['jurisdiccion', 'fecha']).any():
            raise ValueError('A13: claves duplicadas, resolver con evidencia; no sumar automáticamente')
        df_a13 = df_a13[
            (df_a13['fecha'] >= '2005-01-01') &
            (df_a13['fecha'] <= '2025-12-01')
        ]
        print(f"[F0] CuadroA13: {len(df_a13)} filas | "
              f"{df_a13['jurisdiccion'].nunique()} jurisdicciones")
        return df_a13.sort_values(['jurisdiccion', 'fecha']).reset_index(drop=True)

    def cargar_calendario_tributario(filepath):
        raw = pd.read_excel(filepath)
        req = {'fecha_vencimiento', 'fecha_publicacion', 'ejercicio_fiscal', 'grupo_cronograma'}
        if not req.issubset(raw.columns):
            raise ValueError(f"Calendario sin columnas requeridas: {req-set(raw.columns)}")
        raw['fecha_vencimiento'] = pd.to_datetime(raw['fecha_vencimiento'], errors='raise')
        raw['fecha_publicacion'] = pd.to_datetime(raw['fecha_publicacion'], errors='coerce')
        raw['fecha'] = raw['fecha_vencimiento'].dt.to_period('M').dt.to_timestamp()
        raw['corte_origen'] = raw['fecha'].map(lambda d: corte_mes(d-pd.DateOffset(months=1)))
        raw['disponibilidad_tipo'] = np.where(raw['fecha_publicacion'].isna(),
            'Reconstruida-Seudorreal: calendario conocido al origen previo (supuesto)', 'Publicacion_aportada')
        raw['disponible_operativa'] = [c if pd.isna(p) else pd.Timestamp(p).tz_localize('America/Lima')
            for p,c in zip(raw['fecha_publicacion'], raw['corte_origen'])]
        raw['conocido_origen'] = raw['disponible_operativa'] <= raw['corte_origen']
        raw.to_csv(OUTPUT_DIR / 'Calendario_Trazabilidad.csv', index=False)
        fechas = pd.date_range('2005-01-01', '2025-12-01', freq='MS')
        res = pd.DataFrame({'fecha': fechas})
        indicador = raw.groupby('fecha')['conocido_origen'].max().astype(int)
        res['ind_renta_anual'] = res.fecha.map(indicador).fillna(0).astype(int)
        return res

    # =========================================================================
    # Carga de fuentes F0
    # =========================================================================
    path_a13    = buscar_archivo("CuadroA13")
    path_cal    = buscar_archivo("TablaCalendario")
    path_ipc    = buscar_archivo("TablaIPC")
    path_infl   = buscar_archivo("TablaInflacion")
    path_pbi    = buscar_archivo("TablaPBI")
    path_tc     = buscar_archivo("TipoCambio")

    df_a13 = cargar_cuadro_a13(path_a13)
    df_cal = cargar_calendario_tributario(path_cal)

    # IPC (índice base dic-2021 = 100)
    df_ipc = _leer_csv_bcrp(path_ipc, 'ipc_lima')
    print(f"[F0] IPC: {len(df_ipc)} filas | "
          f"rango: {df_ipc['fecha'].min().date()} → "
          f"{df_ipc['fecha'].max().date()}")

    meses_esperados = pd.date_range('2005-01-01', '2025-12-01', freq='MS')
    faltantes = sorted(set(meses_esperados) - set(df_ipc['fecha']))
    if faltantes:
        raise ValueError(f"M20: IPC incompleto; no se interpola con información futura: {faltantes}")

    ipc_dic21 = df_ipc.loc[df_ipc['fecha'] == '2021-12-01', 'ipc_lima'].iloc[0]
    if abs(ipc_dic21 - 100.0) > 0.01:
        print(f"[F0] IPC: reescalando a base dic-2021=100 "
              f"(actual={ipc_dic21:.4f})")
        df_ipc['ipc_lima'] = df_ipc['ipc_lima'] * (100.0 / ipc_dic21)

    # Inflación mensual OFICIAL (TablaInflacion.csv, PN01271PM)
    df_inflacion = _leer_csv_bcrp(path_infl, 'inflacion_mensual')
    print(f"[F0] Inflación: {len(df_inflacion)} filas | "
          f"rango: {df_inflacion['fecha'].min().date()} → "
          f"{df_inflacion['fecha'].max().date()} | "
          f"válidos: {df_inflacion['inflacion_mensual'].notna().sum()}")

    # PBI
    df_pbi = _leer_csv_bcrp(path_pbi, 'pbi_var')
    print(f"[F0] PBI: {len(df_pbi)} filas | "
          f"rango: {df_pbi['fecha'].min().date()} → "
          f"{df_pbi['fecha'].max().date()} | "
          f"válidos: {df_pbi['pbi_var'].notna().sum()}")

    # Tipo de cambio
    df_tc = _leer_csv_bcrp(path_tc, 'tipo_cambio')
    print(f"[F0] TC: {len(df_tc)} filas | "
          f"rango: {df_tc['fecha'].min().date()} → "
          f"{df_tc['fecha'].max().date()} | "
          f"válidos: {df_tc['tipo_cambio'].notna().sum()}")

    # =========================================================================
    # Carga de F1: clima territorial
    # =========================================================================
    print("\n--- CARGANDO F1: CLIMA TERRITORIAL (ERA5-LAND) ---")

    def construir_clima_territorial_f1():
        gpkg_path = buscar_archivo("PROVINCIA.gpkg") or buscar_archivo("DISTRITO.gpkg")
        gdf = gpd.read_file(gpkg_path)

        geo_cols = [c for c in gdf.columns if str(gdf[c].dtype).startswith("geometry")]
        col_geo_real = 'geometry' if 'geometry' in geo_cols else geo_cols[0]
        if col_geo_real != 'geometry':
            gdf = gdf.rename(columns={col_geo_real: 'geometry'})
        if gdf.crs is None: raise ValueError("Cartografía sin CRS documentado")
        gdf = gdf.to_crs("EPSG:4326")
        gdf = gpd.GeoDataFrame(
            gdf.rename(columns={
                c: normalizar_texto(c) for c in gdf.columns if c != 'geometry'
            }),
            geometry='geometry', crs="EPSG:4326"
        )

        dep_col = [c for c in gdf.columns if 'DEP' in c][0]
        prov_col = [c for c in gdf.columns if 'PROV' in c][0]

        def asignar_jurisdiccion(row):
            dep, prov = normalizar_texto(row[dep_col]), normalizar_texto(row[prov_col])
            if 'CALLAO' in dep or 'CALLAO' in prov:
                return 'CALLAO'
            if dep == 'LIMA':
                return 'LIMA METROPOLITANA' if prov == 'LIMA' else 'LIMA PROVINCIAS'
            return homologar_nombre_jurisdiccion(dep)

        gdf['jurisdiccion'] = gdf.apply(asignar_jurisdiccion, axis=1)
        gdf_26 = (gdf[gdf['jurisdiccion'].isin(JURISDICCIONES_26)]
                  [['jurisdiccion', 'geometry']]
                  .dissolve(by='jurisdiccion').reset_index())

        # Lectura nativa de bitmap y metadatos, antes de cualquier promedio.
        tp_vals, times, lon_values, lat_values, meta_p = leer_grib_mensual(buscar_archivo('precipitacion_era5land.grib'), 'tp')
        t2m_vals, times_t, lon_t, lat_t, meta_t = leer_grib_mensual(buscar_archivo('temperatura_era5land.grib'), '2t')
        if not (times.equals(times_t) and np.array_equal(lon_values,lon_t) and np.array_equal(lat_values,lat_t)):
            raise ValueError('GRIB: grillas o meses no coincidentes')
        (OUTPUT_DIR / 'Metadatos_GRIB.json').write_text(json.dumps({'precip':meta_p,'temp':meta_t},indent=2))

        # Soporte congelado del bitmap terrestre de enero 2005, antes del desarrollo.
        soporte = np.isfinite(tp_vals[0]) & np.isfinite(t2m_vals[0])
        if not np.array_equal(np.isfinite(tp_vals[0]), np.isfinite(t2m_vals[0])):
            raise ValueError('Bitmaps terrestres iniciales incompatibles')
        if not (np.isfinite(tp_vals[:,soporte]).all() and np.isfinite(t2m_vals[:,soporte]).all()):
            raise ValueError('Ausencia inesperada dentro del soporte congelado; revisar GRIB')
        if (np.isfinite(tp_vals[:,~soporte]).any() or np.isfinite(t2m_vals[:,~soporte]).any()):
            raise ValueError('Bitmap cambia fuera del soporte inicial; revisar definición terrestre sin usar desempeño')
        np.save(OUTPUT_DIR / 'Soporte_Terrestre_Congelado.npy', soporte)

        from shapely.geometry import box
        lons, lats = np.meshgrid(lon_values, lat_values)
        dlon = abs(lon_values[1] - lon_values[0]) / 2.0
        dlat = abs(lat_values[1] - lat_values[0]) / 2.0

        cells = [{'lat_idx': i, 'lon_idx': j,
                  'geometry': box(lons[i, j] - dlon, lats[i, j] - dlat,
                                  lons[i, j] + dlon, lats[i, j] + dlat)}
                 for i in range(lats.shape[0])
                 for j in range(lons.shape[1])]
        gdf_cells = gpd.GeoDataFrame(cells, geometry='geometry', crs="EPSG:4326")

        inter = gpd.overlay(gdf_cells.to_crs('EPSG:6933'),
                            gdf_26[['jurisdiccion', 'geometry']].to_crs('EPSG:6933'),
                            how='intersection')

        # Igual proyección equivalente para numerador y denominador, incluye islas
        # e intersecciones parciales: EPSG:6933 es una proyección de área equivalente.
        inter['area'] = inter.to_crs('EPSG:6933').area
        area_jur = gdf_26.set_index('jurisdiccion').to_crs('EPSG:6933').area
        inter['soporte_valido'] = soporte[inter.lat_idx.to_numpy(), inter.lon_idx.to_numpy()]
        inter.drop(columns='geometry').to_csv(OUTPUT_DIR / 'Intersecciones_Todas.csv',index=False)
        inter = inter.loc[inter.soporte_valido & inter.area.gt(0)].copy()
        inter['area_jurisdiccional'] = inter.jurisdiccion.map(area_jur)
        inter['area_soporte_valido'] = inter.groupby('jurisdiccion').area.transform('sum')
        inter['coverage_ratio'] = inter.area_soporte_valido / inter.area_jurisdiccional
        if not inter.coverage_ratio.between(0,1+1e-9).all():
            raise ValueError('Cobertura geométrica inconsistente; no limitar a uno')
        inter['weight'] = inter.area / inter.area_soporte_valido
        if set(inter.jurisdiccion) != set(JURISDICCIONES_26): raise ValueError('Jurisdicción sin soporte')
        if not np.allclose(inter.groupby('jurisdiccion').weight.sum(),1,atol=1e-12,rtol=0):
            raise ValueError('Pesos sin normalizar')
        global INTER_PESOS
        INTER_PESOS = inter.copy()
        inter.drop(columns='geometry').to_csv(OUTPUT_DIR / 'Pesos_Climaticos.csv', index=False)

        records = []
        for t_idx, dt in enumerate(times):
            fecha_mes = pd.Timestamp(year=dt.year, month=dt.month, day=1)
            if (fecha_mes < pd.Timestamp('2005-01-01') or
                fecha_mes > pd.Timestamp('2025-12-01')):
                continue
            days = fecha_mes.days_in_month
            for jur, grp in inter.groupby('jurisdiccion'):
                w, li, lj = (grp['weight'].values,
                             grp['lat_idx'].values,
                             grp['lon_idx'].values)
                p_mm = media_soporte(tp_vals[t_idx, li, lj], w)
                t_c = media_soporte(t2m_vals[t_idx, li, lj], w)
                records.append({'jurisdiccion': jur, 'fecha': fecha_mes,
                                'precip': p_mm, 'temp': t_c})

        df_clima = pd.DataFrame(records)
        df_clima['fecha'] = (pd.to_datetime(df_clima['fecha'])
                             .dt.to_period('M').dt.to_timestamp())
        print(f"[F1] Clima territorial: {len(df_clima)} filas | "
              f"jurisdicciones: {df_clima['jurisdiccion'].nunique()} | "
              f"rango: {df_clima['fecha'].min().date()} → "
              f"{df_clima['fecha'].max().date()}")
        return df_clima

    df_clima = construir_clima_territorial_f1()

    # =========================================================================
    # Carga de F2: índices ENOS
    # =========================================================================
    print("\n--- CARGANDO F2: ÍNDICES ENOS ---")

    def cargar_indices_enos_f2():
        icen_rows = []
        with open(buscar_archivo("ICEN.txt"), "r", encoding="latin1") as f:
            for line in f:
                parts = line.strip().replace(',', '.').split()
                nums = [float(p) for p in parts
                        if p.replace('.', '', 1).replace('-', '', 1).isdigit()]
                if len(nums) >= 3 and 1950 <= int(nums[0]) <= 2030:
                    icen_rows.append({
                        'fecha_central': pd.Timestamp(
                            year=int(nums[0]), month=int(nums[1]), day=1),
                        'icen': nums[2],
                    })
        df_icen = (pd.DataFrame(icen_rows)
                   .drop_duplicates('fecha_central')
                   .sort_values('fecha_central'))

        raw_oni = pd.read_excel(buscar_archivo("ONI.xlsx"))
        raw_oni.columns = [normalizar_texto(c) for c in raw_oni.columns]
        oni_rows = []
        trimestres = {'DJF':1,'JFM':2,'FMA':3,'MAM':4,'AMJ':5,'MJJ':6,
                      'JJA':7,'JAS':8,'ASO':9,'SON':10,'OND':11,'NDJ':12}
        for _, r in raw_oni.iterrows():
            try:
                yr = int(float(str(r[raw_oni.columns[0]]).strip()))
                if 1950 <= yr <= 2030:
                    for tri, m_num in trimestres.items():
                        if tri in raw_oni.columns and pd.notna(r[tri]):
                            oni_rows.append({
                                'fecha_central': pd.Timestamp(
                                    year=yr, month=m_num, day=1),
                                'oni': float(r[tri]),
                            })
            except Exception:
                continue
        df_oni = (pd.DataFrame(oni_rows)
                  .drop_duplicates('fecha_central')
                  .sort_values('fecha_central'))
        print(f"[F2] ONI: {len(df_oni)} filas | ICEN: {len(df_icen)} filas")
        return df_oni, df_icen

    df_oni, df_icen = cargar_indices_enos_f2()

    # Fuentes completas, finitas y sin duplicados antes del panel.
    for nombre, marco, fecha, valores, grupos in [
        ('A13',df_a13,'fecha',['recaud_nominal'],['jurisdiccion']),
        ('IPC',df_ipc,'fecha',['ipc_lima'],[]), ('inflacion',df_inflacion,'fecha',['inflacion_mensual'],[]),
        ('PBI',df_pbi,'fecha',['pbi_var'],[]), ('TC',df_tc,'fecha',['tipo_cambio'],[]),
        ('clima',df_clima,'fecha',['precip','temp'],['jurisdiccion'])]:
        validar_fuente(marco,fecha,valores,grupos,nombre)
        marco.to_csv(OUTPUT_DIR / f'Fuente_{nombre}.csv',index=False)
    for f in BASE_DIR.rglob('*'):
        if f.is_file(): registrar_hash(str(f.relative_to(BASE_DIR)),path=f)

    # =========================================================================
    # HASH DE INSUMOS (AUDITORÍA)
    # =========================================================================
    print("\n[HASH DE INSUMOS FUENTE (SHA-256)]")
    print(f"CuadroA13  : {calcular_hash_archivo(path_a13)[:16]}...")
    print(f"TablaIPC   : {calcular_hash_archivo(path_ipc)[:16]}...")
    print(f"TablaInfl  : {calcular_hash_archivo(path_infl)[:16]}...")
    print(f"TablaPBI   : {calcular_hash_archivo(path_pbi)[:16]}...")
    print(f"TipoCambio : {calcular_hash_archivo(path_tc)[:16]}...")
    print(f"Calendario : {calcular_hash_archivo(path_cal)[:16]}...")
    print(f"ONI.xlsx   : {calcular_hash_archivo(buscar_archivo('ONI.xlsx'))[:16]}...")
    print(f"ICEN.txt   : {calcular_hash_archivo(buscar_archivo('ICEN.txt'))[:16]}...")
    print(f"GPKG       : {calcular_hash_archivo(buscar_archivo('PROVINCIA.gpkg'))[:16]}...")

    registrar_hash("CuadroA13.xlsx",   path=path_a13)
    registrar_hash("TablaIPC.csv",     path=path_ipc)
    registrar_hash("TablaInflacion.csv", path=path_infl)
    registrar_hash("TablaPBI.csv",     path=path_pbi)
    registrar_hash("TipoCambio.csv",   path=path_tc)
    registrar_hash("TablaCalendario.xlsx", path=path_cal)
    registrar_hash("ONI.xlsx",         path=buscar_archivo('ONI.xlsx'))
    registrar_hash("ICEN.txt",         path=buscar_archivo('ICEN.txt'))
    registrar_hash("PROVINCIA.gpkg",   path=buscar_archivo('PROVINCIA.gpkg'))

    crear_flag("Celda03_Carga_Fuentes", "OK", {
        "fuentes": 9,
        "ipc_filas": len(df_ipc),
        "inflacion_filas": len(df_inflacion),
        "pbi_filas": len(df_pbi),
        "tc_filas": len(df_tc),
    })

except Exception as e:
    registrar_error("Celda03_Carga_Fuentes", e)
    raise


--- CARGANDO F0: TRIBUTARIO Y MACRO ---
[F0] CuadroA13: 6552 filas | 26 jurisdicciones
[F0] IPC: 252 filas | rango: 2005-01-01 → 2025-12-01
[F0] Inflación: 252 filas | rango: 2005-01-01 → 2025-12-01 | válidos: 252
[F0] PBI: 252 filas | rango: 2005-01-01 → 2025-12-01 | válidos: 252
[F0] TC: 252 filas | rango: 2005-01-01 → 2025-12-01 | válidos: 252

--- CARGANDO F1: CLIMA TERRITORIAL (ERA5-LAND) ---
[F1] Clima territorial: 6552 filas | jurisdicciones: 26 | rango: 2005-01-01 → 2025-12-01

--- CARGANDO F2: ÍNDICES ENOS ---
[F2] ONI: 919 filas | ICEN: 917 filas
[2026-10-30T00:38:32.006544+00:00] [INFO] [HASH] HASH registrado: DataOrigen/F2/ICEN.txt = 53e82d90ad26335b...
[2026-10-30T00:38:32.007480+00:00] [INFO] [HASH] HASH registrado: DataOrigen/F2/ONI.xlsx = c5d03c654e2bd5d2...
[2026-10-30T00:38:32.008923+00:00] [INFO] [HASH] HASH registrado: DataOrigen/F0/CuadroA13.xlsx = f588b39142bd8833...
[2026-10-30T00:38:32.009432+00:00] [INFO] [HASH] HASH registrado: DataOrigen/F0/TablaInflacion.cs

## Paso 04 — Construcción del panel analítico y validación de la muestra

Construye el **panel analítico definitivo** (5 876 observaciones = 26 jurisdicciones × 226 meses objetivo), aplicando:

- Deflactación de la recaudación con el IPC de Lima (base dic-2021).
- Cálculo de la variación real interanual `g` y de la etiqueta binaria `Y_caida` (1 si `g < 0`).
- Anomalías climáticas `z_P`, `z_T`, precipitación acumulada 3m y temperatura media 3m, con **parámetros estimados solo en entrenamiento**.
- Historia tributaria (t-2 a t-13), dummies de mes, calendario, controles macro y variables F1/F2 con sus respectivos **rezagos de disponibilidad**.
- Asignación de los bloques temporales: **ENTRENAMIENTO** (2007-03 a 2019-12), **VALIDACIÓN** (2020-01 a 2022-12) y **PRUEBA** (2023-01 a 2025-12).

Se valida que no haya fuga temporal, se guarda `Panel_Analitico_F0_F1_F2.csv` y se registran los hashes y el flag `Celda04_Panel_Maestro.OK.flag`.


In [6]:
# ==============================================================================
# 04: CONSTRUCCIÓN DEL PANEL Y VALIDACIÓN DE LA MUESTRA
# ==============================================================================
# Esta celda construye el panel analítico definitivo (5,876 obs) desde las
# fuentes F0, F1 y F2 ya cargadas en el paso 03.
# Registra hashes, flags, logs y un resumen de la muestra en 00_AUDITORIA.
# Al final deja disponibles df_panel y panel_base para el (EDA).

# Ocultar únicamente el aviso de obsolescencia generado por Jupyter.
import warnings

warnings.filterwarnings(
    action="ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

import numpy as np
import pandas as pd
from pathlib import Path

# ==============================================================================
# FUNCIÓN PRINCIPAL (decorada para auditoría)
# ==============================================================================
@auditar("Celda04_Panel_Maestro")
def construir_panel_maestro(
    df_a13_g, df_ipc_g, df_inflacion_g, df_clima_g, df_cal_g,
    df_pbi_g, df_tc_g, df_oni_g, df_icen_g,
):
    log("Iniciando construcción del panel maestro",
        celda="Celda04_Panel_Maestro")

    # -------------------------------------------------------------------------
    # 1. Deflactación y variable objetivo
    # -------------------------------------------------------------------------
    log("Deflactando recaudación con IPC Lima", celda="Celda04_Panel_Maestro")
    panel_base = pd.merge(
        df_a13_g, df_ipc_g[['fecha', 'ipc_lima']], on='fecha', how='left'
    )
    panel_base = panel_base.sort_values(['jurisdiccion', 'fecha']).reset_index(drop=True)

    n_nan_ipc = int(panel_base['ipc_lima'].isna().sum())
    if n_nan_ipc > 0:
        raise ValueError(f'{n_nan_ipc} filas sin IPC')

    if (panel_base[['recaud_nominal','ipc_lima']] <= 0).any().any():
        raise ValueError('Montos/IPC no positivos; revisar antes del logaritmo')
    panel_base['recaud_real'] = (
        panel_base['recaud_nominal'] * (100.0 / panel_base['ipc_lima'])
    )
    panel_base['recaud_real_lag12'] = (
        panel_base.groupby('jurisdiccion')['recaud_real'].shift(12)
    )
    panel_base['var_real_yoy'] = 100.0 * (
        np.log(panel_base['recaud_real'].replace(0, np.nan))
        - np.log(panel_base['recaud_real_lag12'].replace(0, np.nan))
    )
    panel_base['caida_real'] = definir_caida(panel_base['recaud_real'],panel_base['recaud_real_lag12'])
    panel_base.loc[panel_base['var_real_yoy'].isna(), 'caida_real'] = np.nan

    n_validos_base = int(panel_base['caida_real'].notna().sum())
    log(f"panel_base: {len(panel_base)} filas | "
        f"caida_real válidos: {n_validos_base}",
        celda="Celda03_Panel_Maestro")

    if n_validos_base == 0:
        raise ValueError(
            "caida_real es NaN en TODAS las filas de panel_base. "
            "Revisar el merge df_a13 ↔ df_ipc_raw y el lag12."
        )

    # -------------------------------------------------------------------------
    # 2. Anomalías climáticas (parámetros SOLO con entrenamiento hasta Ago-2019)
    # -------------------------------------------------------------------------
    log("Calculando anomalías climáticas (train-only hasta 2019-08)",
        celda="Celda03_Panel_Maestro")
    df_clima = df_clima_g.copy()
    df_clima = df_clima.sort_values(['jurisdiccion', 'fecha']).reset_index(drop=True)
    df_clima['mes_cal'] = df_clima['fecha'].dt.month

    # Meses fuente vinculados a objetivos admisibles al cierre inicial.
    objetivos_train = pd.date_range('2007-03-01','2019-12-01',freq='MS')
    objetivos_train = [d for d in objetivos_train if disponibilidad_etiqueta(d) <= CORTE_TRAIN]
    meses_fuente = pd.DatetimeIndex(objetivos_train) - pd.DateOffset(months=4)
    train_ref = df_clima[df_clima.fecha.isin(meses_fuente)].copy()
    stats_clima = train_ref.groupby(['jurisdiccion','mes_cal']).agg(
        p_mean=('precip','mean'),p_std=('precip',lambda x:x.std(ddof=0)),
        t_mean=('temp','mean'),t_std=('temp',lambda x:x.std(ddof=0)),
        n_entrenamiento=('fecha','size'), primer_mes_fuente=('fecha','min'),
        ultimo_mes_fuente=('fecha','max')).reset_index()
    if (stats_clima.n_entrenamiento < 2).any() or (stats_clima[['p_std','t_std']] <= 0).any().any():
        raise ValueError('Anomalías: grupo insuficiente o desviación cero')
    global PARAMETROS_CLIMA
    PARAMETROS_CLIMA = stats_clima.copy()
    PARAMETROS_CLIMA['ddof'] = 0
    PARAMETROS_CLIMA['procedencia_ddof'] = 'Auditoria A06: reconstruccion historica'
    PARAMETROS_CLIMA.to_csv(OUTPUT_DIR / 'Parametros_Anomalias_Train.csv',index=False)

    df_clima = pd.merge(df_clima, stats_clima,
                        on=['jurisdiccion', 'mes_cal'], how='left')
    df_clima['precip_anom'] = (
        (df_clima['precip'] - df_clima['p_mean'])
        / df_clima['p_std']
    )
    df_clima['temp_anom'] = (
        (df_clima['temp'] - df_clima['t_mean'])
        / df_clima['t_std']
    )
    df_clima['precip_acum3m'] = (
        df_clima.groupby('jurisdiccion')['precip']
        .transform(lambda x: x.rolling(3, min_periods=3).sum())
    )
    df_clima['temp_ma3m'] = (
        df_clima.groupby('jurisdiccion')['temp']
        .transform(lambda x: x.rolling(3, min_periods=3).mean())
    )

    # -------------------------------------------------------------------------
    # 3. Ensamblaje del panel (ventana objetivo: Mar 2007 – Dic 2025)
    # -------------------------------------------------------------------------
    log("Ensamblando panel (2007-03 → 2025-12)",
        celda="Celda04_Panel_Maestro")
    fechas_obj = pd.date_range('2007-03-01', '2025-12-01', freq='MS')

    recaud_dict = panel_base.set_index(['jurisdiccion', 'fecha'])[
        ['recaud_real', 'var_real_yoy', 'caida_real']
    ].to_dict('index')
    clima_dict = df_clima.set_index(['jurisdiccion', 'fecha'])[
        ['precip', 'precip_anom', 'precip_acum3m',
         'temp', 'temp_anom', 'temp_ma3m']
    ].to_dict('index')

    cal_dict  = df_cal_g.set_index('fecha')['ind_renta_anual'].to_dict()
    pbi_dict  = df_pbi_g.set_index('fecha')['pbi_var'].to_dict()
    inf_dict = df_inflacion_g.set_index('fecha')['inflacion_mensual'].to_dict()
    tc_dict   = df_tc_g.set_index('fecha')['tipo_cambio'].to_dict()
    oni_dict  = df_oni_g.set_index('fecha_central')['oni'].to_dict()
    icen_dict = df_icen_g.set_index('fecha_central')['icen'].to_dict()

    registros = []
    for f_obj in fechas_obj:
        # Reglas de disponibilidad (Tabla 27 del proyecto)
        f_orig = f_obj - pd.DateOffset(months=1)
        f_pbi  = f_orig - pd.DateOffset(months=2)
        f_inf  = f_orig - pd.DateOffset(months=1)
        f_tc   = f_orig - pd.DateOffset(months=1)
        f_era  = f_orig - pd.DateOffset(months=3)
        f_oni  = f_orig - pd.DateOffset(months=4)
        f_icen = f_orig - pd.DateOffset(months=3)

        for jur in JURISDICCIONES_26:
            target = recaud_dict.get((jur, f_obj), {})
            row = {
                'jurisdiccion':   jur,
                'origen_t':       f_orig,
                'objetivo_t1':    f_obj,
                'anio_objetivo':  f_obj.year,
                'mes_objetivo':   f_obj.month,
                'Y_caida':        target.get('caida_real', np.nan),
                'g_var_real_t1':  target.get('var_real_yoy', np.nan),
            }
            # Historia tributaria (t-2 a t-13)
            for k in range(2, 14):
                f_lag = f_orig - pd.DateOffset(months=k)
                lag_info = recaud_dict.get((jur, f_lag), {})
                row[f'recaud_real_t_{k}'] = lag_info.get('recaud_real', np.nan)
                row[f'var_real_t_{k}']    = lag_info.get('var_real_yoy', np.nan)

            # Estacionalidad (dummies) y calendario
            for m_ind in range(2, 13):
                row[f'mes_obj_{m_ind}'] = 1.0 if f_obj.month == m_ind else 0.0
            row['ind_renta_anual'] = float(cal_dict[f_obj])

            # Controles económicos
            row['ctrl_pbi_var']        = pbi_dict.get(f_pbi, np.nan)
            row['ctrl_inflacion_lima'] = inf_dict.get(f_inf, np.nan)
            row['ctrl_tipo_cambio']    = tc_dict.get(f_tc,  np.nan)

            # Clima territorial (F1)
            c_info = clima_dict.get((jur, f_era), {})
            for var in ['precip', 'precip_anom', 'precip_acum3m',
                        'temp', 'temp_anom', 'temp_ma3m']:
                row[f'f1_{var}'] = c_info.get(var, np.nan)

            # ENOS (F2)
            row['f2_oni']  = oni_dict.get(f_oni,  np.nan)
            row['f2_icen'] = icen_dict.get(f_icen, np.nan)

            registros.append(row)

    df_panel = pd.DataFrame(registros)
    df_panel['fecha_disponible_Y'] = df_panel.objetivo_t1.map(disponibilidad_etiqueta)
    df_panel['corte_origen'] = df_panel.origen_t.map(corte_mes)
    df_panel['tipo_disponibilidad'] = 'Reconstruida-Seudorreal'

    # -------------------------------------------------------------------------
    # 4. Partición temporal
    # -------------------------------------------------------------------------
    log("Asignando bloques temporales (entrenamiento/validación/prueba)",
        celda="Celda04_Panel_Maestro")
    conds = [
        (df_panel['objetivo_t1'] >= '2007-03-01') &
        (df_panel['objetivo_t1'] <= '2019-12-01'),
        (df_panel['objetivo_t1'] >= '2020-01-01') &
        (df_panel['objetivo_t1'] <= '2022-12-01'),
        (df_panel['objetivo_t1'] >= '2023-01-01') &
        (df_panel['objetivo_t1'] <= '2025-12-01'),
    ]
    df_panel['bloque_temporal'] = np.select(
        conds, ['ENTRENAMIENTO', 'VALIDACION', 'PRUEBA'], default='OTRO'
    )

    # -------------------------------------------------------------------------
    # 5. Validaciones obligatorias de la muestra
    # -------------------------------------------------------------------------
    log("Validando muestra analítica", celda="Celda04_Panel_Maestro")
    n_total   = len(df_panel)
    n_validos = int(df_panel['Y_caida'].notna().sum())

    log(f"Total observaciones: {n_total} (esperado: 5,876)",
        celda="Celda03_Panel_Maestro")
    log(f"Y_caida válidos    : {n_validos} (esperado: 5,876)",
        celda="Celda03_Panel_Maestro")

    if n_validos != n_total:
        raise ValueError("Y_caida está vacío en df_panel. Revisa los merges.")

    resumen = df_panel.groupby('bloque_temporal').agg(
        Fecha_Inicio=('objetivo_t1', 'min'),
        Fecha_Fin   =('objetivo_t1', 'max'),
        N_Obs       =('Y_caida', 'size'),
        N_Validos   =('Y_caida', lambda x: x.notna().sum()),
    ).reset_index()
    display(resumen)

    # Verificación de partición esperada
    conteos = df_panel['bloque_temporal'].value_counts().to_dict()
    esperado = {'ENTRENAMIENTO': 4004, 'VALIDACION': 936, 'PRUEBA': 936}
    for bloque, exp in esperado.items():
        got = conteos.get(bloque, 0)
        if got != exp:
            log(f"ALERTA: {bloque} tiene {got} filas (esperado {exp})",
                nivel="WARN", celda="Celda03_Panel_Maestro")

    validar_panel(df_panel)
    auditar_muestras(df_panel)
    exportar_disponibilidad(df_panel)
    panel_base.loc[panel_base.caida_real.isna(), ['jurisdiccion','fecha']].to_csv(
        OUTPUT_DIR / 'Faltantes_Estructurales_Iniciales.csv',index=False)
    panel_base.loc[panel_base.fecha.eq('2013-02-01'),
        ['jurisdiccion','fecha','recaud_nominal','ipc_lima','recaud_real']].to_csv(
        OUTPUT_DIR / 'Deflactacion_Febrero2013.csv',index=False)
    df_clima.to_csv(OUTPUT_DIR / 'Clima_Niveles_Anomalias_Resumenes.csv',index=False)

    # -------------------------------------------------------------------------
    # 6. Exportar y registrar hashes
    # -------------------------------------------------------------------------
    log("Guardando panel y registrando hashes", celda="Celda04_Panel_Maestro")
    panel_path = OUTPUT_DIR / "Panel_Analitico_F0_F1_F2.csv"
    df_panel.to_csv(panel_path, index=False)

    hash_panel = registrar_hash(
        "Panel_Analitico_F0_F1_F2", df=df_panel,
        extra={"path": str(panel_path), "bloques": conteos}
    )
    registrar_hash("Panel_Base_Deflactado", df=panel_base,
                   extra={"filas": len(panel_base)})
    registrar_hash("Panel_Resumen_Bloques", df=resumen)

    log(f"HASH panel: {hash_panel[:16]}...", celda="Celda04_Panel_Maestro")
    log(f"PANEL MAESTRO COMPLETADO — {n_validos}/{n_total} válidos",
        celda="Celda04_Panel_Maestro")

    # Devolver resumen como metadata para el decorador
    return df_panel, panel_base

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
df_panel, panel_base = construir_panel_maestro(
    df_a13, df_ipc, df_inflacion, df_clima, df_cal, df_pbi, df_tc, df_oni, df_icen
)

# ==============================================================================
# MENSAJE FINAL
# ==============================================================================
print(f"\n{'='*70}")
print(f"PASO 04 COMPLETADA — PANEL LISTO PARA EDA")
print(f"{'='*70}")
print(f"df_panel  : {df_panel.shape}")
print(f"panel_base: {panel_base.shape}")
print(f"Y_caida   : {df_panel['Y_caida'].notna().sum():,} válidos")
print(f"Bloques   : {df_panel['bloque_temporal'].value_counts().to_dict()}")
print(f"\nAuditoría en: {AUDIT_DIR}")
print(f"Flag       : {FLAGS_DIR / 'Celda04_Panel_Maestro.OK.flag'}")

[2026-10-30T00:40:09.193559+00:00] [INFO] [Celda04_Panel_Maestro] INICIO construir_panel_maestro
[2026-10-30T00:40:09.196367+00:00] [INFO] [Celda04_Panel_Maestro] Iniciando construcción del panel maestro
[2026-10-30T00:40:09.196455+00:00] [INFO] [Celda04_Panel_Maestro] Deflactando recaudación con IPC Lima
[2026-10-30T00:40:09.236492+00:00] [INFO] [Celda03_Panel_Maestro] panel_base: 6552 filas | caida_real válidos: 6240
[2026-10-30T00:40:09.236685+00:00] [INFO] [Celda03_Panel_Maestro] Calculando anomalías climáticas (train-only hasta 2019-08)
[2026-10-30T00:40:10.391848+00:00] [INFO] [Celda04_Panel_Maestro] Ensamblando panel (2007-03 → 2025-12)
[2026-10-30T00:40:31.017885+00:00] [INFO] [Celda04_Panel_Maestro] Asignando bloques temporales (entrenamiento/validación/prueba)
[2026-10-30T00:40:31.020751+00:00] [INFO] [Celda04_Panel_Maestro] Validando muestra analítica
[2026-10-30T00:40:31.021190+00:00] [INFO] [Celda03_Panel_Maestro] Total observaciones: 5876 (esperado: 5,876)
[2026-10-30T00:

,bloque_temporal,Fecha_Inicio,Fecha_Fin,N_Obs,N_Validos
0,ENTRENAMIENTO,2007-03-01,2019-12-01,4004,4004
1,PRUEBA,2023-01-01,2025-12-01,936,936
2,VALIDACION,2020-01-01,2022-12-01,936,936


[2026-10-30T00:40:37.256031+00:00] [INFO] [Celda04_Panel_Maestro] Guardando panel y registrando hashes
[2026-10-30T00:40:39.487281+00:00] [INFO] [HASH] HASH registrado: Panel_Analitico_F0_F1_F2 = 325e17c68b0c2885...
[2026-10-30T00:40:39.579790+00:00] [INFO] [HASH] HASH registrado: Panel_Base_Deflactado = e2c7da68658f95ca...
[2026-10-30T00:40:39.581487+00:00] [INFO] [HASH] HASH registrado: Panel_Resumen_Bloques = bb7364373dd8d748...
[2026-10-30T00:40:39.582058+00:00] [INFO] [Celda04_Panel_Maestro] HASH panel: 325e17c68b0c2885...
[2026-10-30T00:40:39.582155+00:00] [INFO] [Celda04_Panel_Maestro] PANEL MAESTRO COMPLETADO — 5876/5876 válidos
[2026-10-30T00:40:39.593301+00:00] [INFO] [FLAG] FLAG creado: Celda04_Panel_Maestro.OK.flag
[2026-10-30T00:40:39.593383+00:00] [INFO] [Celda04_Panel_Maestro] FIN construir_panel_maestro — 30.40s

PASO 04 COMPLETADA — PANEL LISTO PARA EDA
df_panel  : (5876, 58)
panel_base: (6552, 8)
Y_caida   : 5,876 válidos
Bloques   : {'ENTRENAMIENTO': 4004, 'VALIDACIO

## Paso 05 — Preparación para el EDA (F0, F1 y F2)

Prepara lo que necesitará el EDA en la celda 06, **sin reconstruir datos**:

- **`W`**: tabla de pesos espaciales construida a partir de `INTER_PESOS` de la celda 03.
- **`QC`**: control de cobertura espacial por jurisdicción (`coverage_ratio`, `n_celdas_pesadas`, `suma_pesos`).
- **`CAT`**: catálogo de las 26 jurisdicciones con `jurisdiccion_id` canónico (`A13_<NOMBRE>`).
- **`Z`**: 312 grupos de parámetros Z (media y desviación estándar por jurisdicción × mes calendario), ajustados **solo con entrenamiento**.
- **`TRAIN_F1` / `VAL_F1`**: paneles de desarrollo con las columnas F1 renombradas (`P_mensual`, `z_P`, `P_3m`, `T_mensual`, `z_T`, `T_3m`).
- **`ET` / `EV`**: orígenes únicos por bloque para el análisis ENOS.

Todos los objetos se exponen como globales para la celda 06. Se registran hashes y el flag `Celda05_Preludio_F1F2.OK.flag`.


In [7]:
# ==============================================================================
# 05: CONSTRUCCION PARA EL EDA EXTENDIDO
# ==============================================================================
# Construye, a partir de INTER_PESOS (celda 03) y df_panel (celda 04):
#   - W      : tabla de pesos espaciales
#   - QC     : QC espacial por jurisdicción (coverage_ratio, n_celdas, suma_pesos)
#   - CAT    : catálogo de 26 jurisdicciones con jurisdiccion_id canónico
#   - Z      : 312 grupos de parámetros Z (media/sd por jurisdicción × mes calendario)
#   - ET/EV  : orígenes únicos por bloque (F2)
#   - TRAIN_F1 / VAL_F1 : paneles con columnas renombradas
#
# Todos los objetos quedan disponibles como GLOBALES para la celda 06.
# Si algún insumo no existe (p.ej. INTER_PESOS), se marca con warning y las
# figuras dependientes se omitirán limpiamente en la celda 06.
#
# Registra flag Celda05a_Preludio.OK.flag y hashes de W, QC, Z, CAT.
# ==============================================================================

# Ocultar únicamente el aviso de obsolescencia generado por Jupyter.=
import warnings

warnings.filterwarnings(
    action="ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

# ==============================================================================
# FUNCIÓN PRINCIPAL (decorada)
# ==============================================================================
@auditar("Celda05_Preludio_F1F2")
def construir_preludio_f1f2(df_panel, inter_pesos, df_oni, df_icen):
    log("Iniciando construcción de insumos para EDA",
        celda="Celda05_Preludio_F1F2")

    df_panel = df_panel.loc[df_panel.fecha_disponible_Y <= CORTE_SELECCION].copy()
    resultado = {
        'W': None, 'QC': None, 'CAT': None,
        'Z': None, 'TRAIN_F1': None, 'VAL_F1': None,
        'ET': None, 'EV': None,
        'omitidas': [],
    }

    # -------------------------------------------------------------------------
    # 1. Verificar disponibilidad de INTER_PESOS
    # -------------------------------------------------------------------------
    if inter_pesos is None:
        log("INTER_PESOS no disponible. Las figuras EDA_Fig15 y "
            "EDA_Fig16 se omitirán. Para generarlas, añade "
            "`global INTER_PESOS` en construir_clima_territorial_f1 "
            "(celda 03).",
            nivel="WARN", celda="Celda05_Preludio_F1F2")
        resultado['omitidas'].append('W/QC (cobertura y soporte espacial)')
    else:
        log(f"INTER_PESOS disponible: {inter_pesos.shape[0]} intersecciones, "
            f"{inter_pesos['jurisdiccion'].nunique()} jurisdicciones",
            celda="Celda05_Preludio_F1F2")

    # -------------------------------------------------------------------------
    # 2. Construir W
    # -------------------------------------------------------------------------
    if inter_pesos is not None:
        try:
            W = inter_pesos.rename(columns={
                'jurisdiccion':  'jurisdiccion_id',
                'lat_idx':       'lat_index',
                'lon_idx':       'lon_index',
                'weight':        'peso',
                'area':          'area_interseccion',
            }).copy()

            # grid_id: identificador único por fila (compatible con el 02)
            W = W.reset_index(drop=True)
            W['grid_id'] = ['G' + str(i).zfill(5) for i in range(len(W))]

            # Cobertura = área válida/área jurisdiccional; viene del Paso 03.
            if 'coverage_ratio' not in W: raise ValueError('Falta cobertura real')

            # Metadatos heredados
            W['version_cartografia'] = 'PROVINCIA.gpkg; hash del insumo de esta corrida'
            W['version_ERA5'] = 'ERA5-Land; metadatos GRIB de esta corrida'
            W['hash_fuente'] = (globals().get('HASH_TP_GRIB')
                                or 'NO_DOCUMENTADO')
            W['hash_cartografia'] = (globals().get('HASH_CART')
                                     or 'NO_DOCUMENTADO')

            # Columnas finales
            columnas_W = [
                'jurisdiccion_id', 'grid_id', 'lat_index', 'lon_index',
                'area_interseccion', 'peso', 'coverage_ratio',
                'version_cartografia', 'version_ERA5',
                'hash_fuente', 'hash_cartografia',
            ]
            W = W[columnas_W]

            # Validaciones mínimas
            assert W['peso'].gt(0).all(), "W: peso no positivo"
            assert W['area_interseccion'].gt(0).all(), "W: área no positiva"
            assert not W.duplicated(['jurisdiccion_id', 'grid_id']).any(), \
                "W: grid_id duplicado"
            assert np.allclose(
                W.groupby('jurisdiccion_id')['peso'].sum(), 1.0, atol=1e-9), \
                "W: suma de pesos distinta de 1"

            resultado['W'] = W
            log(f"W construido: {len(W)} filas, {len(W.columns)} columnas",
                celda="Celda05_Preludio_F1F2")
        except Exception as e:
            log(f"W no pudo construirse: {e}", nivel="WARN",
                celda="Celda05_Preludio_F1F2")
            resultado['omitidas'].append('W')

    # -------------------------------------------------------------------------
    # 3. Construir QC (QC espacial por jurisdicción)
    # -------------------------------------------------------------------------
    if resultado['W'] is not None:
        W = resultado['W']
        QC = W.groupby('jurisdiccion_id', sort=True).agg(
            coverage_ratio=('coverage_ratio', 'first'),
            n_celdas_pesadas=('grid_id', 'size'),
            suma_pesos=('peso', 'sum'),
        ).reset_index()
        resultado['QC'] = QC
        log(f"QC construido: {len(QC)} jurisdicciones",
            celda="Celda05_Preludio_F1F2")

    # -------------------------------------------------------------------------
    # 4. Construir CAT (catálogo de 26 jurisdicciones)
    # -------------------------------------------------------------------------
    try:
        # Intentar reutilizar CAT global si ya existe (de la celda 04)
        CAT = globals().get('CAT', None)
        if CAT is None:
            # Construir desde df_panel
            jur_unicas = sorted(df_panel['jurisdiccion'].dropna().unique())
            CAT = pd.DataFrame({
                'jurisdiccion_id': [
                    j
                    for j in jur_unicas
                ],
                'nombre_jurisdiccion': jur_unicas,
            })
            # Verificación: 26 jurisdicciones esperadas
            assert len(CAT) == 26, \
                f"CAT: se esperaban 26 jurisdicciones, hay {len(CAT)}"

            # Mapeo inverso jurisdiccion → jurisdiccion_id
            MAPEO_JUR = dict(zip(
                CAT['nombre_jurisdiccion'], CAT['jurisdiccion_id']))

            resultado['CAT'] = CAT
            resultado['MAPEO_JUR'] = MAPEO_JUR
            log(f"CAT construido: {len(CAT)} jurisdicciones",
                celda="Celda05_Preludio_F1F2")
        else:
            resultado['CAT'] = CAT
            resultado['MAPEO_JUR'] = dict(zip(
                CAT['nombre_jurisdiccion'],
                CAT['jurisdiccion_id']
            )) if 'jurisdiccion_id' in CAT.columns else None
            log("CAT reutilizado del scope global",
                celda="Celda05_Preludio_F1F2")
    except Exception as e:
        log(f"CAT no pudo construirse: {e}", nivel="WARN",
            celda="Celda05_Preludio_F1F2")
        resultado['omitidas'].append('CAT')

    # -------------------------------------------------------------------------
    # 5. Construir TRAIN_F1 / VAL_F1 (paneles con columnas renombradas)
    # -------------------------------------------------------------------------
    MAPEO_F1 = {
        'f1_precip':        'P_mensual',
        'f1_precip_anom':   'z_P',
        'f1_precip_acum3m': 'P_3m',
        'f1_temp':          'T_mensual',
        'f1_temp_anom':     'z_T',
        'f1_temp_ma3m':     'T_3m',
    }
    df_dev = df_panel[df_panel['bloque_temporal'].isin(
        ['ENTRENAMIENTO', 'VALIDACION'])].copy()

    # Solo renombrar columnas que existan
    renombrar = {k: v for k, v in MAPEO_F1.items()
                 if k in df_dev.columns}
    df_dev_ren = df_dev.rename(columns=renombrar)

    TRAIN_F1 = df_dev_ren[
        df_dev_ren['bloque_temporal'] == 'ENTRENAMIENTO'].copy()
    VAL_F1   = df_dev_ren[
        df_dev_ren['bloque_temporal'] == 'VALIDACION'].copy()

    resultado['TRAIN_F1'] = TRAIN_F1
    resultado['VAL_F1']   = VAL_F1
    cols_presentes = list(renombrar.values())
    log(f"TRAIN_F1: {len(TRAIN_F1)} filas | VAL_F1: {len(VAL_F1)} filas | "
        f"variables climáticas renombradas: {cols_presentes}",
        celda="Celda05_Preludio_F1F2")

    # -------------------------------------------------------------------------
    # 6. Construir ET / EV (orígenes únicos por bloque, para F2)
    # -------------------------------------------------------------------------
    if 'origen_t' in df_dev.columns:
        ET = (TRAIN_F1.drop_duplicates('origen_t')
              .sort_values('origen_t').copy())
        EV = (VAL_F1.drop_duplicates('origen_t')
              .sort_values('origen_t').copy())
        resultado['ET'] = ET
        resultado['EV'] = EV
        log(f"ET: {len(ET)} orígenes únicos | EV: {len(EV)} orígenes únicos",
            celda="Celda05_Preludio_F1F2")
    else:
        log("Columna 'origen_t' no encontrada; ET/EV no se construyen",
            nivel="WARN", celda="Celda05_Preludio_F1F2")
        resultado['omitidas'].append('ET/EV (series ENOS)')

    # -------------------------------------------------------------------------
    # 7. Construir Z (312 grupos de parámetros Z)
    # -------------------------------------------------------------------------
    # A06: exponer los parámetros realmente usados por el panel, sin reestimación.
    Z = PARAMETROS_CLIMA.rename(columns={'jurisdiccion':'jurisdiccion_id',
        'mes_cal':'mes_calendario_clima','p_mean':'media_P','p_std':'sd_P',
        't_mean':'media_T','t_std':'sd_T'}).copy()
    Z['bloque_estimacion'] = 'ENTRENAMIENTO_ADMISIBLE'
    resultado['Z'] = Z
    if resultado['omitidas']: raise ValueError(f"Insumos EDA incompletos: {resultado['omitidas']}")

    # -------------------------------------------------------------------------
    # 8. Registro de hashes y flag
    # -------------------------------------------------------------------------
    for nombre, df in [
        ('Preludio_W',  resultado['W']),
        ('Preludio_QC', resultado['QC']),
        ('Preludio_CAT', resultado['CAT']),
        ('Preludio_Z',  resultado['Z']),
    ]:
        if df is not None:
            registrar_hash(nombre, df=df)

    if resultado['omitidas']:
        log(f"Insumos omitidos (figuras dependientes no se generarán): "
            f"{resultado['omitidas']}",
            nivel="WARN", celda="Celda05_Preludio_F1F2")
    else:
        log("Todos los insumos F1/F2 reconstruidos con éxito",
            celda="Celda05_Preludio_F1F2")

    return resultado


# ==============================================================================
# EJECUCIÓN
# ==============================================================================
preludio = construir_preludio_f1f2(
    df_panel,
    globals().get('INTER_PESOS', None),
    globals().get('df_oni', None),
    globals().get('df_icen', None),
)

# Exponer objetos como GLOBALES para la celda 05 extendida
W        = preludio['W']
QC       = preludio['QC']
CAT      = preludio['CAT']
Z        = preludio['Z']
TRAIN_F1 = preludio['TRAIN_F1']
VAL_F1   = preludio['VAL_F1']
ET       = preludio['ET']
EV       = preludio['EV']
MAPEO_JUR = preludio.get('MAPEO_JUR', None)

# ==============================================================================
# FLAG DE CIERRE
# ==============================================================================
crear_flag("Celda05_Preludio_F1F2", "OK",
           {"W_filas":    len(W) if W is not None else 0,
            "QC_filas":   len(QC) if QC is not None else 0,
            "CAT_filas":  len(CAT) if CAT is not None else 0,
            "Z_grupos":   len(Z) if Z is not None else 0,
            "ET_origenes":len(ET) if ET is not None else 0,
            "EV_origenes":len(EV) if EV is not None else 0,
            "omitidas":   preludio['omitidas']})

# ==============================================================================
# RESUMEN
# ==============================================================================
print(f"\n{'='*70}")
print(f"PASO 05 COMPLETADO")
print(f"{'='*70}")
print(f"W       : {W.shape if W is not None else 'NO DISPONIBLE'}")
print(f"QC      : {QC.shape if QC is not None else 'NO DISPONIBLE'}")
print(f"CAT     : {CAT.shape if CAT is not None else 'NO DISPONIBLE'}")
print(f"Z       : {Z.shape if Z is not None else 'NO DISPONIBLE'}")
print(f"TRAIN_F1: {TRAIN_F1.shape if TRAIN_F1 is not None else 'NO DISPONIBLE'}")
print(f"VAL_F1  : {VAL_F1.shape if VAL_F1 is not None else 'NO DISPONIBLE'}")
print(f"ET      : {ET.shape if ET is not None else 'NO DISPONIBLE'}")
print(f"EV      : {EV.shape if EV is not None else 'NO DISPONIBLE'}")
if preludio['omitidas']:
    print(f"\n⚠ Insumos omitidos: {preludio['omitidas']}")
    print(f"   Las figuras dependientes se omitirán en la celda 06.")
else:
    print(f"\n✓ Todos los insumos construidos. La celda 06 puede generar "
          f"las figuras necesarias.")
print(f"\nAuditoría: {AUDIT_DIR}")

[2026-10-30T00:42:16.743139+00:00] [INFO] [Celda05_Preludio_F1F2] INICIO construir_preludio_f1f2
[2026-10-30T00:42:16.743558+00:00] [INFO] [Celda05_Preludio_F1F2] Iniciando construcción de insumos para EDA
[2026-10-30T00:42:16.749743+00:00] [INFO] [Celda05_Preludio_F1F2] INTER_PESOS disponible: 12299 intersecciones, 26 jurisdicciones
[2026-10-30T00:42:16.767726+00:00] [INFO] [Celda05_Preludio_F1F2] W construido: 12299 filas, 11 columnas
[2026-10-30T00:42:16.776003+00:00] [INFO] [Celda05_Preludio_F1F2] QC construido: 26 jurisdicciones
[2026-10-30T00:42:16.777449+00:00] [INFO] [Celda05_Preludio_F1F2] CAT construido: 26 jurisdicciones
[2026-10-30T00:42:16.784870+00:00] [INFO] [Celda05_Preludio_F1F2] TRAIN_F1: 4004 filas | VAL_F1: 884 filas | variables climáticas renombradas: ['P_mensual', 'z_P', 'P_3m', 'T_mensual', 'z_T', 'T_3m']
[2026-10-30T00:42:16.787914+00:00] [INFO] [Celda05_Preludio_F1F2] ET: 154 orígenes únicos | EV: 34 orígenes únicos
[2026-10-30T00:42:16.967926+00:00] [INFO] [HA

## Paso 06 — EDA descriptivo (OE1)

Realiza el **análisis exploratorio descriptivo** usando exclusivamente **Entrenamiento + Validación** (2007-03 a 2022-12). Nunca toca la prueba 2023-2025. Genera **46 figuras** y **3 tablas CSV** organizadas en cuatro bloques:

*   Bloque A — Panel base (Fig 1–9): Balance de clases, frecuencia anual y territorial de caídas, distribución de `g`, boxplots por jurisdicción, estacionalidad y recaudación real anual.

*   Bloque B — Clima territorial F1 (Fig 10–25): Histogramas de `z_P` y `z_T`, series de Amazonas, correlaciones climáticas, cobertura y soporte espacial, boxplots por jurisdicción, estacionalidad climática, banderas IQR, matrices de anomalías y autocorrelación.

*   Bloque C — Índices ENOS F2 (Fig 26–40): Partición temporal, series ONI/ICEN, distribuciones, ECDF, boxplots por bloque, scatter ONI-ICEN, correlación clima-ENOS, mapa temporal, banderas IQR, outliers y balance del objetivo.

*   Bloque D — Figuras y tablas adicionales: Cobertura temporal, Serie mensual, Meses de campaña Renta Anual por año, Panel 1×2: inflación mensual + tipo de cambio PEN/USD. Estadísticos descriptivos de 5 variable. Comparación Entrenamiento vs Validación por variable.

Todas las figuras llevan etiquetas de extremos y notas aclaratorias. Se guardan en PNG 300 dpi y se registra el flag `Celda06_EDA.OK.flag`.

In [8]:
# ==============================================================================
# 06: EDA DESCRIPTIVO (OE1) — VERSIÓN EXTENDIDA CON FIGURAS F1 Y F2
# ==============================================================================
# Requiere que la celda 05 se haya ejecutado antes.
# Usa los globales: W, QC, CAT, Z, TRAIN_F1, VAL_F1, ET, EV.
#
# Contenido:
#   - Bloque A (EDA_Fig1 … EDA_Fig9)   : panel base (recaudación, Y, g)
#   - Bloque B (EDA_Fig10 … EDA_Fig25) : F1 (clima territorial)
#   - Bloque C (EDA_Fig26 … EDA_Fig40) : F2 (ENOS)
#   - Bloque D : Figuras adicionales
# Todas usan SOLO Entrenamiento + Validación (2007-03 → 2022-12).
# Nunca tocan la prueba 2023-2025.

# ==============================================================================
# 06: FILTROS ESPECÍFICOS PARA LAS ADVERTENCIAS OBSERVADAS
# ==============================================================================
import warnings
from matplotlib import MatplotlibDeprecationWarning
from pyparsing import PyparsingDeprecationWarning

# Ocultar únicamente el aviso de obsolescencia generado por Jupyter.
warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

# Matplotlib: cambio de nombre de labels en boxplot().
warnings.filterwarnings(
    action="ignore",
    message=(
        r"The 'labels' parameter of boxplot\(\) has been "
        r"renamed 'tick_labels'.*"
    ),
    category=MatplotlibDeprecationWarning,
)

# Matplotlib: futura sustitución del parámetro vert.
warnings.filterwarnings(
    action="ignore",
    message=(
        r"vert: bool will be deprecated in a future version\."
        r" Use orientation:.*"
    ),
    category=PendingDeprecationWarning,
)

# Matplotlib/Pyparsing: avisos internos observados al dibujar.
warnings.filterwarnings(
    action="ignore",
    message=(
        r"'(?:parseString|resetCache|oneOf|leaveWhitespace|"
        r"setName|setParseAction)' deprecated.*"
    ),
    category=PyparsingDeprecationWarning,
    module=r"^matplotlib\.(?:_fontconfig_pattern|_mathtext)$",
)

warnings.filterwarnings(
    action="ignore",
    message=(
        r"'(?:endQuoteChar|unquoteResults)' argument is deprecated.*"
    ),
    category=PyparsingDeprecationWarning,
    module=r"^matplotlib\._mathtext$",
)

warnings.filterwarnings(
    action="ignore",
    message=r"'parseAll' argument is deprecated.*",
    category=PyparsingDeprecationWarning,
    module=r"^pyparsing\.util$",
)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 170,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white',
})

AZUL   = '#2563EB'
VERDE  = '#0D9488'
ROJO   = '#E45756'
MORADO = '#7C3AED'
AMBAR  = '#D97706'
GRIS   = '#64748B'
MESES_NOMBRE = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']

UNIDADES = {
    'P_mensual': 'mm/mes', 'P_3m': 'mm/3 meses',
    'T_mensual': '°C', 'T_3m': '°C',
    'z_P': 'adimensional', 'z_T': 'adimensional',
}

EDA_DIR = OUTPUT_DIR / "EDA_Descriptivo"
EDA_DIR.mkdir(parents=True, exist_ok=True)
FIGURAS_META = []


def guardar_figura(fig, nombre, nota=""):
    ruta = EDA_DIR / f"{nombre}.png"
    fig.savefig(ruta, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    FIGURAS_META.append({'archivo': ruta.name, 'nota': nota})
    print(f"  ✓ {ruta.name}")
    return ruta


def nota_fig(fig, texto):
    if texto:
        fig.text(0.01, -0.018, texto, fontsize=9, color='#475569',
                 va='top', ha='left')


# ==============================================================================
# FUNCIÓN PRINCIPAL
# ==============================================================================
@auditar("Celda06_EDA")
def ejecutar_eda_descriptivo(df_panel, panel_base):
    log("Iniciando EDA descriptivo extendido (OE1 + F1 + F2)",
        celda="Celda06_EDA")

    df_panel = df_panel.loc[df_panel.fecha_disponible_Y <= CORTE_SELECCION].copy()
    panel_base = panel_base.loc[panel_base.fecha <= pd.Timestamp("2022-10-01")].copy()
    # Filtro estricto: solo entrenamiento y validación
    df_dev = df_panel[df_panel['bloque_temporal'].isin(
        ['ENTRENAMIENTO', 'VALIDACION'])].copy()
    assert df_dev['objetivo_t1'].max() <= pd.Timestamp('2022-12-01'), \
        "EDA no debe tocar la prueba 2023-2025"

    log(f"Filas usadas: {len(df_dev)} (Entrenamiento + Validación)",
        celda="Celda06_EDA")
    log(f"Rango temporal: {df_dev['objetivo_t1'].min().date()} → "
        f"{df_dev['objetivo_t1'].max().date()}",
        celda="Celda06_EDA")

    # =========================================================================
    # BLOQUE A — 9 FIGURAS BASE DEL PANEL (EDA_Fig1 … EDA_Fig9)
    # =========================================================================
    log("Bloque A: figuras base del panel", celda="Celda06_EDA")

    # FIG 1 y 2: Balance de clases por bloque
    for bloque, color, num in [('ENTRENAMIENTO', AZUL, 1),
                                ('VALIDACION', AMBAR, 2)]:
        sub = df_panel[df_panel['bloque_temporal'] == bloque]
        conteo = sub['Y_caida'].value_counts().reindex([0, 1]).fillna(0)
        pct = 100 * conteo / len(sub)

        fig, ax = plt.subplots(figsize=(8, 5))
        barras = ax.bar(['Sin caída (0)', 'Caída (1)'], conteo.values,
                        color=[color, ROJO], edgecolor='white', width=0.55)
        for b, v, p in zip(barras, conteo.values, pct.values):
            ax.text(b.get_x() + b.get_width() / 2,
                    v + max(conteo.values) * 0.03,
                    f'{int(v):,}\n{p:.2f}%',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
        ax.set_ylabel('Número de observaciones')
        ax.set_title(f'Balance de clases · {bloque}\n'
                     f'(Y=1 si la variación real interanual es negativa)',
                     loc='left', pad=12)
        ax.set_ylim(0, max(conteo.values) * 1.25)
        guardar_figura(fig, f"EDA_Fig{num}_Balance_{bloque}")

    # FIG 3: Frecuencia anual de caídas
    anual = df_dev.groupby('anio_objetivo', as_index=False).agg(
        n_obs=('Y_caida', 'size'), n_caidas=('Y_caida', 'sum'))
    anual['frecuencia'] = anual['n_caidas'] / anual['n_obs']

    fig, ax = plt.subplots(figsize=(13, 6))
    colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(anual)))
    barras = ax.bar(anual['anio_objetivo'].astype(str),
                    anual['frecuencia'] * 100, color=colors)
    for b, v in zip(barras, anual['frecuencia'] * 100):
        ax.text(b.get_x() + b.get_width() / 2, v + 1.5,
                f'{v:.1f}%', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    max_row = anual.loc[anual['frecuencia'].idxmax()]
    min_row = anual.loc[anual['frecuencia'].idxmin()]
    ax.annotate(f"Máx: {int(max_row['anio_objetivo'])}\n"
                f"{max_row['frecuencia']*100:.2f}%",
                (str(max_row['anio_objetivo']), max_row['frecuencia'] * 100),
                xytext=(0, 20), textcoords='offset points', ha='center',
                fontsize=9, fontweight='bold', color=ROJO,
                bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=ROJO))
    ax.annotate(f"Mín: {int(min_row['anio_objetivo'])}\n"
                f"{min_row['frecuencia']*100:.2f}%",
                (str(min_row['anio_objetivo']), min_row['frecuencia'] * 100),
                xytext=(0, -30), textcoords='offset points', ha='center',
                fontsize=9, fontweight='bold', color=VERDE,
                bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=VERDE))
    ax.set_ylabel('Jurisdicciones con caída (%)')
    ax.set_xlabel('Año objetivo')
    ax.set_ylim(0, anual['frecuencia'].max() * 100 * 1.3)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title('Frecuencia Anual de Caídas Interanuales Reales\n'
                 '(Entrenamiento + Validación)', loc='left', pad=12)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig3_Frecuencia_Anual_Caidas")

    # FIG 4: Frecuencia territorial
    territ = df_dev.groupby('jurisdiccion', as_index=False).agg(
        n_obs=('Y_caida', 'size'), n_caidas=('Y_caida', 'sum'))
    territ['frecuencia'] = territ['n_caidas'] / territ['n_obs']
    territ = territ.sort_values('frecuencia', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 10))
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(territ)))
    barras = ax.barh(territ['jurisdiccion'], territ['frecuencia'] * 100,
                     color=colors, height=0.75)
    for b, v in zip(barras, territ['frecuencia'] * 100):
        ax.text(v + 1.5, b.get_y() + b.get_height() / 2,
                f'{v:.2f}%', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Porcentaje de meses con caída')
    ax.set_ylabel('Jurisdicción fiscal')
    ax.set_xlim(0, territ['frecuencia'].max() * 100 * 1.2)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'Frecuencia Territorial de Caídas · {len(df_dev)} obs',
                 loc='left', pad=12)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig4_Frecuencia_Territorial")

    # FIG 5: Distribución de g real
    g_vals = df_dev['g_var_real_t1'].dropna()
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.hist(g_vals, bins=60, color=AZUL, alpha=0.85, edgecolor='white')
    ax.axvline(0, color=ROJO, linestyle='-', linewidth=1.5,
               label='Umbral Y=1 (caída)')
    ax.axvline(g_vals.median(), color=VERDE, linestyle='--', linewidth=1.5,
               label=f'Mediana = {g_vals.median():.2f}')
    ax.text(0.98, 0.95,
            f'n = {len(g_vals):,}\nMedia = {g_vals.mean():.2f}\n'
            f'Desv. = {g_vals.std():.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#CBD5E1'))
    ax.set_xlabel('Variación real interanual (puntos logarítmicos × 100)')
    ax.set_ylabel('Número de observaciones')
    ax.set_title('Distribución de g real · Entrenamiento + Validación',
                 loc='left', pad=12)
    ax.legend(loc='upper left')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig5_Distribucion_g")

    # FIG 6: Boxplot recaudación
    fig, ax = plt.subplots(figsize=(12, 11))
    data_por_jur = [
        df_dev[df_dev['jurisdiccion'] == j]['recaud_real_t_2'] / 1e6
        for j in JURISDICCIONES_26
    ]
    bp = ax.boxplot(data_por_jur, vert=False, patch_artist=True,
                    labels=JURISDICCIONES_26,
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    for patch, c in zip(bp['boxes'],
                        plt.cm.viridis(np.linspace(0.1, 0.9, 26))):
        patch.set_facecolor(c)
    ax.set_xscale('log')
    ax.set_xlabel('Millones de S/ constantes (dic-2021) · eje logarítmico')
    ax.set_ylabel('Jurisdicción fiscal')
    ax.set_title('Recaudación Real por Jurisdicción · '
                 'Entrenamiento + Validación', loc='left', pad=12)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig6_Boxplot_Recaudacion")

    # FIG 7: Boxplot variación real
    fig, ax = plt.subplots(figsize=(12, 11))
    data_g = [
        df_dev[df_dev['jurisdiccion'] == j]['g_var_real_t1'].dropna()
        for j in JURISDICCIONES_26
    ]
    bp = ax.boxplot(data_g, vert=False, patch_artist=True,
                    labels=JURISDICCIONES_26,
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    for patch, c in zip(bp['boxes'],
                        plt.cm.viridis(np.linspace(0.1, 0.9, 26))):
        patch.set_facecolor(c)
    ax.axvline(0, color=ROJO, linestyle='--', linewidth=1)
    ax.set_xlabel('Variación real interanual (puntos logarítmicos × 100)')
    ax.set_ylabel('Jurisdicción fiscal')
    ax.set_title('Variación Real Interanual por Jurisdicción\n'
                 '(Entrenamiento + Validación)', loc='left', pad=12)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig7_Boxplot_Variacion")

    # FIG 8: Estacionalidad
    data_mes = [
        df_dev[df_dev['mes_objetivo'] == m]['g_var_real_t1'].dropna()
        for m in range(1, 13)
    ]
    fig, ax = plt.subplots(figsize=(12, 5))
    bp = ax.boxplot(data_mes, patch_artist=True, labels=MESES_NOMBRE,
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    for patch, c in zip(bp['boxes'],
                        plt.cm.viridis(np.linspace(0.1, 0.9, 12))):
        patch.set_facecolor(c)
    ax.axhline(0, color=ROJO, linestyle='--', linewidth=1.2)
    for i, d in enumerate(data_mes, 1):
        med = d.median()
        ax.text(i, med, f'{med:.2f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='#CBD5E1', alpha=0.9))
    ax.set_xlabel('Mes calendario')
    ax.set_ylabel('Variación real interanual (puntos × 100)')
    ax.set_title('Estacionalidad de la Variación Real · '
                 'Entrenamiento + Validación', loc='left', pad=12)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig8_Estacionalidad")

    # FIG 9: Recaudación real total anual
    panel_base_dev = panel_base[
        (panel_base['fecha'] >= '2007-03-01') &
        (panel_base['fecha'] <= '2022-12-01')
    ].copy()
    panel_base_dev['anio'] = panel_base_dev['fecha'].dt.year
    recaud_anual = panel_base_dev.groupby('anio', as_index=False)[
        'recaud_real'].sum()
    recaud_anual['millones'] = recaud_anual['recaud_real'] / 1e6

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(recaud_anual['anio'], recaud_anual['millones'],
            color='#24547A', lw=2.5, marker='o', markersize=8)
    for _, row in recaud_anual.iterrows():
        ax.annotate(f"{row['millones']:.1f}",
                    (row['anio'], row['millones']),
                    xytext=(0, 12), textcoords='offset points',
                    ha='center', fontsize=8.5, fontweight='bold',
                    color='#24547A')
    ax.fill_between(recaud_anual['anio'], recaud_anual['millones'], 0,
                    color='#25A7B8', alpha=0.1)
    max_row = recaud_anual.loc[recaud_anual['millones'].idxmax()]
    ax.annotate(f"Máx: {int(max_row['anio'])}\n"
                f"{max_row['millones']:.2f} M S/",
                (max_row['anio'], max_row['millones']),
                xytext=(0, 22), textcoords='offset points', ha='center',
                fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', fc='white',
                          ec='#24547A'))
    ax.set_xlabel('Año')
    ax.set_ylabel('Millones de S/ constantes (dic-2021)')
    ax.set_title('Recaudación Real Total Anual · '
                 'Entrenamiento + Validación', loc='left', pad=12)
    ax.set_xticks(recaud_anual['anio'])
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    guardar_figura(fig, "EDA_Fig9_Recaudacion_Anual")

    # =========================================================================
    # BLOQUE B — FIGURAS F1 (EDA_Fig10 … EDA_Fig25)
    # =========================================================================
    log("Bloque B: figuras F1 (clima)", celda="Celda06_EDA")

    if TRAIN_F1 is None or VAL_F1 is None:
        log("BLOQUE B OMITIDO: TRAIN_F1/VAL_F1 no disponibles (¿corriste "
            "la celda 05?)", nivel="WARN", celda="Celda06_EDA")
    else:
        cols_f1 = [c for c in ['P_mensual', 'z_P', 'P_3m',
                                'T_mensual', 'z_T', 'T_3m']
                   if c in TRAIN_F1.columns]

        # FIG 10 y 11: Histogramas z_P y z_T
        for num, c, color in [(10, 'z_P', AZUL), (11, 'z_T', MORADO)]:
            if c not in TRAIN_F1.columns:
                log(f"EDA_Fig{num} OMITIDA: falta {c}", nivel="WARN",
                    celda="Celda06_EDA"); continue
            d = TRAIN_F1[c].dropna()
            fig, ax = plt.subplots(figsize=(11, 4.5))
            ax.hist(d, bins=40, color=color, edgecolor='white', alpha=.9)
            ax.axvline(0, color=VERDE, ls='--', label='Cero de referencia')
            for v in [d.min(), d.max()]:
                ax.axvline(v, color=ROJO, ls=':', lw=1.5)
            ax.text(.98, .96, f'Mín: {d.min():.4f}\nMáx: {d.max():.4f}',
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=10, bbox={'facecolor': 'white', 'alpha': .9,
                                        'edgecolor': '#CBD5E1'})
            ax.set(xlabel=c + ' (adimensional)', ylabel='Observaciones')
            ax.set_title(f'Distribución de {c} · entrenamiento',
                         loc='left', pad=12)
            ax.legend(loc='upper left')
            nota_fig(fig, f'{len(d)} observaciones. La estandarización por '
                          f'jurisdicción y mes no implica normalidad. '
                          f'Todos los extremos se conservan.')
            guardar_figura(fig, f"EDA_Fig{num}_Histograma_{c}")

        # FIG 12 y 13: Serie Tumbes P/T
        if 'jurisdiccion' in TRAIN_F1.columns and \
           'origen_t' in TRAIN_F1.columns:
            TUM = (TRAIN_F1[TRAIN_F1['jurisdiccion'] == 'TUMBES']
                    .sort_values('origen_t'))
            for num, c, color in [(12, 'P_mensual', AZUL),
                                   (13, 'T_mensual', ROJO)]:
                if c not in TUM.columns or len(TUM) == 0:
                    log(f"EDA_Fig{num} OMITIDA: sin Tumbes/{c}",
                        nivel="WARN", celda="Celda06_EDA"); continue
                fig, ax = plt.subplots(figsize=(13, 5))
                ax.plot(pd.to_datetime(TUM['origen_t']), TUM[c],
                        color=color, lw=1.8)
                for tipo, v, dy in [('Mín', TUM[c].min(), -35),
                                     ('Máx', TUM[c].max(), 24)]:
                    q = TUM[TUM[c] == v]
                    for _, r in q.iterrows():
                        ax.scatter(pd.Timestamp(r['origen_t']), v,
                                   color=AMBAR, s=48, zorder=4)
                        ax.annotate(
                            f'{tipo}: {v:.4f}\n'
                            f'{pd.Timestamp(r["origen_t"]).strftime("%Y-%m")}',
                            (pd.Timestamp(r['origen_t']), v),
                            xytext=(0, dy), textcoords='offset points',
                            ha='center', fontsize=9,
                            arrowprops={'arrowstyle': '-', 'color': AMBAR},
                            bbox={'facecolor': 'white', 'alpha': .85,
                                  'edgecolor': 'none'})
                ax.margins(x=.04, y=.28)
                ax.set(xlabel='Mes fuente (t−3)',
                       ylabel=f"{c} ({UNIDADES[c]})")
                ax.set_title(f'Tumbes · {c} · entrenamiento',
                             loc='left', pad=12)
                nota_fig(fig, 'El valor graficado corresponde al clima observado 3 meses antes del origen y 4 meses antes del objetivo.')
                guardar_figura(fig, f"EDA_Fig{num}_Tumbes_{c}")

        # FIG 14: Correlación Pearson clima
        if len(cols_f1) >= 3:
            COR = TRAIN_F1[cols_f1].corr(method='pearson')
            pares = [(cols_f1[i], cols_f1[j], COR.iloc[i, j])
                     for i in range(len(cols_f1))
                     for j in range(i + 1, len(cols_f1))]
            pmin = min(pares, key=lambda x: x[2])
            pmax = max(pares, key=lambda x: x[2])
            fig, ax = plt.subplots(figsize=(9, 7))
            im = ax.imshow(COR, vmin=-1, vmax=1, cmap='RdBu_r')
            ax.set_xticks(range(len(cols_f1)), cols_f1,
                          rotation=30, ha='right')
            ax.set_yticks(range(len(cols_f1)), cols_f1)
            ax.grid(False)
            for i in range(len(cols_f1)):
                for j in range(len(cols_f1)):
                    v = COR.iloc[i, j]
                    ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                            color='white' if abs(v) > .6 else '#111827')
            fig.colorbar(im, ax=ax, label='Pearson r', shrink=.8)
            ax.set_title('Correlaciones de los predictores climáticos · '
                         'entrenamiento', loc='left', pad=12)
            nota_fig(fig, f'Fuera de diagonal: mínimo {pmin[0]}–{pmin[1]} '
                          f'= {pmin[2]:.4f}; máximo {pmax[0]}–{pmax[1]} = '
                          f'{pmax[2]:.4f}. La correlación no es causalidad '
                          f'ni rendimiento predictivo.')
            guardar_figura(fig, "EDA_Fig14_Correlacion_Pearson_Clima")

        # FIG 15 y 16: Cobertura y soporte espacial (requieren W/QC)
        if QC is not None and CAT is not None:
            # Aseguramos el nombre de la columna de jurisdicción
            QC_ren = QC.rename(columns={'jurisdiccion': 'jurisdiccion_id'}) \
                if 'jurisdiccion' in QC.columns and 'jurisdiccion_id' not in QC.columns \
                else QC.copy()

            CAT_ren = CAT.rename(columns={'jurisdiccion': 'jurisdiccion_id'}) \
                if 'jurisdiccion' in CAT.columns and 'jurisdiccion_id' not in CAT.columns \
                else CAT.copy()

            # Merge tolerante
            qc = QC_ren.merge(CAT_ren, on='jurisdiccion_id',
                              how='left').sort_values('coverage_ratio')

            # Usar la columna de nombre si existe, si no usar el ID
            col_y = 'nombre_jurisdiccion' if 'nombre_jurisdiccion' in qc.columns else 'jurisdiccion_id'

            # FIG 15: cobertura
            fig, ax = plt.subplots(figsize=(11, 10))
            bars = ax.barh(qc[col_y],
                           qc['coverage_ratio'] * 100,
                           color=plt.cm.viridis(
                               np.linspace(.1, .9, len(qc))))
            ax.bar_label(bars, fmt='%.2f%%', padding=3, fontsize=9)
            ax.set(xlim=(0, 111), xlabel='Área con soporte terrestre válido (%)')
            ax.axvline(100, color=ROJO, ls='--')
            ax.set_title('Cobertura espacial del soporte válido',
                         loc='left', pad=12)
            nota_fig(fig, f'Mínimo: {qc.coverage_ratio.min()*100:.3f}%; '
                          f'máximo: {qc.coverage_ratio.max()*100:.8f}%. '
                          f'Área válida / área jurisdiccional; pesos normalizados por separado.')
            guardar_figura(fig, "EDA_Fig15_Cobertura_Espacial")

            # FIG 16: soporte
            qc = qc.sort_values('n_celdas_pesadas')
            fig, ax = plt.subplots(figsize=(11, 10))
            bars = ax.barh(qc[col_y],
                           qc['n_celdas_pesadas'],
                           color=plt.cm.plasma(
                               np.linspace(.15, .8, len(qc))))
            ax.bar_label(bars, padding=3, fontsize=9)
            ax.set(xlabel='Celdas/intersecciones con peso',
                   xlim=(0, qc.n_celdas_pesadas.max() * 1.18))
            ax.set_title('Soporte espacial por jurisdicción',
                         loc='left', pad=12)
            nota_fig(fig, f'Mínimo: {qc.n_celdas_pesadas.min()} celdas; '
                          f'máximo: {qc.n_celdas_pesadas.max()}.')
            guardar_figura(fig, "EDA_Fig16_Soporte_Espacial")
        else:
            log("EDA_Fig15 y EDA_Fig16 OMITIDAS: faltan W/QC",
                nivel="WARN", celda="Celda06_EDA")

        # FIG 17: Distribución de los 4 predictores directos
        directos = [c for c in ['P_mensual', 'P_3m', 'T_mensual', 'T_3m']
                    if c in TRAIN_F1.columns]
        if len(directos) == 4:
            fig, axes = plt.subplots(2, 2, figsize=(13, 9))
            for ax, c, col in zip(axes.flat, directos,
                                   [AZUL, VERDE, ROJO, AMBAR]):
                d = TRAIN_F1[c].dropna()
                ax.hist(d, bins=45, color=col, edgecolor='white')
                ax.text(.98, .96,
                        f'Mín: {d.min():.4f}\nMáx: {d.max():.4f}',
                        transform=ax.transAxes, ha='right', va='top',
                        fontsize=10,
                        bbox={'facecolor': 'white', 'alpha': .9,
                              'edgecolor': '#CBD5E1'})
                ax.set(title=c, xlabel=UNIDADES[c], ylabel='Observaciones')
            fig.suptitle('Distribución de los cuatro predictores directos',
                         x=.02, ha='left', fontweight='bold', fontsize=13)
            nota_fig(fig, 'Entrenamiento. Las diferencias territoriales '
                          'forman parte de la distribución combinada; no se '
                          'normalizan ni recortan aquí.')
            guardar_figura(fig, "EDA_Fig17_Distribucion_Directos")

        # FIG 18 y 19: Boxplots por jurisdicción
        if 'jurisdiccion' in TRAIN_F1.columns:
            orden = (CAT['nombre_jurisdiccion'].tolist()
                     if CAT is not None
                     else sorted(TRAIN_F1['jurisdiccion'].unique().tolist()))
            for num, c in [(18, 'P_mensual'), (19, 'T_mensual')]:
                if c not in TRAIN_F1.columns:
                    log(f"EDA_Fig{num} OMITIDA: falta {c}",
                        nivel="WARN", celda="Celda06_EDA"); continue
                grupos = [TRAIN_F1[TRAIN_F1['jurisdiccion'] == j][c]
                          .dropna() for j in orden]
                fig, ax = plt.subplots(figsize=(13, 11))
                bb = ax.boxplot(grupos, vert=False, patch_artist=True,
                                labels=orden,
                                flierprops={'markersize': 3, 'alpha': .5})
                for patch, col in zip(bb['boxes'],
                                       plt.cm.viridis(
                                           np.linspace(.1, .9, len(grupos)))):
                    patch.set_facecolor(col)
                for i, g in enumerate(grupos, 1):
                    if len(g):
                        ax.text(1.01, i, f'{g.min():.2f} / {g.max():.2f}',
                                transform=ax.get_yaxis_transform(),
                                fontsize=8, va='center')
                ax.text(1.01, 1.02, 'Mín / Máx',
                        transform=ax.transAxes, fontsize=9)
                ax.set(xlabel=c + ' (' + UNIDADES[c] + ')')
                ax.set_title(c + ' por jurisdicción · entrenamiento',
                             loc='left', pad=12)
                nota_fig(fig, 'Bigotes a 1.5 IQR. Los valores fuera de '
                              'bigotes permanecen en el dataset; las '
                              'etiquetas muestran mínimos y máximos.')
                guardar_figura(fig, f"EDA_Fig{num}_Boxplot_{c}")

        # FIG 20: Estacionalidad climática
        if all(c in TRAIN_F1.columns for c in ['P_mensual', 'T_mensual']) \
           and 'mes_objetivo' in TRAIN_F1.columns:
            season = TRAIN_F1.groupby('mes_objetivo')[
                ['P_mensual', 'T_mensual']
            ].agg(['median', 'min', 'max'])
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            for ax, c, col in zip(axes, ['P_mensual', 'T_mensual'],
                                   [AZUL, ROJO]):
                v = season[c]
                ax.fill_between(v.index, v['min'], v['max'],
                                alpha=.13, color=col,
                                label='Rango observado')
                ax.plot(v.index, v['median'], 'o-', color=col,
                        label='Mediana del panel')
                for tipo, val in [('Mín', v['median'].min()),
                                   ('Máx', v['median'].max())]:
                    for k in v.index[v['median'] == val]:
                        ax.annotate(f'{tipo} med.: {val:.2f}', (k, val),
                                    xytext=(0, 18 if tipo == 'Máx' else -25),
                                    textcoords='offset points',
                                    ha='center', fontsize=9)
                ax.set(title=c, xlabel='Mes calendario (del objetivo)',
                       ylabel=UNIDADES[c], xticks=range(1, 13))
                ax.legend(fontsize=9)
                ax.margins(y=.15)
            fig.suptitle('Estacionalidad climática del panel · '
                         'entrenamiento', x=.02, ha='left',
                         fontweight='bold', fontsize=13)
            nota_fig(fig, 'El valor graficado corresponde al clima observado 3 meses antes del origen y 4 meses antes del objetivo.')
            guardar_figura(fig, "EDA_Fig20_Estacionalidad_Climatica")

        # FIG 21: Banderas IQR clima
        if len(cols_f1) > 0 and 'jurisdiccion' in TRAIN_F1.columns:
            LIMITES, CONTEOS = [], []
            for j in TRAIN_F1['jurisdiccion'].unique():
                tr = TRAIN_F1[TRAIN_F1['jurisdiccion'] == j]
                va = VAL_F1[VAL_F1['jurisdiccion'] == j]
                for c in cols_f1:
                    if c not in tr.columns:
                        continue
                    q1, q3 = tr[c].quantile([.25, .75])
                    iqr = q3 - q1
                    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                    for b, d in [('ENTRENAMIENTO', tr),
                                  ('VALIDACION', va)]:
                        f1 = (d[c] < lo) | (d[c] > hi)
                        CONTEOS.append({'jurisdiccion': j,
                                        'predictor': c, 'bloque': b,
                                        'n': len(d),
                                        'fuera_1_5': int(f1.sum())})
            CONTEOS = pd.DataFrame(CONTEOS)
            if len(CONTEOS):
                RES_OUT = CONTEOS.groupby(
                    ['predictor', 'bloque']
                )[['n', 'fuera_1_5']].sum().reset_index()
                RES_OUT['porcentaje_1_5'] = (100 * RES_OUT.fuera_1_5 /
                                              RES_OUT.n)
                fig, ax = plt.subplots(figsize=(12, 5))
                piv = RES_OUT.pivot(index='predictor',
                                    columns='bloque',
                                    values='porcentaje_1_5').reindex(cols_f1)
                piv.plot.bar(ax=ax, color=[AZUL, AMBAR], rot=0)
                for container in ax.containers:
                    ax.bar_label(container, fmt='%.2f%%', padding=3,
                                 fontsize=9)
                ax.set(ylabel='Fuera de 1.5 IQR (%)', xlabel='Predictor',
                       ylim=(0, piv.to_numpy().max() * 1.3))
                ax.legend(['Entrenamiento', 'Validación'])
                ax.set_title('Frecuencia de banderas IQR · entrenamiento y '
                             'validación', loc='left', pad=12)
                nota_fig(fig, f'Mínimo y máximo entre tasas: '
                              f'{piv.min().min():.2f}% / '
                              f'{piv.max().max():.2f}%. Cercas estimadas por '
                              f'jurisdicción solo en entrenamiento.')
                guardar_figura(fig, "EDA_Fig21_Banderas_IQR_Clima")

        # FIG 22: Matriz de anomalías
        if all(c in TRAIN_F1.columns for c in ['z_P', 'z_T']) and \
           'origen_t' in TRAIN_F1.columns:
            TRAIN_F1['_mes_src'] = TRAIN_F1['origen_t'].astype(str)
            fig, axes = plt.subplots(2, 1, figsize=(15, 11))
            for ax, c in zip(axes, ['z_P', 'z_T']):
                mat = TRAIN_F1.pivot_table(
                    index='jurisdiccion', columns='_mes_src',
                    values=c, aggfunc='mean')
                im = ax.imshow(mat, aspect='auto', cmap='RdBu_r',
                               vmin=-3.5, vmax=3.5)
                ax.grid(False)
                ax.set_yticks(range(len(mat)), mat.index, fontsize=7)
                pos = [i for i, v in enumerate(mat.columns)
                       if str(v).endswith('-01')]
                ax.set_xticks(pos,
                              [str(mat.columns[i])[:4] for i in pos],
                              fontsize=8)
                ax.set(title=f'{c}: mínimo {np.nanmin(mat):.4f}; '
                             f'máximo {np.nanmax(mat):.4f}',
                       xlabel='Año del mes origen (t)')
                fig.colorbar(im, ax=ax, label=c, shrink=.8)
            fig.suptitle('Anomalías por territorio y mes fuente · '
                         'entrenamiento', x=.02, ha='left',
                         fontweight='bold', fontsize=13)
            nota_fig(fig, 'El valor graficado corresponde al clima observado 3 meses antes del origen y 4 meses antes del objetivo.')
            guardar_figura(fig, "EDA_Fig22_Matriz_Anomalias_Clima")

        # FIG 23: Autocorrelación climática
        if all(c in TRAIN_F1.columns for c in ['P_mensual', 'T_mensual']) \
           and 'jurisdiccion' in TRAIN_F1.columns:
            AC = []
            for j, d in TRAIN_F1.groupby('jurisdiccion'):
                d = d.sort_values('origen_t')
                for c in ['P_mensual', 'T_mensual']:
                    for k in range(1, 13):
                        AC.append({'jurisdiccion': j, 'predictor': c,
                                   'rezago': k,
                                   'autocorrelacion': d[c].autocorr(k)})
            AC = pd.DataFrame(AC)
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            for ax, c, col in zip(axes, ['P_mensual', 'T_mensual'],
                                   [AZUL, ROJO]):
                v = AC[AC.predictor == c].groupby(
                    'rezago').autocorrelacion.agg(
                        mediana='median',
                        q25=lambda x: x.quantile(.25),
                        q75=lambda x: x.quantile(.75))
                ax.plot(v.index, v.mediana, 'o-', color=col)
                ax.fill_between(v.index, v.q25, v.q75, color=col, alpha=.15)
                ax.axhline(0, color='#64748B', lw=1)
                ax.set(title=f'{c}\nMín/Max mediana: '
                             f'{v.mediana.min():.3f} / '
                             f'{v.mediana.max():.3f}',
                       xlabel='Rezago (meses)',
                       ylabel='Autocorrelación', xticks=range(1, 13))
            fig.suptitle('Dependencia temporal dentro de jurisdicción',
                         x=.02, ha='left', fontweight='bold', fontsize=13)
            nota_fig(fig, 'El valor graficado corresponde al clima observado 3 meses antes del origen y 4 meses antes del objetivo.')
            guardar_figura(fig, "EDA_Fig23_Autocorrelacion_Clima")

        # FIG 24: Evolución de la mediana territorial
        if all(c in TRAIN_F1.columns for c in ['P_mensual', 'T_mensual']) \
           and 'origen_t' in TRAIN_F1.columns:
            fig, axes = plt.subplots(2, 1, figsize=(13, 8))
            for ax, c, col in zip(axes, ['P_mensual', 'T_mensual'],
                                   [AZUL, ROJO]):
                v = TRAIN_F1.groupby('origen_t')[c].agg(
                    mediana='median', minimo='min',
                    maximo='max').reset_index()
                x = pd.to_datetime(v['origen_t'])
                ax.plot(x, v.mediana, color=col)
                ax.fill_between(x, v.minimo, v.maximo, color=col, alpha=.1)
                for label, val in [('Mín mediana', v.mediana.min()),
                                    ('Máx mediana', v.mediana.max())]:
                    r = v[v.mediana == val].iloc[0]
                    ax.annotate(
                        f'{label}: {val:.2f}\n'
                        f'{pd.Timestamp(r["origen_t"]).strftime("%Y-%m")}',
                        (pd.Timestamp(r['origen_t']), val),
                        xytext=(0, 20), textcoords='offset points',
                        ha='center', fontsize=8)
                ax.set(xlabel='Mes de origen (t)', ylabel=f'{c} ({UNIDADES[c]})')
                ax.margins(y=.2)
            fig.suptitle('Evolución de la mediana territorial · '
                         'entrenamiento', x=.02, ha='left',
                         fontweight='bold', fontsize=13)
            nota_fig(fig, 'El valor graficado corresponde al clima observado 3 meses antes del origen y 4 meses antes del objetivo.')
            guardar_figura(fig, "EDA_Fig24_Evolucion_Mediana_Clima")

        # FIG 25: Distribución por jurisdicción con etiquetas Mín/Máx
        if 'jurisdiccion' in TRAIN_F1.columns and \
           'P_mensual' in TRAIN_F1.columns:
            fig, ax = plt.subplots(figsize=(12, 10))
            orden2 = (CAT['nombre_jurisdiccion'].tolist()
                      if CAT is not None
                      else sorted(TRAIN_F1['jurisdiccion'].unique().tolist()))
            grupos = [TRAIN_F1[TRAIN_F1['jurisdiccion'] == j]['P_mensual']
                      .dropna() for j in orden2]
            bp = ax.boxplot(grupos, vert=False, patch_artist=True,
                            labels=orden2,
                            flierprops={'markersize': 3, 'alpha': .5})
            for patch, col in zip(bp['boxes'],
                                   plt.cm.viridis(
                                       np.linspace(.1, .9, len(grupos)))):
                patch.set_facecolor(col)
            for i, g in enumerate(grupos, 1):
                if len(g):
                    ax.text(1.01, i, f'{g.min():.2f} / {g.max():.2f}',
                            transform=ax.get_yaxis_transform(),
                            fontsize=8, va='center')
            ax.text(1.01, 1.02, 'Mín / Máx',
                    transform=ax.transAxes, fontsize=9)
            ax.set(xlabel='P_mensual (mm/mes)')
            ax.set_title('P_mensual por jurisdicción · etiquetas extremas',
                         loc='left', pad=12)
            nota_fig(fig, 'Etiquetas al margen derecho para no superponer '
                          'con las cajas. Los extremos se conservan.')
            guardar_figura(fig, "EDA_Fig25_Por_Jurisdiccion_Etiquetas")

    # =========================================================================
    # BLOQUE C — FIGURAS F2
    # =========================================================================
    log("Bloque C: figuras F2 (ENOS)", celda="Celda06_EDA")

    if ET is None or 'f2_oni' not in ET.columns or \
       'f2_icen' not in ET.columns:
        log("BLOQUE C OMITIDO: faltan ET o columnas f2_oni/f2_icen "
            "(¿corriste la celda 05a?)",
            nivel="WARN", celda="Celda06_EDA")
    else:
        ET_ENOS = ET.rename(columns={'f2_oni': 'oni', 'f2_icen': 'icen'})
        EV_ENOS = (EV.rename(columns={'f2_oni': 'oni', 'f2_icen': 'icen'})
                   if EV is not None else None)

        # FIG 26: Partición temporal
        fig, ax = plt.subplots(figsize=(10, 4))
        b = ax.bar(['Entrenamiento', 'Validación'],
                   [len(TRAIN_F1) if TRAIN_F1 is not None else 0,
                    len(VAL_F1) if VAL_F1 is not None else 0],
                   color=[AZUL, AMBAR])
        ax.bar_label(b)
        ax.set(ylabel='Filas',
               ylim=(0, max(len(TRAIN_F1) if TRAIN_F1 is not None else 0,
                            len(VAL_F1) if VAL_F1 is not None else 0) * 1.15))
        ax.set_title('Partición temporal conservada', loc='left', pad=12)
        nota_fig(fig, 'Solo conteos estructurales. La prueba no participa '
                      'en EDA ni ajuste de límites.')
        guardar_figura(fig, "EDA_Fig26_Particion_Temporal")

        # FIG 27 y 28: Series ONI e ICEN
        for num, c, color in [(27, 'oni', AZUL), (28, 'icen', ROJO)]:
            fig, ax = plt.subplots(figsize=(14, 5))
            ax.plot(pd.to_datetime(ET_ENOS['origen_t']),
                    ET_ENOS[c], color=color, lw=2)
            ax.axhline(0, color='#64748B', ls='--')
            for tipo, v, dy in [('Mín', ET_ENOS[c].min(), -40),
                                 ('Máx', ET_ENOS[c].max(), 24)]:
                q = ET_ENOS[ET_ENOS[c] == v]
                ax.scatter(pd.to_datetime(q['origen_t']), q[c],
                           color=AMBAR, s=45, zorder=4)
                r = q.iloc[0]
                ax.annotate(
                    f'{tipo}: {v:.2f} °C · {len(q)} origen(es)\n'
                    f'Origen {pd.Timestamp(r["origen_t"]).strftime("%Y-%m")}',
                    (pd.Timestamp(r['origen_t']), v),
                    xytext=(0, dy), textcoords='offset points',
                    ha='center', fontsize=9,
                    bbox={'facecolor': 'white', 'alpha': .85,
                          'edgecolor': 'none'},
                    arrowprops={'arrowstyle': '-', 'color': AMBAR})
            ax.margins(x=.07, y=.33)
            ax.set(xlabel='Origen del pronóstico',
                   ylabel=c.upper() + ' (°C de anomalía TSM)')
            ax.set_title(c.upper() + ' disponible · entrenamiento',
                         loc='left', pad=12)
            nota_fig(fig, f'{len(ET_ENOS)} orígenes únicos. Etiquetas con '
                          f'origen; la selección del mes central se '
                          f'documenta en el dataset.')
            guardar_figura(fig, f"EDA_Fig{num}_Serie_{c.upper()}")

        # FIG 29: ONI e ICEN alineados
        fig, ax = plt.subplots(figsize=(14, 5))
        for c, col in [('oni', AZUL), ('icen', ROJO)]:
            ax.plot(pd.to_datetime(ET_ENOS['origen_t']),
                    ET_ENOS[c], color=col,
                    label=f'{c.upper()} · mín '
                          f'{ET_ENOS[c].min():.2f} / máx '
                          f'{ET_ENOS[c].max():.2f}')
        ax.axhline(0, color='#64748B', ls='--')
        ax.legend()
        ax.set(ylabel='Anomalía TSM (°C)',
               xlabel='Origen del pronóstico')
        ax.set_title('ONI e ICEN alineados por disponibilidad',
                     loc='left', pad=12)
        nota_fig(fig, 'ONI: central t−4; ICEN: central t−3. No se crean '
                      'categorías ENOS ni se reinterpretan umbrales.')
        guardar_figura(fig, "EDA_Fig29_ONI_ICEN_Alineados")

        # FIG 30: Distribuciones ENOS
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        for ax, c, col in zip(axes, ['oni', 'icen'], [AZUL, ROJO]):
            d = ET_ENOS[c].dropna()
            ax.hist(d, bins=18, color=col, edgecolor='white')
            ax.text(.98, .96,
                    f'Mín: {d.min():.2f}\nMáx: {d.max():.2f}',
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=10,
                    bbox={'facecolor': 'white', 'alpha': .9,
                          'edgecolor': '#CBD5E1'})
            ax.set(title=c.upper(), xlabel='Anomalía TSM (°C)',
                   ylabel='Orígenes')
        fig.suptitle('Distribuciones ENOS · entrenamiento',
                     x=.02, ha='left', fontweight='bold', fontsize=13)
        nota_fig(fig, f'{len(ET_ENOS)} valores por índice; no '
                      f'{len(TRAIN_F1) if TRAIN_F1 is not None else 0} '
                      f'mediciones independientes. Se conserva la escala '
                      f'original.')
        guardar_figura(fig, "EDA_Fig30_Distribucion_ENOS")

        # FIG 31: ECDF ENOS
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        for ax, c, col in zip(axes, ['oni', 'icen'], [AZUL, ROJO]):
            x = np.sort(ET_ENOS[c].dropna().to_numpy())
            y = np.arange(1, len(x) + 1) / len(x)
            ax.step(x, y, where='post', color=col, lw=2)
            ax.text(.98, .96,
                    f'Mín: {x.min():.2f}\nMáx: {x.max():.2f}',
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=10,
                    bbox={'facecolor': 'white', 'alpha': .9,
                          'edgecolor': '#CBD5E1'})
            ax.set(title=c.upper(), xlabel='Anomalía TSM (°C)',
                   ylabel='Proporción acumulada', ylim=(0, 1.05))
        fig.suptitle('Distribuciones acumuladas empíricas',
                     x=.02, ha='left', fontweight='bold', fontsize=13)
        nota_fig(fig, 'Entrenamiento. La curva no presupone normalidad; '
                      'los empates conservan el peso de todos sus orígenes.')
        guardar_figura(fig, "EDA_Fig31_ECDF_ENOS")

        # FIG 32: Boxplot ENOS por bloque
        if EV_ENOS is not None:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
            for ax, c in zip(axes, ['oni', 'icen']):
                b = ax.boxplot([ET_ENOS[c].dropna(),
                                 EV_ENOS[c].dropna()],
                                patch_artist=True,
                                labels=['Entrenamiento', 'Validación'],
                                flierprops={'markersize': 5, 'alpha': .7})
                for p, col in zip(b['boxes'], [AZUL, AMBAR]):
                    p.set_facecolor(col)
                for k, d in enumerate([ET_ENOS[c].dropna(),
                                        EV_ENOS[c].dropna()], 1):
                    ax.text(k, .98,
                            f'n={len(d)}\nMín {d.min():.2f}\n'
                            f'Máx {d.max():.2f}',
                            transform=ax.get_xaxis_transform(),
                            ha='center', va='top', fontsize=9)
                ax.margins(y=.4)
                ax.set(title=c.upper(), ylabel='Anomalía TSM (°C)')
            fig.suptitle('Dispersión ENOS por bloque de desarrollo',
                         x=.02, ha='left', fontweight='bold', fontsize=13)
            nota_fig(fig, 'Cada caja describe su bloque. Los límites '
                          'formales para señalar outliers se ajustan solo '
                          'con entrenamiento.')
            guardar_figura(fig, "EDA_Fig32_Boxplot_ENOS_Bloques")

        # FIG 33: Scatter ONI–ICEN
        pe = float(ET_ENOS.oni.corr(ET_ENOS.icen))
        sp = float(ET_ENOS.oni.corr(ET_ENOS.icen, method='spearman'))
        fig, ax = plt.subplots(figsize=(9, 6))
        anios = pd.to_datetime(ET_ENOS['origen_t']).dt.year
        sc = ax.scatter(ET_ENOS.oni, ET_ENOS.icen, c=anios,
                        cmap='viridis', s=40, alpha=.8,
                        edgecolor='white', linewidth=.3)
        fig.colorbar(sc, ax=ax, label='Año del origen')
        ax.set(xlabel='ONI disponible (°C)',
               ylabel='ICEN disponible (°C)')
        ax.text(.02, .97,
                f'Pearson r = {pe:.3f}\nSpearman ρ = {sp:.3f}\n'
                f'ONI: [{ET_ENOS.oni.min():.2f}, '
                f'{ET_ENOS.oni.max():.2f}]\n'
                f'ICEN: [{ET_ENOS.icen.min():.2f}, '
                f'{ET_ENOS.icen.max():.2f}]',
                transform=ax.transAxes, va='top', fontsize=10,
                bbox={'facecolor': 'white', 'alpha': .8,
                      'edgecolor': '#CBD5E1'})
        ax.set_title('Asociación ONI–ICEN por origen · entrenamiento',
                     loc='left', pad=12)
        nota_fig(fig, 'Diferentes regiones oceánicas y meses centrales. '
                      'No hay prueba de significación ni atribución causal.')
        guardar_figura(fig, "EDA_Fig33_Scatter_ONI_ICEN")

        # FIG 34: Correlación clima-ENOS
        cols_corr = [c for c in ['P_mensual', 'z_P', 'P_3m',
                                  'T_mensual', 'z_T', 'T_3m']
                     if c in TRAIN_F1.columns]
        cols_corr += [c for c in ['oni', 'icen']
                      if c in TRAIN_F1.columns]
        if len(cols_corr) >= 3 and 'origen_t' in TRAIN_F1.columns:
            TRAIN_CORR = TRAIN_F1.merge(
                ET_ENOS[['origen_t', 'oni', 'icen']],
                on='origen_t', how='left')
            if all(c in TRAIN_CORR.columns for c in cols_corr):
                COR = TRAIN_CORR[cols_corr].corr(method='spearman')
                pares = [(cols_corr[i], cols_corr[j], COR.iloc[i, j])
                         for i in range(len(cols_corr))
                         for j in range(i + 1, len(cols_corr))]
                pmin = min(pares, key=lambda z: z[2])
                pmax = max(pares, key=lambda z: z[2])
                fig, ax = plt.subplots(figsize=(11, 9))
                im = ax.imshow(COR, cmap='RdBu_r', vmin=-1, vmax=1)
                ax.set_xticks(range(len(cols_corr)), cols_corr,
                              rotation=35, ha='right')
                ax.set_yticks(range(len(cols_corr)), cols_corr)
                ax.grid(False)
                for i in range(len(cols_corr)):
                    for j in range(len(cols_corr)):
                        v = COR.iloc[i, j]
                        ax.text(j, i, f'{v:.2f}', ha='center',
                                va='center',
                                color='white' if abs(v) > .6
                                else '#111827',
                                fontsize=9)
                fig.colorbar(im, ax=ax, label='Spearman ρ', shrink=.8)
                ax.set_title('Asociación entre clima y ENOS · '
                             'panel de entrenamiento', loc='left', pad=12)
                nota_fig(fig, f'Extremos fuera de diagonal: '
                              f'{pmin[0]}–{pmin[1]} = {pmin[2]:.3f}; '
                              f'{pmax[0]}–{pmax[1]} = {pmax[2]:.3f}. '
                              f'Los índices se repiten por territorio; '
                              f'no se calculan p-valores.')
                guardar_figura(fig, "EDA_Fig34_Correlacion_Clima_ENOS")

        # FIG 35: Mapa temporal ONI/ICEN
        fig, axes = plt.subplots(1, 2, figsize=(15, 8))
        lim = max(abs(ET_ENOS.oni.min()), abs(ET_ENOS.oni.max()),
                  abs(ET_ENOS.icen.min()), abs(ET_ENOS.icen.max()))
        cmap = plt.colormaps['RdBu_r'].copy()
        cmap.set_bad('#E5E7EB')
        for ax, c in zip(axes, ['oni', 'icen']):
            tmp = ET_ENOS.assign(
                anio=pd.to_datetime(ET_ENOS['origen_t']).dt.year,
                mes=pd.to_datetime(ET_ENOS['origen_t']).dt.month)
            tab = tmp.pivot(index='anio', columns='mes',
                            values=c).reindex(columns=range(1, 13))
            im = ax.imshow(tab, aspect='auto', cmap=cmap,
                           vmin=-lim, vmax=lim)
            ax.grid(False)
            ax.set_xticks(range(12), range(1, 13), fontsize=8)
            ax.set_yticks(range(len(tab)), tab.index)
            ax.set(title=f'{c.upper()} · mín {ET_ENOS[c].min():.2f} / '
                         f'máx {ET_ENOS[c].max():.2f}',
                   xlabel='Mes del origen', ylabel='Año del origen')
            for i in range(len(tab)):
                for j in range(12):
                    val = tab.iloc[i, j]
                    if pd.notna(val):
                        ax.text(j, i, f'{val:.1f}', ha='center',
                                va='center', fontsize=6,
                                color='white' if abs(val) > .65 * lim
                                else '#111827')
            fig.colorbar(im, ax=ax, label='°C', shrink=.65)
        fig.suptitle('Mapa temporal de índices disponibles · '
                     'entrenamiento', x=.02, ha='left',
                     fontweight='bold', fontsize=13)
        nota_fig(fig, 'Celdas grises: orígenes fuera del bloque, no datos '
                      'imputados. Las etiquetas representan el origen, no '
                      'el mes central del índice.')
        guardar_figura(fig, "EDA_Fig35_Mapa_Temporal_ENOS")

        # FIG 36: Banderas IQR ENOS
        LIM_E, C_E = [], []
        for c in ['oni', 'icen']:
            q1, q3 = ET_ENOS[c].quantile([.25, .75])
            iqr = q3 - q1
            lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            LIM_E.append({'indice': c, 'q1': q1, 'q3': q3, 'iqr': iqr,
                          'inferior_1_5': lo, 'superior_1_5': hi})
            for b, d in [('ENTRENAMIENTO', ET_ENOS),
                          ('VALIDACION', EV_ENOS)]:
                if d is None:
                    continue
                f1 = (d[c] < lo) | (d[c] > hi)
                C_E.append({'indice': c, 'bloque': b,
                            'n_origenes': len(d),
                            'fuera_1_5': int(f1.sum()),
                            'porcentaje_1_5': 100 * f1.mean()})
        C_E = pd.DataFrame(C_E)
        if len(C_E):
            fig, ax = plt.subplots(figsize=(10, 5))
            piv = C_E.pivot(index='indice', columns='bloque',
                            values='porcentaje_1_5')
            piv.plot.bar(ax=ax, color=[AZUL, AMBAR], rot=0)
            for b in ax.containers:
                ax.bar_label(b, fmt='%.2f%%', padding=3)
            ax.set(ylabel='Orígenes fuera de 1.5 IQR (%)',
                   xlabel='Índice',
                   ylim=(0, max(1, piv.to_numpy().max() * 1.3)))
            ax.legend(['Entrenamiento', 'Validación'])
            ax.set_title('Frecuencia de banderas IQR · ENOS',
                         loc='left', pad=12)
            nota_fig(fig, f'Mínimo y máximo entre tasas: '
                          f'{piv.min().min():.2f}% / '
                          f'{piv.max().max():.2f}%. No se modifica ningún '
                          f'índice ni se analiza prueba.')
            guardar_figura(fig, "EDA_Fig36_Banderas_IQR_ENOS")

        # FIG 37: Outliers en el tiempo
        if EV_ENOS is not None:
            DEV = pd.concat([ET_ENOS, EV_ENOS]).sort_values('origen_t')
            fig, axes = plt.subplots(2, 1, figsize=(14, 9))
            for ax, c, col in zip(axes, ['oni', 'icen'], [AZUL, ROJO]):
                l = [d for d in LIM_E if d['indice'] == c][0]
                fuera = (DEV[c] < l['inferior_1_5']) | \
                        (DEV[c] > l['superior_1_5'])
                ax.plot(pd.to_datetime(DEV['origen_t']), DEV[c],
                        color=col, label=c.upper())
                ax.scatter(pd.to_datetime(DEV.loc[fuera, 'origen_t']),
                            DEV.loc[fuera, c], color=AMBAR, zorder=4,
                            label='Fuera de 1.5 IQR')
                for v in [l['inferior_1_5'], l['superior_1_5']]:
                    ax.axhline(v, color=VERDE, ls='--')
                ax.axvspan(pd.Timestamp(EV_ENOS['origen_t'].min()),
                            pd.Timestamp(EV_ENOS['origen_t'].max()),
                            color=AMBAR, alpha=.1, label='Validación')
                ax.text(.98, .96,
                        f'Mín: {DEV[c].min():.2f}\n'
                        f'Máx: {DEV[c].max():.2f}',
                        transform=ax.transAxes, ha='right', va='top',
                        fontsize=10,
                        bbox={'facecolor': 'white', 'alpha': .9,
                              'edgecolor': '#CBD5E1'})
                ax.set(ylabel='Anomalía TSM (°C)', xlabel='Origen')
                ax.legend(loc='upper left', fontsize=9)
            fig.suptitle('Outliers conservados en el tiempo · desarrollo',
                         x=.02, ha='left', fontweight='bold', fontsize=13)
            nota_fig(fig, 'Cercas aprendidas solo en entrenamiento. El '
                          'sombreado marca el bloque de validación.')
            guardar_figura(fig, "EDA_Fig37_Outliers_ENOS_Tiempo")

        # FIG 38: Autocorrelación ENOS
        AC_E = pd.DataFrame([
            {'indice': c, 'rezago_meses': k,
             'autocorrelacion': ET_ENOS[c].autocorr(k)}
            for c in ['oni', 'icen'] for k in range(1, 13)
        ])
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        for ax, c, col in zip(axes, ['oni', 'icen'], [AZUL, ROJO]):
            q = AC_E[AC_E.indice == c]
            ax.plot(q.rezago_meses, q.autocorrelacion, 'o-', color=col)
            ax.axhline(0, color='#64748B')
            ax.set(title=f'{c.upper()} · mín '
                         f'{q.autocorrelacion.min():.3f} / máx '
                         f'{q.autocorrelacion.max():.3f}',
                   xlabel='Rezago diagnóstico (meses)',
                   ylabel='Autocorrelación', xticks=range(1, 13))
        fig.suptitle('Dependencia temporal de los índices · '
                     'entrenamiento', x=.02, ha='left',
                     fontweight='bold', fontsize=13)
        nota_fig(fig, 'Las ventanas móviles del índice comparten meses. '
                      'No se realizan pruebas de significación.')
        guardar_figura(fig, "EDA_Fig38_Autocorrelacion_ENOS")

        # FIG 39: Distribución por mes calendario
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        for ax, c, col in zip(axes, ['oni', 'icen'], [AZUL, ROJO]):
            meses = pd.to_datetime(ET_ENOS['origen_t']).dt.month
            grupos = [ET_ENOS[meses == k][c].dropna()
                      for k in range(1, 13)]
            b = ax.boxplot(grupos, patch_artist=True, labels=range(1, 13))
            for p in b['boxes']:
                p.set_facecolor(col)
            ax.set(title=f'{c.upper()} · mín {ET_ENOS[c].min():.2f} / '
                         f'máx {ET_ENOS[c].max():.2f}',
                   xlabel='Mes de origen (aprox. central)',
                   ylabel='Anomalía TSM (°C)')
        fig.suptitle('Distribución por mes calendario · entrenamiento',
                     x=.02, ha='left', fontweight='bold', fontsize=13)
        nota_fig(fig, 'Cada grupo contiene los orígenes de ese mes. '
                      'Esta descripción no crea un ajuste estacional.')
        guardar_figura(fig, "EDA_Fig39_ENOS_Por_Mes")

        # FIG 40: Balance del objetivo
        fig, ax = plt.subplots(figsize=(8, 5))
        ct = TRAIN_F1.Y_caida.value_counts().reindex([0, 1])
        bars = ax.bar(['Sin caída (0)', 'Caída (1)'], ct,
                      color=[AZUL, ROJO])
        ax.bar_label(bars,
                     labels=[f'{n:,}\n{n/len(TRAIN_F1)*100:.2f}%'
                             for n in ct],
                     padding=5)
        ax.set(ylabel='Filas del panel de entrenamiento',
               ylim=(0, ct.max() * 1.2))
        ax.set_title('Objetivo heredado · entrenamiento',
                     loc='left', pad=12)
        nota_fig(fig, f'Mínimo {ct.min()}; máximo {ct.max()} '
                      f'observaciones. No se balancean clases ni se '
                      f'entrenan modelos.')
        guardar_figura(fig, "EDA_Fig40_Balance_Objetivo")


    # =========================================================================
    # BLOQUE D — FIGURAS ADICIONALES
    # =========================================================================
    #
    # Fig41  Cobertura temporal (historia inicial / entrenamiento / validación)
    # Fig42  Serie mensual recaud_real_t_2 (mediana + IQR)
    # Fig43  Serie mensual g_var_real_t1 (mediana + IQR + cero)
    # Fig44  Meses de campaña Renta Anual por año
    # Fig45  Serie mensual ctrl_pbi_var
    # Fig46  Panel: inflación y tipo de cambio mensual
    # =========================================================================
    log("Bloque D: figuras adicionales del notebook F0",
        celda="Celda06_EDA")

    # -------------------------------------------------------------------------
    # FIG 41: Cobertura temporal
    # -------------------------------------------------------------------------
    import matplotlib.dates as mdates

    fig, ax = plt.subplots(figsize=(14, 4))
    segmentos = [
        ('Historia inicial', pd.Timestamp('2005-01-01'),
                            pd.Timestamp('2007-02-28'), AZUL),
        ('Entrenamiento',    pd.Timestamp('2007-03-01'),
                            pd.Timestamp('2019-12-31'), VERDE),
        ('Validación',       pd.Timestamp('2020-01-01'),
                            pd.Timestamp('2022-12-31'), AMBAR),
    ]
    for nombre, ini, fin, color in segmentos:
        ax.barh(0, (fin - ini).days, left=ini, height=0.55,
                color=color, edgecolor='white')
        ax.text(ini + (fin - ini) / 2, 0, nombre,
                ha='center', va='center', fontsize=10,
                fontweight='bold', color='white')
    ax.set_yticks([])
    ax.set_xlim(pd.Timestamp('2005-01-01'),
                pd.Timestamp('2022-12-31'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.set_xlabel('Año')
    ax.set_title('Cobertura temporal del EDA · Entrenamiento + Validación',
                 loc='left', pad=12)
    ax.grid(axis='x', alpha=0.25)
    nota_fig(fig, 'La prueba 2023-2025 está FUERA del EDA (BARRERA activa). '
                  'Solo se grafica el período de desarrollo.')
    guardar_figura(fig, "EDA_Fig41_Cobertura_Temporal")

    # -------------------------------------------------------------------------
    # FIG 42 y FIG 43: series mensuales r_lag_01 y g_lag_01
    # (en df_panel: recaud_real_t_2 ≈ r_lag_01; var_real_t_2 = g_lag_01)
    # -------------------------------------------------------------------------
    mensual_dev = (
        df_dev.groupby('objetivo_t1')
        .agg(
            r_mediana=('recaud_real_t_2', 'median'),
            r_q25=('recaud_real_t_2', lambda x: x.quantile(0.25)),
            r_q75=('recaud_real_t_2', lambda x: x.quantile(0.75)),
            g_mediana=('var_real_t_2', 'median'),
            g_q25=('var_real_t_2', lambda x: x.quantile(0.25)),
            g_q75=('var_real_t_2', lambda x: x.quantile(0.75)),
            pbi=('ctrl_pbi_var', 'first'),
            inf=('ctrl_inflacion_lima', 'first'),
            tc=('ctrl_tipo_cambio', 'first'),
            renta_anual=('ind_renta_anual', 'first'),
        )
        .reset_index()
    )

    # FIG 42: recaudación real t-2 (r_lag_01)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(mensual_dev['objetivo_t1'], mensual_dev['r_mediana'],
            linewidth=1.5, label='Mediana', color=AZUL)
    ax.fill_between(mensual_dev['objetivo_t1'],
                    mensual_dev['r_q25'], mensual_dev['r_q75'],
                    alpha=0.20, label='Rango intercuartílico', color=AZUL)
    ax.set_title('Recaudación real más reciente disponible '
                 '(recaud_real_t_2 ≈ r_lag_01) · Entrenamiento + Validación',
                 loc='left', pad=12)
    ax.set_xlabel('Mes objetivo')
    ax.set_ylabel('Millones de soles constantes (dic-2021)')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    nota_fig(fig, 'Serie mensual de las 26 jurisdicciones. '
                  'La banda es el IQR entre jurisdicciones, no '
                  'un intervalo de confianza.')
    guardar_figura(fig, "EDA_Fig42_Serie_r_lag_01")

    # FIG 43: variación interanual real t-2 (g_lag_01)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(mensual_dev['objetivo_t1'], mensual_dev['g_mediana'],
            linewidth=1.5, label='Mediana', color=MORADO)
    ax.fill_between(mensual_dev['objetivo_t1'],
                    mensual_dev['g_q25'], mensual_dev['g_q75'],
                    alpha=0.20, label='Rango intercuartílico', color=MORADO)
    ax.axhline(0, linewidth=1, linestyle='--', color=ROJO)
    ax.set_title('Variación interanual real disponible '
                 '(var_real_t_2 = g_lag_01) · Entrenamiento + Validación',
                 loc='left', pad=12)
    ax.set_xlabel('Mes objetivo')
    ax.set_ylabel('Variación logarítmica interanual (%)')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    nota_fig(fig, 'La línea punteada marca Y=0 (umbral de caída). '
                  'La banda es el IQR entre jurisdicciones.')
    guardar_figura(fig, "EDA_Fig43_Serie_g_lag_01")

    # -------------------------------------------------------------------------
    # FIG 44: meses de campaña de Renta Anual por año
    # -------------------------------------------------------------------------
    renta_anual_anual = (
        mensual_dev
        .assign(anio=mensual_dev['objetivo_t1'].dt.year)
        .groupby('anio')['renta_anual']
        .sum()
    )
    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(renta_anual_anual.index.astype(str),
                  renta_anual_anual.values, color=VERDE)
    ax.bar_label(bars, padding=3, fontsize=9, fontweight='bold')
    ax.set_title('Meses con indicador de campaña de Renta Anual = 1 '
                 '· Entrenamiento + Validación',
                 loc='left', pad=12)
    ax.set_xlabel('Año objetivo')
    ax.set_ylabel('Número de meses de campaña')
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=45)
    nota_fig(fig, 'Conteo anual de meses con ind_renta_anual=1. '
                  'Cada año completo aporta 12 meses objetivo.')
    guardar_figura(fig, "EDA_Fig44_Renta_Anual_Meses")

    # -------------------------------------------------------------------------
    # FIG 45: PBI yoy mensual
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(mensual_dev['objetivo_t1'], mensual_dev['pbi'],
            linewidth=1.4, color=AZUL)
    ax.axhline(0, linewidth=1, linestyle='--', color=ROJO)
    ax.set_title('PBI · variación porcentual interanual disponible '
                 'en el origen · Entrenamiento + Validación',
                 loc='left', pad=12)
    ax.set_xlabel('Mes objetivo')
    ax.set_ylabel('PBI interanual (%)')
    ax.grid(alpha=0.25)
    nota_fig(fig, 'Serie única por mes (control macro). '
                  'No es 26 mediciones independientes.')
    guardar_figura(fig, "EDA_Fig45_Serie_PBI")

    # -------------------------------------------------------------------------
    # FIG 46: inflación mensual + tipo de cambio (panel 1×2)
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    axes[0].plot(mensual_dev['objetivo_t1'], mensual_dev['inf'],
                 linewidth=1.4, color=ROJO)
    axes[0].axhline(0, linewidth=1, linestyle='--', color='#64748B')
    axes[0].set_ylabel('Inflación mensual (%)')
    axes[0].set_title('Inflación mensual de Lima Metropolitana disponible '
                      'en el origen', loc='left', pad=8)
    axes[0].grid(alpha=0.25)

    axes[1].plot(mensual_dev['objetivo_t1'], mensual_dev['tc'],
                 linewidth=1.4, color=MORADO)
    axes[1].set_xlabel('Mes objetivo')
    axes[1].set_ylabel('PEN/USD')
    axes[1].set_title('Tipo de cambio promedio PEN/USD disponible '
                      'en el origen', loc='left', pad=8)
    axes[1].grid(alpha=0.25)

    fig.suptitle('Controles macroeconómicos · Entrenamiento + Validación',
                 x=.02, ha='left', fontweight='bold', fontsize=13)
    nota_fig(fig, 'Series únicas por mes. La prueba 2023-2025 '
                  'no se muestra.')
    guardar_figura(fig, "EDA_Fig46_Serie_Inflacion_TC")

    # -------------------------------------------------------------------------
    # TABLA 27: estadísticos descriptivos de 5 variables
    # -------------------------------------------------------------------------
    VARIABLES_RESUMEN = {
        'recaud_real_t_2':     'r_lag_01',
        'var_real_t_2':        'g_lag_01',
        'ctrl_pbi_var':        'pbi_yoy',
        'ctrl_inflacion_lima': 'inflacion_mom',
        'ctrl_tipo_cambio':    'tc_pen_usd',
    }
    est = (
        df_dev[list(VARIABLES_RESUMEN.keys())]
        .agg(['min', 'mean', 'median', 'std', 'max'])
        .T
        .rename(index=VARIABLES_RESUMEN)
    )
    est.columns = ['Mínimo', 'Media', 'Mediana', 'Desv. estándar', 'Máximo']
    est = est.round(4)

    # 1) Guardar CSV con valores NUMÉRICOS reales (auditoría + reutilización)
    est.to_csv(EDA_DIR / "EDA_Tabla_Estadisticos_5_Variables.csv")
    registrar_hash("EDA_Tabla_Estadisticos_5_Variables",
                   path=EDA_DIR / "EDA_Tabla_Estadisticos_5_Variables.csv")

    # 2) Display con formato bonito (SOLO para el notebook)
    est_display = est.copy()
    for col in ['Mínimo', 'Media', 'Mediana', 'Desv. estándar', 'Máximo']:
        est_display[col] = est_display[col].apply(
            lambda x: f'{x:.4f}' if pd.notna(x) else 'NaN'
        )
    display(est_display)

    # -------------------------------------------------------------------------
    # TABLA 28: entrenamiento vs validación para las 5 variables
    # -------------------------------------------------------------------------
    filas_t28 = []
    for col_orig, nombre in VARIABLES_RESUMEN.items():
        for bloque, sub in [
            ('Entrenamiento', df_panel[
                df_panel['bloque_temporal'] == 'ENTRENAMIENTO']),
            ('Validación',    df_panel[
                df_panel['bloque_temporal'] == 'VALIDACION']),
        ]:
            s = sub[col_orig].dropna()
            filas_t28.append({
                'variable': nombre,
                'bloque':   bloque,
                'n':        len(s),
                'min':      round(float(s.min()), 4),
                'media':    round(float(s.mean()), 4),
                'max':      round(float(s.max()), 4),
            })
    tabla28 = pd.DataFrame(filas_t28)

    # 1) Guardar CSV con valores NUMÉRICOS reales
    tabla28.to_csv(EDA_DIR / "EDA_Tabla_Train_vs_Val_5_Variables.csv",
                   index=False)
    registrar_hash("EDA_Tabla_Train_vs_Val_5_Variables",
                   path=EDA_DIR / "EDA_Tabla_Train_vs_Val_5_Variables.csv")

    # 2) Display con formato bonito (SOLO para el notebook)
    tabla28_display = tabla28.copy()
    for col in ['min', 'media', 'max']:
        tabla28_display[col] = tabla28_display[col].apply(
            lambda x: f'{x:.4f}' if pd.notna(x) else 'NaN'
        )
    display(tabla28_display)

    log("Bloque D completado: 6 figuras y 2 tablas adicionales",
        celda="Celda06_EDA")

    # =========================================================================
    # TABLA RESUMEN + REGISTRO DE HASHES
    # =========================================================================
    log("Generando tabla resumen del EDA", celda="Celda06_EDA")
    resumen_eda = df_dev.groupby('bloque_temporal').agg(
        n_obs=('Y_caida', 'size'),
        n_caidas=('Y_caida', 'sum'),
        tasa_caida_pct=('Y_caida', lambda x: 100 * x.mean()),
        g_media=('g_var_real_t1', 'mean'),
        g_mediana=('var_real_t_2', 'median'),
    ).round(3).reset_index()
    resumen_eda.to_csv(EDA_DIR / "Resumen_EDA_Por_Bloque.csv", index=False)
    display(resumen_eda)

    registrar_hash("EDA_Resumen_Por_Bloque",
                   path=EDA_DIR / "Resumen_EDA_Por_Bloque.csv")
    for f in sorted(EDA_DIR.glob("*.png")):
        registrar_hash(f"EDA_{f.stem}", path=f)

    n_fig = len(list(EDA_DIR.glob("*.png")))
    log(f"EDA extendido completado: {n_fig} figuras generadas",
        celda="Celda06_EDA")

    resumen_figuras = pd.DataFrame(FIGURAS_META)
    resumen_figuras.to_csv(EDA_DIR / "Indice_Figuras_EDA.csv", index=False)

    return resumen_eda, resumen_figuras


# ==============================================================================
# EJECUCIÓN
# ==============================================================================
resumen_eda, indice_figuras_eda = ejecutar_eda_descriptivo(
    df_panel, panel_base)

# ==============================================================================
# RESUMEN FINAL
# ==============================================================================
print(f"\n{'='*70}")
print(f"EDA EXTENDIDO COMPLETADO")
print(f"{'='*70}")
print(f"Figuras generadas: {len(list(EDA_DIR.glob('*.png')))}")
for f in sorted(EDA_DIR.glob("*.png")):
    print(f"  - {f.name} ({f.stat().st_size // 1024} KB)")
print(f"\nDirectorio: {EDA_DIR}")
print(f"Hash y flag registrados en: {AUDIT_DIR}")

[2026-10-30T01:54:43.985601+00:00] [INFO] [Celda06_EDA] INICIO ejecutar_eda_descriptivo
[2026-10-30T01:54:43.986493+00:00] [INFO] [Celda06_EDA] Iniciando EDA descriptivo extendido (OE1 + F1 + F2)
[2026-10-30T01:54:44.023760+00:00] [INFO] [Celda06_EDA] Filas usadas: 4888 (Entrenamiento + Validación)
[2026-10-30T01:54:44.024400+00:00] [INFO] [Celda06_EDA] Rango temporal: 2007-03-01 → 2022-10-01
[2026-10-30T01:54:44.024515+00:00] [INFO] [Celda06_EDA] Bloque A: figuras base del panel
  ✓ EDA_Fig1_Balance_ENTRENAMIENTO.png
  ✓ EDA_Fig2_Balance_VALIDACION.png
  ✓ EDA_Fig3_Frecuencia_Anual_Caidas.png
  ✓ EDA_Fig4_Frecuencia_Territorial.png
  ✓ EDA_Fig5_Distribucion_g.png
  ✓ EDA_Fig6_Boxplot_Recaudacion.png
  ✓ EDA_Fig7_Boxplot_Variacion.png
  ✓ EDA_Fig8_Estacionalidad.png
  ✓ EDA_Fig9_Recaudacion_Anual.png
[2026-10-30T01:54:52.666149+00:00] [INFO] [Celda06_EDA] Bloque B: figuras F1 (clima)
  ✓ EDA_Fig10_Histograma_z_P.png
  ✓ EDA_Fig11_Histograma_z_T.png
  ✓ EDA_Fig12_Tumbes_P_mensual.png
  

,Mínimo,Media,Mediana,Desv. estándar,Máximo
r_lag_01,1084381.7332,296869072.6078,26198760.8023,1256592246.0661,16014211223.0392
g_lag_01,-234.9088,6.9355,6.1610,28.8261,262.9263
pbi_yoy,-39.2770,4.5863,4.1323,8.1407,59.8948
inflacion_mom,-0.4692,0.2820,0.2516,0.3161,1.4827
tc_pen_usd,2.5522,3.1393,3.1647,0.3624,4.1081


[2026-10-30T01:55:27.747799+00:00] [INFO] [HASH] HASH registrado: EDA_Tabla_Train_vs_Val_5_Variables = 3c158653df6f136d...


,variable,bloque,n,min,media,max
0,r_lag_01,Entrenamiento,4004,1084381.7332,282038382.0884,11181217662.3588
1,r_lag_01,Validación,884,1196197.8195,364043376.7248,16014211223.0392
2,g_lag_01,Entrenamiento,4004,-234.9088,6.5490,262.9263
3,g_lag_01,Validación,884,-162.5324,8.6862,206.4630
4,pbi_yoy,Entrenamiento,4004,-2.8585,4.9943,14.0844
5,pbi_yoy,Validación,884,-39.2770,2.7383,59.8948
6,inflacion_mom,Entrenamiento,4004,-0.4692,0.2500,1.3046
7,inflacion_mom,Validación,884,-0.2661,0.4269,1.4827
8,tc_pen_usd,Entrenamiento,4004,2.5522,3.0162,3.5070
9,tc_pen_usd,Validación,884,3.3278,3.6968,4.1081


[2026-10-30T01:55:27.765144+00:00] [INFO] [Celda06_EDA] Bloque D completado: 6 figuras y 2 tablas adicionales
[2026-10-30T01:55:27.765335+00:00] [INFO] [Celda06_EDA] Generando tabla resumen del EDA


,bloque_temporal,n_obs,n_caidas,tasa_caida_pct,g_media,g_mediana
0,ENTRENAMIENTO,4004,1440.0,35.964,6.384,6.174
1,VALIDACION,884,350.0,39.593,8.271,6.029


[2026-10-30T01:55:27.789775+00:00] [INFO] [HASH] HASH registrado: EDA_Resumen_Por_Bloque = fd8cc58ecf894f65...
[2026-10-30T01:55:27.793233+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig10_Histograma_z_P = 842dae17691dc7d2...
[2026-10-30T01:55:27.795078+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig11_Histograma_z_T = 642101511ff0a86d...
[2026-10-30T01:55:27.797153+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig12_Tumbes_P_mensual = 2d0a69d64a3c0afd...
[2026-10-30T01:55:27.798900+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig13_Tumbes_T_mensual = 7772348eee3c8934...
[2026-10-30T01:55:27.800603+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig14_Correlacion_Pearson_Clima = 5182c3f94b7fa774...
[2026-10-30T01:55:27.802576+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig15_Cobertura_Espacial = ba2d1f4260c8e410...
[2026-10-30T01:55:27.804610+00:00] [INFO] [HASH] HASH registrado: EDA_EDA_Fig16_Soporte_Espacial = a43d3853ef5eb983...
[2026-10-30T01:55:27.806427+00:00] [INFO] [HASH] 

## Paso 07 — Barrera: la prueba permanece cerrada

Activa una **barrera de seguridad** antes del tuning de hiperparámetros:

- Pone `AUTORIZAR_APERTURA_PRUEBA = False`.
- Registra en el log que la prueba 2023-2025 está bloqueada.
- Crea el flag `BARRERA_PRUEBA.BLOQUEADA.flag` indicando qué celdas podrían verse afectadas.

La prueba **no puede abrirse** hasta que el paso 09 active explícitamente la bandera. El objetivo es garantizar que ningún proceso de búsqueda de hiperparámetros (paso 08) toque las observaciones de prueba.

In [9]:
# ==============================================================================
# 07: GARANTIZAR LA PRUEBA CERRADA
# ==============================================================================
AUTORIZAR_APERTURA_PRUEBA = False
log("PRUEBA 2023-2025 BLOQUEADA. Cambiar AUTORIZAR_APERTURA_PRUEBA=True solo "
    "ANTES del paso 09 (no es necesario re-ejecutar el paso 07).",
    celda="Celda07_Barrera_Prueba")
crear_flag("BARRERA_PRUEBA", "BLOQUEADA",
           {"motivo": "Búsqueda de hiperparámetros en curso; prueba no abierta",
            "celdas_afectadas": ["Celda10_Evaluacion_Prueba"]})

[2026-10-30T01:55:56.234592+00:00] [INFO] [Celda07_Barrera_Prueba] PRUEBA 2023-2025 BLOQUEADA. Cambiar AUTORIZAR_APERTURA_PRUEBA=True solo ANTES del paso 09 (no es necesario re-ejecutar el paso 07).
[2026-10-30T01:55:56.236718+00:00] [INFO] [FLAG] FLAG creado: BARRERA_PRUEBA.BLOQUEADA.flag


PosixPath('/content/resultados_tesis/20261030T002900_112193Z/00_AUDITORIA/20261030T002900_112193Z/flags/BARRERA_PRUEBA.BLOQUEADA.flag')

## Paso 08 — Búsqueda de hiperparámetros (OE2)

Ejecuta la **búsqueda exhaustiva de hiperparámetros** para los 4 algoritmos × 3 familias de información:

- **Elastic Net:** 35 combinaciones.
- **Random Forest:** 36 combinaciones.
- **XGBoost:** 54 combinaciones.
- **LSTM:** 24 combinaciones.

Total: **149 combinaciones × 3 familias = 447 ajustes**. Se entrena **solo con ENTRENAMIENTO** y se evalúa **solo con VALIDACIÓN**, usando el Brier Score macro por jurisdicción como métrica de selección.

Al final:
- Guarda `Hiperparametros_Seleccionados.json` con los mejores hiperparámetros por algoritmo y familia.
- Guarda `OE2_Resultados_Validacion.csv` con los Brier de validación.
- Crea el flag `Celda08_Hiperparametros.OK.flag` y reafirma la barrera de prueba.

La prueba sigue bloqueada hasta el paso 09.


In [10]:
# ==============================================================================
# 08: ESTIMACIÓN Y BÚSQUEDA DE HIPERPARÁMETROS (OE2)
# ==============================================================================
# Esta celda hace la búsqueda EXHAUSTIVA de hiperparámetros para los 4 algoritmos
# × 3 familias de información (F0, F1, F2).
# Entrena SOLO con ENTRENAMIENTO y evalúa SOLO con VALIDACIÓN.
# La PRUEBA (936 obs) permanece intacta hasta el siguiente paso.
#
# Grillas:
#   - Elastic Net: 7×5 = 35 combinaciones
#   - Random Forest: 2×3×3×2 = 36 combinaciones
#   - XGBoost: 3×3×3×2 = 54 combinaciones
#   - LSTM: 3×2×2×2 = 24 combinaciones
# Total: 149 combinaciones × 3 familias = 447 ajustes
#
# Registra en 00_AUDITORIA:
#   - Log de cada combinación evaluada
#   - CSV de resultados de validación por familia
#   - Hash del JSON de hiperparámetros seleccionados
#   - Flag de barrera (prueba sigue bloqueada)


# Ocultar unicamente el aviso de .obsolescencia generado por .Jupyter.
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import xgboost as xgb
import warnings
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
tf.config.experimental.enable_op_determinism()

tf.keras.utils.set_random_seed(SEED_MODELADO)

# ==============================================================================
# FUNCIONES AUXILIARES
# ==============================================================================
def obtener_listas_predictores():
    """Devuelve las listas de predictores por familia (F0, F1, F2)."""
    cols_seq = (
        [f'recaud_real_t_{k}' for k in range(13, 1, -1)] +
        [f'var_real_t_{k}'    for k in range(13, 1, -1)]
    )
    cols_f0 = (
        [f'mes_obj_{m}' for m in range(2, 13)] +
        ['ind_renta_anual', 'ctrl_pbi_var',
         'ctrl_inflacion_lima', 'ctrl_tipo_cambio']
    )
    cols_f1 = ['f1_precip', 'f1_precip_anom', 'f1_precip_acum3m',
               'f1_temp', 'f1_temp_anom', 'f1_temp_ma3m']
    cols_f2 = ['f2_oni', 'f2_icen']

    cols_tab = [f'recaud_real_t_{k}' for k in range(2,14)] + [f'var_real_t_{k}' for k in range(2,14)]
    tabulares = {
        'F0': cols_tab + cols_f0,
        'F1': cols_tab + cols_f0 + cols_f1,
        'F2': cols_tab + cols_f0 + cols_f1 + cols_f2,
    }
    lstm_static = {
        'F0': cols_f0,
        'F1': cols_f0 + cols_f1,
        'F2': cols_f0 + cols_f1 + cols_f2,
    }
    return tabulares, lstm_static, cols_seq[:12], cols_seq[12:]


def construir_lstm(n_static, units=32, dense=16, dropout=0.2, lr=1e-3):
    seq_in = Input(shape=(12, 2))
    x = LSTM(units, activation='tanh')(seq_in)
    stat_in = Input(shape=(n_static,))
    concat = Concatenate()([x, stat_in])
    x = Dense(dense, activation='relu')(concat)
    x = Dropout(dropout)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model([seq_in, stat_in], out)
    model.compile(optimizer=Adam(learning_rate=lr,clipnorm=1.0), loss='binary_crossentropy')
    return model


# Transformadores serializables y sin imputación; orden preservado.
PREPROCESADORES={}
CANDIDATOS=[]
MEJORES_CANDIDATOS={}

def ajustar_preprocesador(df,alg,fam):
    tab,stat,cr,cv=obtener_listas_predictores()
    binarias=[f'mes_obj_{m}' for m in range(2,13)]+['ind_renta_anual']
    columnas=stat[fam] if alg=='LSTM' else tab[fam]
    continuas=[c for c in columnas if c not in binarias] if alg in ['EN','LSTM'] else []
    pre={'alg':alg,'fam':fam,'columnas':columnas,'continuas':continuas,'cr':cr,'cv':cv,
         'hash_claves_train':sha256_dataframe(df[['jurisdiccion','origen_t','objetivo_t1']])}
    if continuas:
        scaler=StandardScaler().fit(df[continuas]);pre['media']=scaler.mean_.tolist();pre['escala']=scaler.scale_.tolist()
    if alg=='LSTM':
        seq=np.stack([df[cr].to_numpy(float),df[cv].to_numpy(float)],axis=-1)
        media=seq.mean(axis=(0,1));sd=seq.std(axis=(0,1),ddof=0)
        if (sd<=0).any():raise ValueError('LSTM: canal constante')
        pre['seq_media']=media.tolist();pre['seq_sd']=sd.tolist()
    return pre

def transformar(df,pre):
    columnas=pre['columnas'];x=df[columnas].to_numpy(float).copy()
    if not np.isfinite(x).all(): raise ValueError('Faltantes no permitidos: no imputar')
    for k,c in enumerate(pre['continuas']):
        j=columnas.index(c);x[:,j]=(x[:,j]-pre['media'][k])/pre['escala'][k]
    if pre['alg']=='LSTM':
        seq=np.stack([df[pre['cr']].to_numpy(float),df[pre['cv']].to_numpy(float)],axis=-1)
        seq=(seq-np.array(pre['seq_media']))/np.array(pre['seq_sd'])
        if not np.isfinite(seq).all():raise ValueError('LSTM: historia no finita')
        return [seq,x]
    return x

def construir_tabular(alg,hp):
    if alg=='EN':return LogisticRegression(penalty='elasticnet',solver='saga',C=hp['C'],l1_ratio=hp['l1_ratio'],max_iter=10000,random_state=SEED_MODELADO,n_jobs=1)
    if alg=='RF':return RandomForestClassifier(**hp,random_state=SEED_MODELADO,n_jobs=1)
    if alg=='XGB':return xgb.XGBClassifier(**hp,colsample_bytree=0.8,random_state=SEED_MODELADO,n_jobs=1,eval_metric='logloss')
    raise ValueError(alg)

def probabilidades(modelo,X,alg):
    p=modelo.predict(X,verbose=0).reshape(-1) if alg=='LSTM' else modelo.predict_proba(X)[:,1]
    if not np.isfinite(p).all() or ((p<0)|(p>1)).any():raise ValueError('Probabilidad inválida')
    return p

def evaluar_candidato(modelo,alg,fam,hp,X_tr,y_tr,X_va,y_va,dtr,dva,ajuste_lstm=False):
    import warnings,joblib
    from sklearn.exceptions import ConvergenceWarning
    inicio=time.perf_counter();cid=f'{alg}_{fam}_{len(CANDIDATOS)+1:04d}'
    row={'candidate_id':cid,'algoritmo':alg,'conjunto':fam,'hiperparametros':json.dumps(hp,sort_keys=True),
         'semilla':SEED_MODELADO,'estado':'FALLIDO','Brier_Val':None,'Brier_Train':None,
         'inicio_train':str(dtr.objetivo_t1.min()),'fin_train':str(dtr.objetivo_t1.max()),'inicio_valid':str(dva.objetivo_t1.min()),'fin_valid':str(dva.objetivo_t1.max()),'n_train':len(dtr),'n_valid':len(dva),'hash_claves_train':sha256_dataframe(dtr[['jurisdiccion','objetivo_t1']]),
         'hash_claves_valid':sha256_dataframe(dva[['jurisdiccion','objetivo_t1']]),'best_epoch':None}
    avisos=[]
    try:
        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter('always')
            if ajuste_lstm:
                es=EarlyStopping(monitor='val_loss',mode='min',min_delta=0.0,patience=10,restore_best_weights=True)
                history=modelo.fit(X_tr,y_tr,validation_data=(X_va,y_va),epochs=150,batch_size=64,callbacks=[es],verbose=0)
                losses=np.asarray(history.history['val_loss'])
                if not np.isfinite(losses).all():raise ValueError('LSTM: loss no finita')
                row['best_epoch']=int(np.argmin(losses)+1)
                row['criterio_epoca']='min val_loss BCE; primer mínimo; paciencia=10; max=150; batch=64; restore_best_weights=True'
                (OUTPUT_DIR/'Historiales').mkdir(exist_ok=True)
                (OUTPUT_DIR/'Historiales'/f'{cid}.json').write_text(json.dumps(history.history))
            else:modelo.fit(X_tr,y_tr)
            pt=probabilidades(modelo,X_tr,alg);pv=probabilidades(modelo,X_va,alg)
            row['Brier_Train']=brier_macro(dtr,pt);row['Brier_Val']=brier_macro(dva,pv)
            row['convergencia']=not any(issubclass(w.category,ConvergenceWarning) for w in avisos)
            if not row['convergencia']: raise ValueError('Candidato sin convergencia; no puede ganar')
            row['estado']='OK'

            # Predicciones identificadas: permiten reconstruir ambas métricas del mismo modelo.
            pred=pd.concat([dtr[['jurisdiccion','objetivo_t1','Y_caida']].assign(p=pt,bloque='TRAIN'),
                            dva[['jurisdiccion','objetivo_t1','Y_caida']].assign(p=pv,bloque='VALID')])
            row['hash_predicciones']=sha256_dataframe(pred)
            key=f'{alg}_{fam}'
            if key not in MEJORES_CANDIDATOS or row['Brier_Val'] < MEJORES_CANDIDATOS[key]['Brier_Val']:
                (OUTPUT_DIR/'Modelos_Seleccion').mkdir(exist_ok=True)
                pth=OUTPUT_DIR/'Modelos_Seleccion'/f'{key}.keras' if alg=='LSTM' else OUTPUT_DIR/'Modelos_Seleccion'/f'{key}.joblib'
                if alg=='LSTM':modelo.save(pth);recargado=tf.keras.models.load_model(pth)
                else:joblib.dump(modelo,pth);recargado=joblib.load(pth)
                np.testing.assert_allclose(probabilidades(recargado,X_va,alg),pv,rtol=0,atol=1e-8)
                pred.to_csv(OUTPUT_DIR/'Modelos_Seleccion'/f'{key}_Predicciones.csv',index=False)
                MEJORES_CANDIDATOS[key]=row.copy()
        return pv,row['Brier_Val'],row['best_epoch']
    except Exception as exc:
        row['error']=str(exc);row['estado']='FALLIDO';row['Brier_Val']=None
        raise
    finally:
        row['advertencias']=' | '.join(str(w.message) for w in avisos)
        row['duracion_seg']=time.perf_counter()-inicio
        CANDIDATOS.append(row)
        log(f"Candidato {len(CANDIDATOS)}/447 | {alg}-{fam} | {row['estado']} | {row['duracion_seg']:.1f}s",celda='Celda08_Hiperparametros')
        pd.DataFrame(CANDIDATOS).to_csv(OUTPUT_DIR/'Candidatos_Completos.csv',index=False)

def verificar_preprocesadores_v8(dtr):
    tab,stat,cr,cv=obtener_listas_predictores()
    for fam in ['F0','F1','F2']:
        if set(tab[fam]) != set(cr+cv+stat[fam]):raise ValueError('Equivalencia informativa fallida')
        for alg in ['EN','RF','XGB','LSTM']:
            p=PREPROCESADORES[f'{alg}_{fam}'];antes=json.dumps(p,sort_keys=True)
            x=transformar(dtr,p);despues=json.dumps(p,sort_keys=True)
            if antes!=despues:raise ValueError('Transformador mutado')
            xx=x[1] if alg=='LSTM' else x
            for c in [f'mes_obj_{m}' for m in range(2,13)]+['ind_renta_anual']:
                np.testing.assert_array_equal(xx[:,p['columnas'].index(c)],dtr[c])
    return True


# ==============================================================================
# FUNCIÓN PRINCIPAL (decorada para auditoría)
# ==============================================================================
@auditar("Celda08_Hiperparametros")
def buscar_hiperparametros(df_panel):
    log("Iniciando búsqueda de hiperparámetros (OE2)",
        celda="Celda08_Hiperparametros")
    t_inicio_total = time.time()
    CANDIDATOS.clear(); MEJORES_CANDIDATOS.clear(); PREPROCESADORES.clear()
    global DESARROLLO_CERRADO
    DESARROLLO_CERRADO = False

    # -------------------------------------------------------------------------
    # 1. Filtro y preparación de matrices
    # -------------------------------------------------------------------------
    df_dev = df_panel[
        df_panel['bloque_temporal'].isin(['ENTRENAMIENTO', 'VALIDACION'])
    ].copy()
    assert df_dev['objetivo_t1'].max() <= pd.Timestamp('2022-12-01'), \
        "Hiperparámetros no deben tocar la prueba 2023-2025"

    log(f"Filas de desarrollo: {len(df_dev)} (Train + Valid)",
        celda="Celda08_Hiperparametros")
    log(f"Train: {(df_dev['bloque_temporal']=='ENTRENAMIENTO').sum()} | "
        f"Valid: {(df_dev['bloque_temporal']=='VALIDACION').sum()}",
        celda="Celda08_Hiperparametros")

    conj_tab, conj_stat, cols_r, cols_v = obtener_listas_predictores()

    mask_tr = mascara_admisible(df_dev,'train')
    mask_va = mascara_admisible(df_dev,'valid')
    df_clean = df_dev.copy()
    dtr=df_clean.loc[mask_tr].copy();dva=df_clean.loc[mask_va].copy()
    if not np.isfinite(df_clean[conj_tab['F2']].to_numpy(float)).all() or df_clean.Y_caida.isna().any():
        raise ValueError('No se permite imputar predictores ni etiquetas')
    for fam in ['F0','F1','F2']:
        for alg in ['EN','RF','XGB','LSTM']:
            PREPROCESADORES[f'{alg}_{fam}']=ajustar_preprocesador(dtr,alg,fam)
    verificar_preprocesadores_v8(dtr)
    (OUTPUT_DIR/'Preprocesadores.json').write_text(json.dumps(PREPROCESADORES,indent=2))
    global HASH_PREPROCESADORES
    HASH_PREPROCESADORES=sha256_file(OUTPUT_DIR/'Preprocesadores.json')
    esquema={'tabulares':conj_tab,'contexto_lstm':conj_stat,'historia_lstm':[cols_r,cols_v],
             'tipos':{c:str(df_clean[c].dtype) for c in conj_tab['F2']},'orden_tabular':'reciente a antiguo',
             'orden_lstm':'antiguo a reciente','procedencia':'V7 grillas; auditoria A08/A09 orden y transformaciones',
             'desempate':'primer candidato en orden de grilla; comparación estricta <; sin tolerancia adaptativa'}
    assert [len(conj_tab[f]) for f in ['F0','F1','F2']]==[39,45,47]
    assert [len(conj_stat[f]) for f in ['F0','F1','F2']]==[15,21,23]
    assert set(conj_tab['F0']) < set(conj_tab['F1']) < set(conj_tab['F2'])
    (OUTPUT_DIR/'Esquema_Predictores.json').write_text(json.dumps(esquema,indent=2))
    registrar_hash('Esquema_Predictores',path=OUTPUT_DIR/'Esquema_Predictores.json')
    for f in ['F0','F1','F2']:
        df_clean[['jurisdiccion','origen_t','objetivo_t1','Y_caida']+conj_tab[f]].to_csv(OUTPUT_DIR/f'Matriz_Desarrollo_{f}.csv',index=False)

    resultados = []
    hiperparametros = {}

    # =========================================================================
    # 2. Bucle principal: 3 familias × 4 algoritmos
    # =========================================================================
    for fam in ['F0', 'F1', 'F2']:
        log(f"\n=== FAMILIA {fam} ===", celda="Celda08_Hiperparametros")

        cols_x = conj_tab[fam]
        X_tr = transformar(dtr,PREPROCESADORES[f'EN_{fam}'])
        y_tr = dtr['Y_caida'].to_numpy()
        X_va = transformar(dva,PREPROCESADORES[f'EN_{fam}'])
        y_va = dva['Y_caida'].to_numpy()
        df_val_eval = df_clean.loc[mask_va, ['jurisdiccion', 'Y_caida']].copy()

        # ---------------------------------------------------------------------
        # 2.1 Elastic Net — 35 combinaciones
        # ---------------------------------------------------------------------
        log(f"[{fam}] ElasticNet: 35 combinaciones...",
            celda="Celda08_Hiperparametros")
        t0 = time.time()
        best_bs, best_cfg = np.inf, None
        for c in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
            for l1 in [0.0, 0.25, 0.5, 0.75, 1.0]:
                try:
                    cfg={'C':float(c),'l1_ratio':float(l1)}
                    clf = construir_tabular('EN',cfg)
                    df_val_eval['p'], _, _ = evaluar_candidato(clf,'EN',fam,cfg,X_tr,y_tr,X_va,y_va,dtr,dva)
                    bs = ((df_val_eval['p'] - df_val_eval['Y_caida']) ** 2) \
                        .groupby(df_val_eval['jurisdiccion']).mean().mean()
                    if bs < best_bs:
                        best_bs, best_cfg = bs, (float(c), float(l1))
                except Exception as e:
                    log(f"[{fam}] EN C={c} l1={l1} ERROR: {e}",
                        nivel="WARN", celda="Celda08_Hiperparametros")

        if best_cfg is None: raise RuntimeError(f'Sin candidatos válidos: EN_{fam}; prueba bloqueada')
        log(f"[{fam}] Mejor EN: C={best_cfg[0]} l1_ratio={best_cfg[1]} "
            f"Brier={best_bs:.5f} ({time.time()-t0:.1f}s)",
            celda="Celda08_Hiperparametros")
        hiperparametros[f'EN_{fam}'] = {'C': best_cfg[0], 'l1_ratio': best_cfg[1]}
        resultados.append({'Familia_Algoritmica': 'EN', 'Familia_Informacion': fam,
                           'Brier_Val': best_bs, 'N_Combinaciones': 35})

        # ---------------------------------------------------------------------
        # 2.2 Random Forest — 36 combinaciones
        # ---------------------------------------------------------------------
        log(f"[{fam}] RandomForest: 36 combinaciones...",
            celda="Celda08_Hiperparametros")
        X_tr=transformar(dtr,PREPROCESADORES[f'RF_{fam}'])
        X_va=transformar(dva,PREPROCESADORES[f'RF_{fam}'])
        t0 = time.time()
        best_bs, best_cfg = np.inf, None
        for n_est in [500, 1000]:
            for depth in [8, 16, None]:
                for leaf in [1, 5, 10]:
                    for mf in ['sqrt', 0.5]:
                        try:
                            cfg={'n_estimators':int(n_est),'max_depth':depth,'min_samples_leaf':int(leaf),'max_features':mf}
                            rf = construir_tabular('RF',cfg)
                            df_val_eval['p'], _, _ = evaluar_candidato(rf,'RF',fam,cfg,X_tr,y_tr,X_va,y_va,dtr,dva)
                            bs = ((df_val_eval['p'] - df_val_eval['Y_caida']) ** 2) \
                                .groupby(df_val_eval['jurisdiccion']).mean().mean()
                            if bs < best_bs:
                                best_bs = bs
                                best_cfg = (int(n_est), depth, int(leaf), mf)
                        except Exception as e:
                            log(f"[{fam}] RF n={n_est} d={depth} ERROR: {e}",
                                nivel="WARN", celda="Celda08_Hiperparametros")

        if best_cfg is None: raise RuntimeError(f'Sin candidatos válidos: RF_{fam}; prueba bloqueada')
        log(f"[{fam}] Mejor RF: n={best_cfg[0]} depth={best_cfg[1]} "
            f"leaf={best_cfg[2]} mf={best_cfg[3]} Brier={best_bs:.5f} "
            f"({time.time()-t0:.1f}s)",
            celda="Celda08_Hiperparametros")
        hiperparametros[f'RF_{fam}'] = {
            'n_estimators': best_cfg[0], 'max_depth': best_cfg[1],
            'min_samples_leaf': best_cfg[2], 'max_features': best_cfg[3],
        }
        resultados.append({'Familia_Algoritmica': 'RF', 'Familia_Informacion': fam,
                           'Brier_Val': best_bs, 'N_Combinaciones': 36})

        # ---------------------------------------------------------------------
        # 2.3 XGBoost — 54 combinaciones
        # ---------------------------------------------------------------------
        log(f"[{fam}] XGBoost: 54 combinaciones...",
            celda="Celda08_Hiperparametros")
        t0 = time.time()
        best_bs, best_cfg = np.inf, None
        for n_est in [300, 600, 900]:
            for depth in [2, 4, 6]:
                for lr in [0.03, 0.07, 0.12]:
                    for subs in [0.8, 1.0]:
                        try:
                            cfg={'n_estimators':int(n_est),'max_depth':int(depth),'learning_rate':float(lr),'subsample':float(subs)}
                            xgb_m = construir_tabular('XGB',cfg)
                            df_val_eval['p'], _, _ = evaluar_candidato(xgb_m,'XGB',fam,cfg,X_tr,y_tr,X_va,y_va,dtr,dva)
                            bs = ((df_val_eval['p'] - df_val_eval['Y_caida']) ** 2) \
                                .groupby(df_val_eval['jurisdiccion']).mean().mean()
                            if bs < best_bs:
                                best_bs = bs
                                best_cfg = (int(n_est), int(depth), float(lr), float(subs))
                        except Exception as e:
                            log(f"[{fam}] XGB n={n_est} d={depth} ERROR: {e}",
                                nivel="WARN", celda="Celda08_Hiperparametros")

        if best_cfg is None: raise RuntimeError(f'Sin candidatos válidos: XGB_{fam}; prueba bloqueada')
        log(f"[{fam}] Mejor XGB: n={best_cfg[0]} depth={best_cfg[1]} "
            f"lr={best_cfg[2]} subs={best_cfg[3]} Brier={best_bs:.5f} "
            f"({time.time()-t0:.1f}s)",
            celda="Celda08_Hiperparametros")
        hiperparametros[f'XGB_{fam}'] = {
            'n_estimators': best_cfg[0], 'max_depth': best_cfg[1],
            'learning_rate': best_cfg[2], 'subsample': best_cfg[3],
        }
        resultados.append({'Familia_Algoritmica': 'XGB', 'Familia_Informacion': fam,
                           'Brier_Val': best_bs, 'N_Combinaciones': 54})

        # ---------------------------------------------------------------------
        # 2.4 LSTM — 24 combinaciones
        # ---------------------------------------------------------------------
        log(f"[{fam}] LSTM: 24 combinaciones...",
            celda="Celda08_Hiperparametros")
        t0 = time.time()

        Xs_tr,Xst_tr=transformar(dtr,PREPROCESADORES[f'LSTM_{fam}'])
        Xs_va,Xst_va=transformar(dva,PREPROCESADORES[f'LSTM_{fam}'])

        best_bs, best_cfg = np.inf, None
        for units in [16, 32, 64]:
            for dense in [16, 32]:
                for dropout in [0.0, 0.2]:
                    for lr in [0.0003, 0.001]:
                        try:
                            tf.keras.backend.clear_session()
                            tf.keras.utils.set_random_seed(SEED_MODELADO)
                            m = construir_lstm(
                                Xst_tr.shape[1],
                                units=units, dense=dense,
                                dropout=dropout, lr=lr,
                            )
                            cfg={'units':int(units),'dense':int(dense),'dropout':float(dropout),'learning_rate':float(lr)}
                            df_val_eval['p'], _, best_epoch = evaluar_candidato(m,'LSTM',fam,cfg,
                                [Xs_tr,Xst_tr],y_tr,[Xs_va,Xst_va],y_va,dtr,dva,ajuste_lstm=True)
                            bs = ((df_val_eval['p'] - df_val_eval['Y_caida']) ** 2) \
                                .groupby(df_val_eval['jurisdiccion']).mean().mean()
                            if bs < best_bs:
                                best_bs = bs
                                best_cfg = (int(units), int(dense),
                                            float(dropout), float(lr), int(best_epoch))
                        except Exception as e:
                            log(f"[{fam}] LSTM u={units} d={dense} ERROR: {e}",
                                nivel="WARN", celda="Celda08_Hiperparametros")
                        finally:
                            tf.keras.backend.clear_session()

        if best_cfg is None: raise RuntimeError(f'Sin candidatos válidos: LSTM_{fam}; prueba bloqueada')
        log(f"[{fam}] Mejor LSTM: units={best_cfg[0]} dense={best_cfg[1]} "
            f"dropout={best_cfg[2]} lr={best_cfg[3]} Brier={best_bs:.5f} "
            f"({time.time()-t0:.1f}s)",
            celda="Celda08_Hiperparametros")
        hiperparametros[f'LSTM_{fam}'] = {
            'units': best_cfg[0], 'dense': best_cfg[1],
            'dropout': best_cfg[2], 'learning_rate': best_cfg[3], 'best_epoch':best_cfg[4],
        }
        resultados.append({'Familia_Algoritmica': 'LSTM', 'Familia_Informacion': fam,
                           'Brier_Val': best_bs, 'N_Combinaciones': 24})

    # =========================================================================
    # 3. Cierre: guardar hiperparámetros, resultados y flag
    # =========================================================================
    log(f"\n=== BÚSQUEDA COMPLETADA en "
        f"{(time.time()-t_inicio_total)/60:.1f} minutos ===",
        celda="Celda08_Hiperparametros")

    df_res = pd.DataFrame(resultados).sort_values(
        ['Familia_Algoritmica', 'Familia_Informacion']
    ).reset_index(drop=True)

    if len(CANDIDATOS)!=447 or len(MEJORES_CANDIDATOS)!=12:
        raise ValueError('Registro de búsqueda incompleto; no abrir prueba')
    for i,r in df_res.iterrows():
        key=f"{r.Familia_Algoritmica}_{r.Familia_Informacion}"
        best=MEJORES_CANDIDATOS[key]
        np.testing.assert_allclose(r.Brier_Val,best['Brier_Val'],atol=1e-12,rtol=0)
        df_res.loc[i,'Brier_Train']=best['Brier_Train'];df_res.loc[i,'candidate_id']=best['candidate_id']
        df_res.loc[i,'model_id']=key+'_seleccion';df_res.loc[i,'N_Train']=len(dtr);df_res.loc[i,'N_Valid']=len(dva)
        df_res.loc[i,'hash_predicciones']=best['hash_predicciones']
    if sha256_file(OUTPUT_DIR/'Preprocesadores.json')!=HASH_PREPROCESADORES:
        raise ValueError('Preprocesadores cambiaron durante selección')

    # Guardar JSON de hiperparámetros (estructura unificada)
    hp_path = OUTPUT_DIR / "Hiperparametros_Seleccionados.json"
    with open(hp_path, "w", encoding="utf-8") as f:
        json.dump(hiperparametros, f, ensure_ascii=False, indent=2)

    # Guardar CSV de resultados de validación por configuración
    res_path = OUTPUT_DIR / "OE2_Resultados_Validacion.csv"
    df_res.to_csv(res_path, index=False)

    # Registrar hashes
    registrar_hash("Hiperparametros_Seleccionados", path=hp_path)
    registrar_hash("OE2_Resultados_Validacion", df=df_res)

    # Guardar hiperparámetros en memoria para la Celda 6
    # (Ya son diccionarios, listos para usar con **hp_optimos[k])

    log("Hiperparámetros cerrados y guardados",
        celda="Celda08_Hiperparametros")
    display(df_res.round(5))

    return df_res, hiperparametros

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
df_res_oe2, hp_optimos = buscar_hiperparametros(df_panel)

# Reconstruir rutas, el decorador ya creó el flag OK
hp_path  = OUTPUT_DIR / "Hiperparametros_Seleccionados.json"
res_path = OUTPUT_DIR / "OE2_Resultados_Validacion.csv"

log("BARRERA: prueba 2023-2025 permanece BLOQUEADA. "
    "Cambiar AUTORIZAR_APERTURA_PRUEBA=True solo antes del paso 10.",
    celda="Celda08_Hiperparametros")
crear_flag("BARRERA_PRUEBA", "BLOQUEADA_DESPUES_TUNING",
           {"motivo": "Hiperparámetros cerrados; Paso 09 aún no ejecutada"})

print(f"\n{'='*70}")
print(f"PASO 08 COMPLETADO — HIPERPARÁMETROS CERRADOS")
print(f"{'='*70}")
print(f"Configuraciones evaluadas: {len(df_res_oe2)}")
for alg in ['EN', 'RF', 'XGB', 'LSTM']:
    sub = df_res_oe2[df_res_oe2['Familia_Algoritmica'] == alg]
    best = sub.loc[sub['Brier_Val'].idxmin()]
    print(f"  {alg:4s} mejor en {best['Familia_Informacion']}: "
          f"Brier={best['Brier_Val']:.5f}")
print(f"\nHiperparámetros : {hp_path}")
print(f"Resultados      : {res_path}")
print(f"Auditoría       : {AUDIT_DIR}")
print(f"\nPRÓXIMO PASO: cambiar AUTORIZAR_APERTURA_PRUEBA=True (paso 09) "
      f"y ejecutar el paso 10.")
DESARROLLO_CERRADO=True
HASH_HP_CERRADO=sha256_file(hp_path)


[2026-10-30T01:56:06.670538+00:00] [INFO] [Celda08_Hiperparametros] INICIO buscar_hiperparametros
[2026-10-30T01:56:06.671119+00:00] [INFO] [Celda08_Hiperparametros] Iniciando búsqueda de hiperparámetros (OE2)
[2026-10-30T01:56:06.676422+00:00] [INFO] [Celda08_Hiperparametros] Filas de desarrollo: 4940 (Train + Valid)
[2026-10-30T01:56:06.678812+00:00] [INFO] [Celda08_Hiperparametros] Train: 4004 | Valid: 936
[2026-10-30T01:56:07.096228+00:00] [INFO] [HASH] HASH registrado: Esquema_Predictores = 8cc11e670d537502...
[2026-10-30T01:56:09.286224+00:00] [INFO] [Celda08_Hiperparametros] 
=== FAMILIA F0 ===
[2026-10-30T01:56:09.292613+00:00] [INFO] [Celda08_Hiperparametros] [F0] ElasticNet: 35 combinaciones...
[2026-10-30T01:56:09.546245+00:00] [INFO] [Celda08_Hiperparametros] Candidato 1/447 | EN-F0 | OK | 0.3s
[2026-10-30T01:56:09.695054+00:00] [INFO] [Celda08_Hiperparametros] Candidato 2/447 | EN-F0 | OK | 0.1s
[2026-10-30T01:56:09.831920+00:00] [INFO] [Celda08_Hiperparametros] Candidato 

,Familia_Algoritmica,Familia_Informacion,Brier_Val,N_Combinaciones,Brier_Train,candidate_id,model_id,N_Train,N_Valid,hash_predicciones
0,EN,F0,0.20849,35,0.19587,EN_F0_0005,EN_F0_seleccion,3952.0,884.0,ff7a619ccf6512a21e1423a2aac9ab208ba9f94fad7d79...
1,EN,F1,0.20849,35,0.19587,EN_F1_0154,EN_F1_seleccion,3952.0,884.0,81b2580febdf41a3c4eccff58076decdd4fbdece6af52a...
2,EN,F2,0.20737,35,0.19581,EN_F2_0303,EN_F2_seleccion,3952.0,884.0,86bafb353906e5c4da0949c669e1c180a4a53439bbe684...
3,LSTM,F0,0.18774,24,0.20477,LSTM_F0_0137,LSTM_F0_seleccion,3952.0,884.0,01c394b7e4fbcefec6edd7753746d89ee31255c0cf2743...
4,LSTM,F1,0.18574,24,0.19426,LSTM_F1_0286,LSTM_F1_seleccion,3952.0,884.0,ac51f44b99a77271d01802c33e2a6d3255fb0dbe5b2571...
5,LSTM,F2,0.18485,24,0.19060,LSTM_F2_0435,LSTM_F2_seleccion,3952.0,884.0,86b7be04174bc4317e8f3b418ec208f0a813989f5a2f9b...
6,RF,F0,0.20238,36,0.12118,RF_F0_0046,RF_F0_seleccion,3952.0,884.0,8e93f6ef01251c8946fa425c86f7998651e7c9af820e1f...
7,RF,F1,0.20142,36,0.12043,RF_F1_0201,RF_F1_seleccion,3952.0,884.0,fa4927237438c3ed00dd1163e89c67d7d9836906a624ee...
8,RF,F2,0.20062,36,0.11938,RF_F2_0368,RF_F2_seleccion,3952.0,884.0,e3281675108737da175ef2389cca637193e38fb00e069e...
9,XGB,F0,0.20580,54,0.16606,XGB_F0_0072,XGB_F0_seleccion,3952.0,884.0,c3211808848a547ecaba5d752d5f02d46fc6857ee7132c...


[2026-10-30T03:26:14.088907+00:00] [INFO] [FLAG] FLAG creado: Celda08_Hiperparametros.OK.flag
[2026-10-30T03:26:14.090008+00:00] [INFO] [Celda08_Hiperparametros] FIN buscar_hiperparametros — 5407.42s
[2026-10-30T03:26:14.090727+00:00] [INFO] [Celda08_Hiperparametros] BARRERA: prueba 2023-2025 permanece BLOQUEADA. Cambiar AUTORIZAR_APERTURA_PRUEBA=True solo antes del paso 10.
[2026-10-30T03:26:14.091380+00:00] [INFO] [FLAG] FLAG creado: BARRERA_PRUEBA.BLOQUEADA_DESPUES_TUNING.flag

PASO 08 COMPLETADO — HIPERPARÁMETROS CERRADOS
Configuraciones evaluadas: 12
  EN   mejor en F2: Brier=0.20737
  RF   mejor en F2: Brier=0.20062
  XGB  mejor en F1: Brier=0.20454
  LSTM mejor en F2: Brier=0.18485

Hiperparámetros : /content/resultados_tesis/20261030T002900_112193Z/Hiperparametros_Seleccionados.json
Resultados      : /content/resultados_tesis/20261030T002900_112193Z/OE2_Resultados_Validacion.csv
Auditoría       : /content/resultados_tesis/20261030T002900_112193Z/00_AUDITORIA/20261030T002900_112

## Paso 09 — Autorización manual de apertura de la prueba

Activa **manualmente** la bandera `AUTORIZAR_APERTURA_PRUEBA = True` para habilitar la evaluación final sobre la prueba:

- Registra en el log que la apertura fue autorizada manualmente.
- Crea el flag `Celda09_Apertura_Prueba.AUTORIZADA.flag` con la marca temporal.

Sin este paso, la celda 10 lanzará un `RuntimeError` de "PRUEBA NO ABIERTA". La activación debe ser **deliberada y explícita** por parte del analista, después de haber cerrado todos los pasos de desarrollo.


In [11]:
# ==============================================================================
# 09: AUTORIZACIÓN DE APERTURA DE PRUEBA (GATE MANUAL)
# ==============================================================================
# Bandera manual que habilita el paso 09. Sin esto, el paso 09 lanza RuntimeError.
# La prueba 2023-2025 permanece bloqueada hasta que se active esta bandera.

try:
    AUTORIZAR_APERTURA_PRUEBA = True
    log("Bandera de apertura activada manualmente (paso 09)",
        celda="Celda09_Apertura_Prueba")
    crear_flag("Celda09_Apertura_Prueba", "AUTORIZADA",
               {"autorizacion_manual": True,
                "run_id": RUN_ID,
                "timestamp": _ahora()})
    print(">>> Bandera AUTORIZAR_APERTURA_PRUEBA = True")
    print(">>> PRUEBA 2023-2025 habilitada para el paso 10.")
except Exception as e:
    registrar_error("Celda09_Apertura_Prueba", e)
    raise

[2026-10-30T03:26:56.334931+00:00] [INFO] [Celda09_Apertura_Prueba] Bandera de apertura activada manualmente (paso 09)
[2026-10-30T03:26:56.336821+00:00] [INFO] [FLAG] FLAG creado: Celda09_Apertura_Prueba.AUTORIZADA.flag
>>> Bandera AUTORIZAR_APERTURA_PRUEBA = True
>>> PRUEBA 2023-2025 habilitada para el paso 10.


## Paso 10 — Evaluación final en prueba (OE3 y OE4)

Abre la prueba 2023-2025 (936 observaciones) y realiza la **evaluación final**:

1. **Entrena los 12 modelos** (4 algoritmos × 3 familias) con ENTRENAMIENTO + VALIDACIÓN usando los hiperparámetros cerrados en el paso 08.
2. **Predice** sobre la prueba y guarda `Predicciones_Prueba_12_Modelos.csv`.
3. Calcula los **contrastes de valor incremental** ΔBrier (F1-F0, F2-F1, F2-F0) por algoritmo y en resumen multimodelo, con **IC95% vía bootstrap de bloques móviles**.
4. Ejecuta la **sensibilidad sin Lima Metropolitana** (25 jurisdicciones).
5. Construye la **matriz de verificación de hipótesis** (H1, HE1, HE2, HE3).

Se registra todo con hashes y flags `Celda10_Evaluacion_Prueba.OK.flag` y `Celda10_Contrastes_Bootstrap.OK.flag`.

**Exportación sin pérdida:** las probabilidades se guardan con 17 cifras significativas y su lectura se comprueba mediante igualdad exacta antes de calcular métricas.


In [12]:
# ==============================================================================
# 10: EVALUACIÓN FINAL EN PRUEBA (OE3 y OE4)
# ==============================================================================
# Requiere AUTORIZAR_APERTURA_PRUEBA=True (paso 08).
# Entrena cada uno de los 12 modelos (4 algos × 3 familias) con
# ENTRENAMIENTO + VALIDACIÓN usando los hiperparámetros cerrados en el paso 07,
# predice sobre PRUEBA (936 obs) y calcula:
#   - Predicciones congeladas (CSV con hash)
#   - Contrastes de Δ Brier con bootstrap de bloques móviles
#   - Matriz de verificación de hipótesis (H1, HE1, HE2, HE3)
# Todo queda registrado en 00_AUDITORIA vía @auditar.

# Ocultar unicamente el aviso de .obsolescencia generado por .Jupyter.
import warnings

warnings.filterwarnings(
    action="ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.optimizers import Adam
import xgboost as xgb

tf.keras.utils.set_random_seed(SEED_MODELADO)

# ------------------------------------------------------------------------------
# GATE: la prueba no se abre sin autorización explícita
# ------------------------------------------------------------------------------
if 'AUTORIZAR_APERTURA_PRUEBA' not in globals() or AUTORIZAR_APERTURA_PRUEBA is not True:
    raise RuntimeError(
        "PRUEBA NO ABIERTA. Ejecuta el paso 09 "
        "(`AUTORIZAR_APERTURA_PRUEBA = True`) y vuelve a correr esta celda."
    )

log(">>> ABRIENDO PRUEBA 2023-2025 <<<", celda="Celda10_Evaluacion_Prueba")
df_test = df_panel[df_panel['bloque_temporal'] == 'PRUEBA'].copy()
log(f"Filas de prueba: {len(df_test)} | "
    f"Rango: {df_test['objetivo_t1'].min().date()} → "
    f"{df_test['objetivo_t1'].max().date()}",
    celda="Celda10_Evaluacion_Prueba")


# ==============================================================================
# 10.1 ENTRENAMIENTO + PREDICCIÓN SOBRE PRUEBA (12 modelos)
# ==============================================================================
@auditar("Celda10_Evaluacion_Prueba")
def evaluar_modelos_finales(df_test, hp_optimos):
    """Entrena los 12 modelos con train+valid y predice sobre prueba."""
    conj_tab, conj_stat, cols_r, cols_v = obtener_listas_predictores()

    import joblib
    if sha256_file(OUTPUT_DIR/'Hiperparametros_Seleccionados.json')!=HASH_HP_CERRADO:
        raise ValueError('Hiperparámetros alterados después del cierre')
    if sha256_file(OUTPUT_DIR/'Preprocesadores.json')!=HASH_PREPROCESADORES:
        raise ValueError('Transformaciones alteradas después del cierre')
    preprocesadores=json.loads((OUTPUT_DIR/'Preprocesadores.json').read_text())

    # Recomputar sobre TODO el bloque nominal, recuperando nov-dic 2019.
    df_pre=df_panel.loc[mascara_admisible(df_panel,'reajuste')].copy()
    if (df_pre.fecha_disponible_Y>CORTE_SELECCION).any():raise ValueError('Etiqueta futura en reajuste')
    modelos_dir=OUTPUT_DIR/'Modelos_Reajuste';modelos_dir.mkdir(exist_ok=True)
    congelados=[];registro=[]
    # Fase 1: todos los ajustes preprueba y su persistencia, sin consultar y_test.
    for fam in ['F0','F1','F2']:
        for alg in ['EN','RF','XGB','LSTM']:
            key=f'{alg}_{fam}';hp=hp_optimos[key];pre=preprocesadores[key]
            X=transformar(df_pre,pre);y=df_pre.Y_caida.to_numpy()
            inicio=time.perf_counter()
            with warnings.catch_warnings(record=True) as avisos:
                warnings.simplefilter('always')
                if alg=='LSTM':
                    tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(SEED_MODELADO)
                    modelo=construir_lstm(len(pre['columnas']),units=hp['units'],dense=hp['dense'],dropout=hp['dropout'],lr=hp['learning_rate'])
                    modelo.fit(X,y,epochs=int(hp['best_epoch']),batch_size=64,verbose=0)
                    path=modelos_dir/f'{key}.keras';modelo.save(path)
                    cargado=tf.keras.models.load_model(path)
                else:
                    modelo=construir_tabular(alg,hp);modelo.fit(X,y)
                    path=modelos_dir/f'{key}.joblib';joblib.dump(modelo,path);cargado=joblib.load(path)
                original=probabilidades(modelo,X,alg);recarga=probabilidades(cargado,X,alg)
                np.testing.assert_allclose(original,recarga,rtol=0,atol=1e-8)
                # Interpretabilidad nativa, sin causalidad ni métodos nuevos.
                if hasattr(modelo,'coef_'):
                    pd.DataFrame({'predictor':pre['columnas'],'coeficiente_logit':modelo.coef_[0]}).to_csv(modelos_dir/f'{key}_Coeficientes.csv',index=False)
                elif hasattr(modelo,'feature_importances_'):
                    pd.DataFrame({'predictor':pre['columnas'],'importancia_nativa':modelo.feature_importances_}).to_csv(modelos_dir/f'{key}_Importancias.csv',index=False)
            registro.append({'model_id':key+'_reajuste','n':len(df_pre),'corte':str(CORTE_SELECCION),
                'hash_claves':sha256_dataframe(df_pre[['jurisdiccion','origen_t','objetivo_t1']]),
                'Brier_Ajuste_No_Validacion':brier_macro(df_pre,original),'duracion_seg':time.perf_counter()-inicio,
                'advertencias':' | '.join(str(w.message) for w in avisos),'sha256':sha256_file(path),'path':str(path),
                'epochs':hp.get('best_epoch'),'recarga_verificada':True})
            congelados.append((alg,fam,path,sha256_file(path)))
            del modelo,cargado
    pd.DataFrame(registro).to_csv(OUTPUT_DIR/'Registro_Reajuste.csv',index=False)
    (OUTPUT_DIR/'Modelos_Congelados.json').write_text(json.dumps(registro,indent=2))

    # Fase 2: cargar los doce modelos congelados y generar predicciones.
    # La etiqueta se une al resultado SOLO después de producir todas las probabilidades.
    df_test_preds=df_test[['jurisdiccion','objetivo_t1']].copy()
    for alg,fam,path,digest in congelados:
        if sha256_file(path)!=digest:raise ValueError('Modelo cambiado')
        pre=preprocesadores[f'{alg}_{fam}'];X=transformar(df_test,pre)
        modelo=tf.keras.models.load_model(path) if alg=='LSTM' else joblib.load(path)
        df_test_preds[f'prob_{alg}_{fam}']=probabilidades(modelo,X,alg)
        del modelo
    df_test_preds.insert(2,'Y_caida',df_test.Y_caida.to_numpy())
    if json.dumps(preprocesadores,sort_keys=True)!=json.dumps(PREPROCESADORES,sort_keys=True):
        raise ValueError('Transformadores cambiaron tras entrenamiento')
    validar_predicciones(df_test_preds)
    return df_test_preds


# ==============================================================================
# 10.2 CONTRASTES DE VALOR INCREMENTAL CON BOOTSTRAP DE BLOQUES MÓVILES
# ==============================================================================
@auditar("Celda10_Contrastes_Bootstrap")
def calcular_contrastes(df_preds):
    """Calcula Δ Brier (F1-F0, F2-F1, F2-F0) por algoritmo + resumen multimodelo,
    con IC95% por bootstrap de bloques móviles. Incluye sensibilidad sin Lima."""

    validar_predicciones(df_preds)
    indices = indices_mbb(df_preds.objetivo_t1.nunique(),6,2000,SEED_BOOTSTRAP)
    np.save(OUTPUT_DIR / 'Indices_Bootstrap.npy',indices)
    def moving_block_bootstrap(serie):
        medias = np.asarray(serie)[indices].mean(axis=1)
        return np.percentile(medias,[2.5,97.5]), float(np.mean(serie))

    meses = sorted(df_preds['objetivo_t1'].unique())
    algos = ['EN', 'RF', 'XGB', 'LSTM']

    def get_serie_mensual(col1, col2, excluir_lima=False):
        sub = (df_preds[df_preds['jurisdiccion'] != 'LIMA METROPOLITANA']
               if excluir_lima else df_preds)
        brier_m = []
        for m in meses:
            b = sub[sub['objetivo_t1'] == m]
            bs1 = (((b[col1] - b['Y_caida']) ** 2)
                   .groupby(b['jurisdiccion']).mean().mean())
            bs2 = (((b[col2] - b['Y_caida']) ** 2)
                   .groupby(b['jurisdiccion']).mean().mean())
            brier_m.append(bs1 - bs2)
        return np.array(brier_m)

    contrastes = []
    for escenario, excluir in [
        ('Panel 26 Jurisdicciones', False),
        ('Sensibilidad 25 Jurisdicciones', True),
    ]:
        for alg in algos:
            for contraste, c1, c2 in [
                ('Aporte Territorial (F1-F0)', f'prob_{alg}_F1', f'prob_{alg}_F0'),
                ('Aporte ENOS (F2-F1)', f'prob_{alg}_F2', f'prob_{alg}_F1'),
                ('Aporte Total (F2-F0)', f'prob_{alg}_F2', f'prob_{alg}_F0'),
            ]:
                dif = get_serie_mensual(c1, c2, excluir)
                (ic_l, ic_u), media = moving_block_bootstrap(dif)
                contrastes.append({
                    'Escenario': escenario,
                    'Familia_Algoritmica': alg,
                    'Contraste': contraste,
                    'Delta_Brier': media,
                    'Centro_Bootstrap_Diagnostico': float(dif[indices].mean()),
                    'IC95_Inferior': ic_l,
                    'IC95_Superior': ic_u,
                })

        # Resumen multimodelo (promedio de las 4 series por algoritmo)
        for contraste, idx in [
            ('Aporte Territorial (F1-F0)', 0),
            ('Aporte ENOS (F2-F1)', 1),
            ('Aporte Total (F2-F0)', 2),
        ]:
            series_alg = []
            for alg in algos:
                c1, c2 = [
                    (f'prob_{alg}_F1', f'prob_{alg}_F0'),
                    (f'prob_{alg}_F2', f'prob_{alg}_F1'),
                    (f'prob_{alg}_F2', f'prob_{alg}_F0'),
                ][idx]
                series_alg.append(get_serie_mensual(c1, c2, excluir))
            mm_serie = np.mean(series_alg, axis=0)
            (ic_l, ic_u), media = moving_block_bootstrap(mm_serie)
            contrastes.append({
                'Escenario': escenario,
                'Familia_Algoritmica': 'RESUMEN MULTIMODELO',
                'Contraste': contraste,
                'Delta_Brier': media,
                'Centro_Bootstrap_Diagnostico': float(mm_serie[indices].mean()),
                'IC95_Inferior': ic_l,
                'IC95_Superior': ic_u,
            })

    df_c = pd.DataFrame(contrastes)
    verificar_contrastes(df_preds, df_c)
    print("\n### CONTRASTES DE VALOR INCREMENTAL (Δ Brier Score)")
    display(df_c.round(5))

    # ---------------------------------------------------------------------
    # Matriz de hipótesis
    # ---------------------------------------------------------------------
    resumen_mm = df_c[
        (df_c['Familia_Algoritmica'] == 'RESUMEN MULTIMODELO') &
        (df_c['Escenario'] == 'Panel 26 Jurisdicciones')
    ]
    total_delta = resumen_mm[
        resumen_mm['Contraste'] == 'Aporte Total (F2-F0)'
    ].iloc[0]
    terr_delta = resumen_mm[
        resumen_mm['Contraste'] == 'Aporte Territorial (F1-F0)'
    ].iloc[0]
    enos_delta = resumen_mm[
        resumen_mm['Contraste'] == 'Aporte ENOS (F2-F1)'
    ].iloc[0]

    sub_alg = df_c[
        (df_c['Familia_Algoritmica'] != 'RESUMEN MULTIMODELO') &
        (df_c['Contraste'] == 'Aporte Total (F2-F0)') &
        (df_c['Escenario'] == 'Panel 26 Jurisdicciones')
    ]
    n_negativos = int((sub_alg['Delta_Brier'] < 0).sum())

    sin_lima_delta = df_c[
        (df_c['Familia_Algoritmica'] == 'RESUMEN MULTIMODELO') &
        (df_c['Contraste'] == 'Aporte Total (F2-F0)') &
        (df_c['Escenario'] == 'Sensibilidad 25 Jurisdicciones')
    ].iloc[0]

    hipotesis = pd.DataFrame([
        {'Hipotesis': 'H1 (Total)',
         'Delta_Brier': total_delta['Delta_Brier'],
         'IC95': f"[{total_delta['IC95_Inferior']:.4f}, {total_delta['IC95_Superior']:.4f}]",
         'Mejora_Puntual': bool(total_delta['Delta_Brier'] < 0),
         'Veredicto': 'CUMPLE' if total_delta['IC95_Superior'] < 0 else 'EVIDENCIA INSUFICIENTE'},
        {'Hipotesis': 'HE1 (Territorial)',
         'Delta_Brier': terr_delta['Delta_Brier'],
         'IC95': f"[{terr_delta['IC95_Inferior']:.4f}, {terr_delta['IC95_Superior']:.4f}]",
         'Mejora_Puntual': bool(terr_delta['Delta_Brier'] < 0),
         'Veredicto': 'CUMPLE' if terr_delta['IC95_Superior'] < 0 else 'EVIDENCIA INSUFICIENTE'},
        {'Hipotesis': 'HE2 (ENOS)',
         'Delta_Brier': enos_delta['Delta_Brier'],
         'IC95': f"[{enos_delta['IC95_Inferior']:.4f}, {enos_delta['IC95_Superior']:.4f}]",
         'Mejora_Puntual': bool(enos_delta['Delta_Brier'] < 0),
         'Veredicto': 'CUMPLE' if enos_delta['IC95_Superior'] < 0 else 'EVIDENCIA INSUFICIENTE'},
        {'Hipotesis': 'HE3 (Robustez)',
         'Delta_Brier': sin_lima_delta['Delta_Brier'],
         'IC95': (f"Alg. Neg: {n_negativos}/4 | "
                  f"IC Sin Lima: [{sin_lima_delta['IC95_Inferior']:.4f}, "
                  f"{sin_lima_delta['IC95_Superior']:.4f}]"),
         'Veredicto': ('CUMPLE' if (n_negativos >= 3 and
                                    sin_lima_delta['Delta_Brier'] < 0)
                       else 'NO CUMPLE')},
    ])

    print("\n### MATRIZ DE VERIFICACIÓN DE HIPÓTESIS")
    display(hipotesis)

    return df_c, hipotesis


# ==============================================================================
# 10.3 EJECUCIÓN + REGISTRO DE ARTEFACTOS
# ==============================================================================
df_preds = evaluar_modelos_finales(df_test, hp_optimos)

pred_path = OUTPUT_DIR / "Predicciones_Prueba_12_Modelos.csv"
df_preds.to_csv(pred_path, index=False, float_format='%.17g')
registrar_hash("Predicciones_Prueba_12_Modelos", path=pred_path)

# Evaluar solamente después de congelar el CSV; tolerancia de serialización fija.
validar_predicciones(df_preds)
df_leido = pd.read_csv(pred_path,parse_dates=['objetivo_t1'],float_precision='round_trip')
for c in [x for x in df_preds if x.startswith('prob_')]:
    np.testing.assert_array_equal(df_leido[c].to_numpy(),df_preds[c].to_numpy().astype(np.float64))
df_preds = df_leido
df_contrastes, df_hipotesis = calcular_contrastes(df_preds)
df_hipotesis.to_csv(OUTPUT_DIR / 'Hipotesis_V8.csv',index=False)
exportar_diagnosticos(df_preds)
PREDICCIONES_CONGELADAS_HASH = sha256_file(pred_path)

contr_path = OUTPUT_DIR / "OE3_OE4_Contrastes.csv"
df_contrastes.to_csv(contr_path, index=False)
registrar_hash("OE3_OE4_Contrastes", path=contr_path)

crear_flag("Celda10_Evaluacion_Prueba", "OK",
           {"filas_prueba": len(df_test),
            "n_predicciones": len(df_preds.columns) - 3,
            "hipotesis_cumplidas": int(
                (df_hipotesis['Veredicto'] == 'CUMPLE').sum()
            )})

print(f"\n{'='*70}")
print(f"PASO 10 COMPLETADO — PRUEBA EVALUADA (OE3 y OE4)")
print(f"{'='*70}")
print(f"Predicciones : {pred_path}")
print(f"Contrastes   : {contr_path}")
print(f"Hipótesis    :")
display(df_hipotesis)
print(f"\nAuditoría en: {AUDIT_DIR}")

[2026-10-30T03:27:03.951300+00:00] [INFO] [Celda10_Evaluacion_Prueba] >>> ABRIENDO PRUEBA 2023-2025 <<<
[2026-10-30T03:27:03.959240+00:00] [INFO] [Celda10_Evaluacion_Prueba] Filas de prueba: 936 | Rango: 2023-01-01 → 2025-12-01
[2026-10-30T03:27:03.962911+00:00] [INFO] [Celda10_Evaluacion_Prueba] INICIO evaluar_modelos_finales
[2026-10-30T03:28:45.320271+00:00] [INFO] [FLAG] FLAG creado: Celda10_Evaluacion_Prueba.OK.flag
[2026-10-30T03:28:45.320793+00:00] [INFO] [Celda10_Evaluacion_Prueba] FIN evaluar_modelos_finales — 101.36s
[2026-10-30T03:28:45.355437+00:00] [INFO] [HASH] HASH registrado: Predicciones_Prueba_12_Modelos = 8d23aa857b1bffd9...
[2026-10-30T03:28:45.396242+00:00] [INFO] [Celda10_Contrastes_Bootstrap] INICIO calcular_contrastes

### CONTRASTES DE VALOR INCREMENTAL (Δ Brier Score)


,Escenario,Familia_Algoritmica,Contraste,Delta_Brier,Centro_Bootstrap_Diagnostico,IC95_Inferior,IC95_Superior
0,Panel 26 Jurisdicciones,EN,Aporte Territorial (F1-F0),0.00195,0.00197,-0.00315,0.00685
1,Panel 26 Jurisdicciones,EN,Aporte ENOS (F2-F1),-0.00529,-0.00714,-0.01428,0.00034
2,Panel 26 Jurisdicciones,EN,Aporte Total (F2-F0),-0.00334,-0.00516,-0.01508,0.00336
3,Panel 26 Jurisdicciones,RF,Aporte Territorial (F1-F0),0.00501,0.00525,0.00039,0.01013
4,Panel 26 Jurisdicciones,RF,Aporte ENOS (F2-F1),0.00302,-0.00067,-0.01344,0.01218
5,Panel 26 Jurisdicciones,RF,Aporte Total (F2-F0),0.00804,0.00458,-0.01167,0.01993
6,Panel 26 Jurisdicciones,XGB,Aporte Territorial (F1-F0),-0.00491,-0.00521,-0.01151,0.00065
7,Panel 26 Jurisdicciones,XGB,Aporte ENOS (F2-F1),-0.01785,-0.02118,-0.04621,0.00299
8,Panel 26 Jurisdicciones,XGB,Aporte Total (F2-F0),-0.02276,-0.02640,-0.05713,0.00273
9,Panel 26 Jurisdicciones,LSTM,Aporte Territorial (F1-F0),-0.00441,-0.00581,-0.02564,0.01304



### MATRIZ DE VERIFICACIÓN DE HIPÓTESIS


,Hipotesis,Delta_Brier,IC95,Mejora_Puntual,Veredicto
0,H1 (Total),-0.005451,"[-0.0289, 0.0100]",True,EVIDENCIA INSUFICIENTE
1,HE1 (Territorial),-0.000590,"[-0.0093, 0.0068]",True,EVIDENCIA INSUFICIENTE
2,HE2 (ENOS),-0.004861,"[-0.0211, 0.0063]",True,EVIDENCIA INSUFICIENTE
3,HE3 (Robustez),-0.004909,"Alg. Neg: 3/4 | IC Sin Lima: [-0.0275, 0.0099]",NaN,CUMPLE


[2026-10-30T03:28:49.126915+00:00] [INFO] [FLAG] FLAG creado: Celda10_Contrastes_Bootstrap.OK.flag
[2026-10-30T03:28:49.127056+00:00] [INFO] [Celda10_Contrastes_Bootstrap] FIN calcular_contrastes — 3.73s
[2026-10-30T03:28:56.062482+00:00] [INFO] [HASH] HASH registrado: OE3_OE4_Contrastes = 92f0c89fd54764b0...
[2026-10-30T03:28:56.065042+00:00] [INFO] [FLAG] FLAG creado: Celda10_Evaluacion_Prueba.OK.flag

PASO 10 COMPLETADO — PRUEBA EVALUADA (OE3 y OE4)
Predicciones : /content/resultados_tesis/20261030T002900_112193Z/Predicciones_Prueba_12_Modelos.csv
Contrastes   : /content/resultados_tesis/20261030T002900_112193Z/OE3_OE4_Contrastes.csv
Hipótesis    :


,Hipotesis,Delta_Brier,IC95,Mejora_Puntual,Veredicto
0,H1 (Total),-0.005451,"[-0.0289, 0.0100]",True,EVIDENCIA INSUFICIENTE
1,HE1 (Territorial),-0.000590,"[-0.0093, 0.0068]",True,EVIDENCIA INSUFICIENTE
2,HE2 (ENOS),-0.004861,"[-0.0211, 0.0063]",True,EVIDENCIA INSUFICIENTE
3,HE3 (Robustez),-0.004909,"Alg. Neg: 3/4 | IC Sin Lima: [-0.0275, 0.0099]",NaN,CUMPLE



Auditoría en: /content/resultados_tesis/20261030T002900_112193Z/00_AUDITORIA/20261030T002900_112193Z


## Paso 11 — Tablas y gráficos de resultados

Genera las **tablas y figuras de resultados** que contrastan con el documento Word de la tesis:

**Tablas:**
- **Tabla 36:** Brier Score por configuración y bloque (validación vs prueba).
- **Tabla 37:** Brier Score macro por año de prueba (2023, 2024, 2025).
- **Tabla 38:** Variables con mayor importancia en las configuraciones.
- **Tabla 39:** Máximos anuales de frecuencia mensual.
- **Tabla 40:** Extremos de precipitación mensual por año (2021-2025).
- **Tabla 41:** Perfil mensual de precipitación y temperatura entre 2005 y 2025.
- **Tabla 42:** Métricas de las 12 configuraciones en la prueba final.
- **Tabla 43:** Δ Brier por familia y comparación.
- **Tabla 44:** Sensibilidad sin Lima Metropolitana.
- **Tabla 45:** Evaluación de hipótesis (H1, HE1, HE2, HE3).

**Figuras:**
- **Fig 41:** Brier por configuración en prueba.
- **Fig 42:** Δ Brier por familia y comparación.
- **Fig 43:** Contrastes multimodelo e IC95% bootstrap.
- **Fig 44:** Sensibilidad sin Lima Metropolitana.
- **Fig 45**: Recaudación real anual 2005-2025 (incluye prueba).
- **Fig 46:** Frecuencia anual de caídas 2007-2025.
- **Fig 47**: Comportamiento mensual de caídas 2021-2025.
- **Fig 48**: Frecuencia de caídas por jurisdicción.
- **Fig 49:** ONI e ICEN anuales 2005-2025.
- **Fig 50:** Precipitación Mensual Media (2021-2025).
- **Fig 51:** Temperatura Mensual Media (2021-2025).

Todas las figuras llevan etiquetas de datos y se guardan en PNG 300 dpi. Se registra el flag `Celda11_Tablas_Resultados.OK.flag`.

In [14]:
# ==============================================================================
# 11: TABLAS Y GRÁFICOS DE RESULTADOS
# ==============================================================================
# Genera tablas (Tabla 36, 37, 43, 44, 45) y figuras (Fig41–Fig58) que
# contrastan con el documento Word de resultados.
# Todas las figuras llevan etiquetas de datos en cada punto/barra.
# Se guardan en PNG 300dpi + CSV, y se registran hashes en 00_AUDITORIA.

# Ocultar unicamente el aviso de .obsolescencia generado por .Jupyter.
import warnings

warnings.filterwarnings(
    action="ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"^jupyter_client\.session$",
)

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch
from pathlib import Path

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 110, 'axes.axisbelow': True, 'savefig.facecolor': 'white',
})

FIG_DIR = OUTPUT_DIR / "Graficos_Resultados"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLORES_ALG = {'EN': '#0072B2', 'RF': '#D55E00', 'XGB': '#009E73', 'LSTM': '#CC79A7'}
COLORES_F = {'F0': '#4C72B0', 'F1': '#DD8452', 'F2': '#55A868'}
MESES_NOMBRE = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']


def guardar(fig, nombre):
    ruta = FIG_DIR / f"{nombre}.png"
    fig.savefig(ruta, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  ✓ {ruta.name}")
    return ruta


# ==============================================================================
# FUNCIÓN PRINCIPAL (decorada)
# ==============================================================================
@auditar("Celda11_Tablas_Resultados")
def generar_tablas_y_figuras(df_panel, panel_base, df_preds,
                             df_res_oe2, df_contrastes, df_hipotesis,
                             df_oni, df_icen):
    log("Iniciando tablas y figuras de resultados (OE2–OE4)",
        celda="Celda11_Tablas_Resultados")

    if sha256_file(OUTPUT_DIR / 'Predicciones_Prueba_12_Modelos.csv') != PREDICCIONES_CONGELADAS_HASH:
        raise ValueError('Probabilidades cambiaron tras congelación')
    algos = ['EN', 'RF', 'XGB', 'LSTM']
    fams = ['F0', 'F1', 'F2']

    # -------------------------------------------------------------------------
    # TABLA 35: Hiperparámetros finales seleccionados
    # -------------------------------------------------------------------------
    print("\n=== TABLA 35: Hiperparámetros finales seleccionados ===")
    t35_rows = []
    # df_res_oe2 tiene las columnas: Familia_Algoritmica, Familia_Informacion, Brier_Val, candidate_id
    for _, row in df_res_oe2.iterrows():
        alg = row['Familia_Algoritmica']
        fam = row['Familia_Informacion']
        cand = row['candidate_id']
        bs_val = row['Brier_Val']

        # Obtener hiperparámetros del diccionario
        hp_dict = hp_optimos.get(f'{alg}_{fam}', {})
        # Formatear hiperparámetros como string legible
        if alg == 'EN':
            hp_str = f"C={hp_dict.get('C', '')}; l1_ratio={hp_dict.get('l1_ratio', '')}"
        elif alg == 'RF':
            hp_str = f"n_estimators={hp_dict.get('n_estimators', '')}; max_depth={hp_dict.get('max_depth', '')}; min_samples_leaf={hp_dict.get('min_samples_leaf', '')}; max_features={hp_dict.get('max_features', '')}"
        elif alg == 'XGB':
            hp_str = f"n_estimators={hp_dict.get('n_estimators', '')}; max_depth={hp_dict.get('max_depth', '')}; learning_rate={hp_dict.get('learning_rate', '')}; subsample={hp_dict.get('subsample', '')}"
        elif alg == 'LSTM':
            hp_str = f"units={hp_dict.get('units', '')}; dense={hp_dict.get('dense', '')}; dropout={hp_dict.get('dropout', '')}; lr={hp_dict.get('learning_rate', '')}; epoch={hp_dict.get('best_epoch', '')}"
        else:
            hp_str = str(hp_dict)

        t35_rows.append({
            'Familia': alg,
            'Conjunto': fam,
            'Candidato': cand,
            'Hiperparámetros finales': hp_str,
            'Brier validación': bs_val
        })

    tabla35 = pd.DataFrame(t35_rows)
    tabla35.to_csv(FIG_DIR / "Tabla35_Hiperparametros_Finales.csv", index=False)
    display(tabla35.round(6))


    # =========================================================================
    # TABLA 36: BRIER SCORE MACRO POR CONFIGURACIÓN Y BLOQUE
    # =========================================================================
    print("\n=== TABLA 36: Brier Score por Configuración y Bloque ===")

    def brier_por_config(df_preds, col_p):
        return (((df_preds[col_p] - df_preds['Y_caida']) ** 2)
                .groupby(df_preds['jurisdiccion']).mean().mean())

    resultados_tabla36 = []
    for alg in algos:
        for fam in fams:
            bs_val = df_res_oe2[
                (df_res_oe2['Familia_Algoritmica'] == alg) &
                (df_res_oe2['Familia_Informacion'] == fam)
            ]['Brier_Val'].values[0]
            bs_prueba = brier_por_config(df_preds, f'prob_{alg}_{fam}')
            resultados_tabla36.append({
                'Algoritmo': alg, 'Familia_Informacion': fam,
                'Brier_Entrenamiento': float(df_res_oe2.loc[(df_res_oe2.Familia_Algoritmica==alg) & (df_res_oe2.Familia_Informacion==fam),'Brier_Train'].iloc[0]),
                'Modelo_Seleccion': f'{alg}_{fam}_seleccion', 'Modelo_Prueba': f'{alg}_{fam}_reajuste',
                'Brier_Validacion': bs_val, 'Brier_Prueba': bs_prueba,
                'Diferencia_Prueba_Menos_Validacion': bs_prueba - bs_val,
            })

    tabla36 = (pd.DataFrame(resultados_tabla36)
               .sort_values(['Algoritmo', 'Familia_Informacion'])
               .reset_index(drop=True))
    tabla36.to_csv(FIG_DIR / "Tabla36_Brier_Validacion_Prueba.csv", index=False)
    display(tabla36.round(6))

    # =========================================================================
    # FIGURA 41: BRIER SCORE MACRO EN PRUEBA (12 CONFIGURACIONES)
    # =========================================================================
    print("\n=== FIGURA 41: Brier Score Macro en Prueba ===")
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(algos))
    width = 0.26
    for i, fam in enumerate(fams):
        vals = [tabla36[
            (tabla36['Algoritmo'] == a) &
            (tabla36['Familia_Informacion'] == fam)
        ]['Brier_Prueba'].values[0] for a in algos]
        bars = ax.bar(x + (i - 1) * width, vals, width,
                      label=fam, color=COLORES_F[fam], edgecolor='white')
        ax.bar_label(bars, fmt='%.4f', padding=3,
                     fontsize=8.5, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(algos)
    ax.set_ylabel('Brier Score Macro')
    ax.set_xlabel('Familia Algorítmica')
    ax.set_title('OE2: Brier Score Macro en Prueba Final (2023-2025)',
                 loc='left', pad=12)
    ax.legend(title='Conjunto de Información', loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig41_Brier_12_Configuraciones")

    # =========================================================================
    # FIGURA 42: DELTA BRIER POR FAMILIA Y COMPARACIÓN
    # =========================================================================
    print("\n=== FIGURA 42: Delta Brier por Familia y Comparación ===")
    fig, ax = plt.subplots(figsize=(12, 6))
    contrastes = ['Aporte Territorial (F1-F0)',
                  'Aporte ENOS (F2-F1)',
                  'Aporte Total (F2-F0)']
    nombres_cortos = ['F1-F0', 'F2-F1', 'F2-F0']
    colores_c = ['#4C72B0', '#DD8452', '#55A868']

    x = np.arange(len(algos))
    width = 0.26
    for i, (contraste, nombre) in enumerate(zip(contrastes, nombres_cortos)):
        vals = []
        for a in algos:
            row = df_contrastes[
                (df_contrastes['Familia_Algoritmica'] == a) &
                (df_contrastes['Contraste'] == contraste) &
                (df_contrastes['Escenario'] == 'Panel 26 Jurisdicciones')
            ]
            vals.append(row['Delta_Brier'].values[0] if len(row) > 0 else np.nan)
        bars = ax.bar(x + (i - 1) * width, vals, width,
                      label=nombre, color=colores_c[i], edgecolor='white')
        ax.bar_label(bars, fmt='%+.5f', padding=3,
                     fontsize=8.5, fontweight='bold')

    ax.axhline(0, color='black', linewidth=1.2)
    ax.set_xticks(x); ax.set_xticklabels(algos)
    ax.set_ylabel('Δ Brier Score')
    ax.set_xlabel('Familia Algorítmica')
    ax.set_title('OE3: Delta Brier Score por Familia y Comparación',
                 loc='left', pad=12)
    ax.legend(title='Comparación', loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig42_Delta_Brier_Familia")

    # =========================================================================
    # FIGURA 43: CONTRASTES MULTIMODELO E IC95% BOOTSTRAP
    # =========================================================================
    print("\n=== FIGURA 43: Contrastes Multimodelo e IC95% ===")
    fig, ax = plt.subplots(figsize=(11, 6))
    multi = df_contrastes[
        (df_contrastes['Familia_Algoritmica'] == 'RESUMEN MULTIMODELO') &
        (df_contrastes['Escenario'] == 'Panel 26 Jurisdicciones')
    ].copy().sort_values('Contraste', ascending=False)

    y_pos = np.arange(len(multi))
    for i, (_, row) in enumerate(multi.iterrows()):
        color = '#009E73' if row['Delta_Brier'] < 0 else '#D55E00'
        ax.plot([row['IC95_Inferior'], row['IC95_Superior']],
                [i, i], color=color, linewidth=2.5, alpha=0.6)
        ax.scatter(row['Delta_Brier'], i, color=color, s=120,
                   zorder=5, edgecolor='white', linewidth=1.5)
        ax.annotate(
            f"{row['Delta_Brier']:+.5f}\n"
            f"[{row['IC95_Inferior']:+.5f}, {row['IC95_Superior']:+.5f}]",
            (row['Delta_Brier'], i), xytext=(0, 18),
            textcoords='offset points', ha='center',
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white',
                      ec=color, alpha=0.9),
        )

    ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([c.replace(' (', '\n(') for c in multi['Contraste']])
    ax.set_xlabel('Delta Brier Score Macro')
    ax.set_title('OE3: Contrastes Multimodelo e IC95% Bootstrap',
                 loc='left', pad=12)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.margins(y=0.15)
    plt.tight_layout()
    guardar(fig, "Fig43_Contrastes_Multimodelo_IC95")

    # =========================================================================
    # FIGURA 44: SENSIBILIDAD SIN LIMA METROPOLITANA
    # =========================================================================
    print("\n=== FIGURA 44: Sensibilidad sin Lima Metropolitana ===")
    fig, ax = plt.subplots(figsize=(12, 6))
    algos_plot = ['EN', 'RF', 'XGB', 'LSTM', 'RESUMEN MULTIMODELO']
    x = np.arange(len(algos_plot))
    width = 0.35

    vals_orig, vals_sin = [], []
    for a in algos_plot:
        row_o = df_contrastes[
            (df_contrastes['Familia_Algoritmica'] == a) &
            (df_contrastes['Contraste'] == 'Aporte Total (F2-F0)') &
            (df_contrastes['Escenario'] == 'Panel 26 Jurisdicciones')
        ]
        row_s = df_contrastes[
            (df_contrastes['Familia_Algoritmica'] == a) &
            (df_contrastes['Contraste'] == 'Aporte Total (F2-F0)') &
            (df_contrastes['Escenario'] == 'Sensibilidad 25 Jurisdicciones')
        ]
        vals_orig.append(row_o['Delta_Brier'].values[0] if len(row_o) > 0 else np.nan)
        vals_sin.append(row_s['Delta_Brier'].values[0] if len(row_s) > 0 else np.nan)

    bars1 = ax.bar(x - width/2, vals_orig, width,
                   label='Original', color='#4C72B0', edgecolor='white')
    bars2 = ax.bar(x + width/2, vals_sin, width,
                   label='Sin Lima Metropolitana',
                   color='#D55E00', edgecolor='white')
    ax.bar_label(bars1, fmt='%+.5f', padding=3, fontsize=8, fontweight='bold')
    ax.bar_label(bars2, fmt='%+.5f', padding=3, fontsize=8, fontweight='bold')

    ax.axhline(0, color='black', linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [a.replace('RESUMEN MULTIMODELO', 'MULTIMODELO') for a in algos_plot],
        rotation=15,
    )
    ax.set_ylabel('Δ Brier Score Macro F2-F0')
    ax.set_xlabel('Familia Algorítmica')
    ax.set_title('OE4: Sensibilidad del Aporte Climático Total al Excluir '
                 'Lima Metropolitana', loc='left', pad=12)
    ax.legend(loc='upper left')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig44_Sensibilidad_Sin_Lima")

    # =========================================================================
    # FIGURA 45: RECAUDACIÓN REAL ANUAL (2005–2025, INCLUYE PRUEBA)
    # =========================================================================
    print("\n=== FIGURA 45: Recaudación Real Anual (2005-2025) ===")
    panel_base_graf = panel_base.copy()
    panel_base_graf['anio'] = panel_base_graf['fecha'].dt.year
    recaud_anual = (panel_base_graf.groupby('anio', as_index=False)
                    ['recaud_real'].sum())
    recaud_anual['millones'] = recaud_anual['recaud_real'] / 1e6

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(recaud_anual['anio'], recaud_anual['millones'],
            color='#24547A', lw=2.5, zorder=3)
    ax.scatter(recaud_anual['anio'], recaud_anual['millones'],
               c=recaud_anual['millones'], cmap='viridis', s=80,
               edgecolor='white', linewidth=1.2, zorder=4)
    ax.fill_between(recaud_anual['anio'], recaud_anual['millones'], 0,
                    color='#25A7B8', alpha=0.1)

    for _, row in recaud_anual.iterrows():
        ax.annotate(f"{row['millones']:.1f}",
                    (row['anio'], row['millones']),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=8, fontweight='bold',
                    color='#24547A')

    max_row = recaud_anual.loc[recaud_anual['millones'].idxmax()]
    min_row = recaud_anual.loc[recaud_anual['millones'].idxmin()]
    ax.annotate(
        f"Máximo · {int(max_row['anio'])}\n"
        f"{max_row['millones']:.2f} millones de S/",
        (max_row['anio'], max_row['millones']),
        xytext=(0, 22), textcoords='offset points',
        ha='center', fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#24547A'),
        arrowprops=dict(arrowstyle='->', color='#24547A'),
    )
    ax.annotate(
        f"Mínimo · {int(min_row['anio'])}\n"
        f"{min_row['millones']:.2f} millones de S/",
        (min_row['anio'], min_row['millones']),
        xytext=(0, -22), textcoords='offset points',
        ha='center', fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#D55E00'),
        arrowprops=dict(arrowstyle='->', color='#D55E00'),
    )

    ax.set_xlabel('Año')
    ax.set_ylabel('Millones de S/ constantes (dic-2021)')
    ax.set_title('Evolución Anual de la Recaudación Real Agregada, 2005-2025',
                 loc='left', pad=12)
    ax.set_xticks(recaud_anual['anio'])
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig45_Recaudacion_Anual")

    # =========================================================================
    # FIGURA 46: FRECUENCIA ANUAL DE CAÍDAS (2007–2025, INCLUYE PRUEBA)
    # =========================================================================
    print("\n=== FIGURA 46: Frecuencia Anual de Caídas ===")
    caidas_anual = df_panel.groupby('anio_objetivo', as_index=False).agg(
        n_caidas=('Y_caida', 'sum'), n_total=('Y_caida', 'size'))
    caidas_anual['porcentaje'] = (
        100 * caidas_anual['n_caidas'] / caidas_anual['n_total']
    )

    fig, ax = plt.subplots(figsize=(13, 6))
    colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(caidas_anual)))
    bars = ax.bar(caidas_anual['anio_objetivo'].astype(str),
                  caidas_anual['porcentaje'], color=colors)
    ax.bar_label(bars, fmt='%.2f%%', padding=3,
                 fontsize=9, fontweight='bold')

    max_row = caidas_anual.loc[caidas_anual['porcentaje'].idxmax()]
    min_row = caidas_anual.loc[caidas_anual['porcentaje'].idxmin()]
    ax.annotate(
        f"Máximo · {int(max_row['anio_objetivo'])}\n{max_row['porcentaje']:.2f}%",
        (str(max_row['anio_objetivo']), max_row['porcentaje']),
        xytext=(0, 20), textcoords='offset points',
        ha='center', fontsize=9, fontweight='bold', color='#D55E00',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#D55E00'),
    )
    ax.annotate(
        f"Mínimo · {int(min_row['anio_objetivo'])}\n{min_row['porcentaje']:.2f}%",
        (str(min_row['anio_objetivo']), min_row['porcentaje']),
        xytext=(0, -25), textcoords='offset points',
        ha='center', fontsize=9, fontweight='bold', color='#009E73',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#009E73'),
    )

    ax.set_ylim(0, caidas_anual['porcentaje'].max() * 1.3)
    ax.set_ylabel('Jurisdicciones con caída (%)')
    ax.set_xlabel('Año objetivo')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(
        'Frecuencia Anual de Caídas Interanuales Reales, 2007-2025',
        loc='left', pad=12,
    )
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig46_Caidas_Anuales")

    # =========================================================================
    # FIGURA 47: COMPORTAMIENTO MENSUAL DE CAÍDAS 2021–2025
    # =========================================================================
    print("\n=== FIGURA 47: Comportamiento Mensual 2021-2025 ===")
    mensual = df_panel.groupby(
        ['anio_objetivo', 'mes_objetivo'], as_index=False
    ).agg(n_caidas=('Y_caida', 'sum'), n_total=('Y_caida', 'size'))
    mensual['porcentaje'] = 100 * mensual['n_caidas'] / mensual['n_total']

    fig, ax = plt.subplots(figsize=(15, 7.5))
    anios_plot = [2021, 2022, 2023, 2024, 2025]
    marcadores = ['o', 's', 'D', '^', 'v']
    paleta = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#6F51A8']

    for i, anio in enumerate(anios_plot):
        sub = mensual[mensual['anio_objetivo'] == anio].sort_values(
            'mes_objetivo')
        if sub.empty:
            continue
        ax.plot(sub['mes_objetivo'], sub['porcentaje'],
                marker=marcadores[i], ms=7, lw=2,
                label=str(anio), color=paleta[i])
        for _, row in sub.iterrows():
            ax.annotate(f"{row['porcentaje']:.1f}",
                        (row['mes_objetivo'], row['porcentaje']),
                        xytext=(0, 8), textcoords='offset points',
                        ha='center', fontsize=7.5,
                        color=paleta[i], fontweight='bold')

    ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES_NOMBRE)
    ax.set_xlim(0.5, 12.5); ax.set_ylim(0, 105)
    ax.set_ylabel('Jurisdicciones con caída (%)')
    ax.set_xlabel('Mes objetivo')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title('Comportamiento Mensual de las Caídas, 2021-2025',
                 loc='left', pad=12)
    ax.legend(title='Año', loc='upper left', bbox_to_anchor=(1.01, 1))
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig47_Caidas_Mensuales_2021_2025")

    # =========================================================================
    # FIGURA 48: FRECUENCIA DE CAÍDAS POR JURISDICCIÓN (2007–2025)
    # =========================================================================
    print("\n=== FIGURA 48: Frecuencia de Caídas por Jurisdicción ===")
    jurisdiccion_rank = df_panel.groupby(
        'jurisdiccion', as_index=False
    ).agg(n_caidas=('Y_caida', 'sum'), n_total=('Y_caida', 'size'))
    jurisdiccion_rank['porcentaje'] = (
        100 * jurisdiccion_rank['n_caidas'] / jurisdiccion_rank['n_total']
    )
    jurisdiccion_rank = jurisdiccion_rank.sort_values(
        'porcentaje', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 10))
    colors = plt.cm.viridis(
        np.linspace(0.15, 0.85, len(jurisdiccion_rank)))
    bars = ax.barh(jurisdiccion_rank['jurisdiccion'],
                   jurisdiccion_rank['porcentaje'],
                   color=colors, height=0.75)
    ax.bar_label(bars, fmt='%.2f%%', padding=6,
                 fontsize=9, fontweight='bold')

    ax.set_xlim(0, jurisdiccion_rank['porcentaje'].max() * 1.2)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xlabel('Meses con caída (%)')
    ax.set_ylabel('')
    ax.set_title(
        'Frecuencia de Caídas por Jurisdicción Fiscal, 2007-2025',
        loc='left', pad=12,
    )
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig48_Caidas_Jurisdiccion")

    # =========================================================================
    # FIGURA 49: ONI E ICEN PROMEDIO ANUAL (2005–2025)
    # =========================================================================
    print("\n=== FIGURA 49: ONI e ICEN Anual ===")

    def enso_anual(df, col):
        d = df.copy()
        d['anio'] = d['fecha_central'].dt.year
        return d.groupby('anio', as_index=False)[col].mean()

    enso_oni = enso_anual(df_oni, 'oni')
    enso_icen = enso_anual(df_icen, 'icen')
    enso_anual_df = (enso_oni.merge(enso_icen, on='anio')
                     .query("2005 <= anio <= 2025"))

    fig, ax = plt.subplots(figsize=(14, 6.5))
    ax.plot(enso_anual_df['anio'], enso_anual_df['oni'],
            marker='o', lw=2.3, ms=6, color='#0072B2', label='ONI')
    ax.plot(enso_anual_df['anio'], enso_anual_df['icen'],
            marker='D', lw=2.3, ms=6, color='#D55E00', label='ICEN')

    for _, row in enso_anual_df.iterrows():
        ax.annotate(f"{row['oni']:.2f}", (row['anio'], row['oni']),
                    xytext=(0, 8), textcoords='offset points',
                    ha='center', fontsize=7, color='#0072B2')
        ax.annotate(f"{row['icen']:.2f}", (row['anio'], row['icen']),
                    xytext=(0, -12), textcoords='offset points',
                    ha='center', fontsize=7, color='#D55E00')

    for col, color in [('oni', '#0072B2'), ('icen', '#D55E00')]:
        max_row = enso_anual_df.loc[enso_anual_df[col].idxmax()]
        min_row = enso_anual_df.loc[enso_anual_df[col].idxmin()]
        ax.annotate(
            f"Máx {col.upper()}: {max_row[col]:.2f}°C ({int(max_row['anio'])})",
            (max_row['anio'], max_row[col]),
            xytext=(0, 22), textcoords='offset points',
            ha='center', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color),
        )
        ax.annotate(
            f"Mín {col.upper()}: {min_row[col]:.2f}°C ({int(min_row['anio'])})",
            (min_row['anio'], min_row[col]),
            xytext=(0, -25), textcoords='offset points',
            ha='center', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=color),
        )

    ax.axhline(0, color='gray', lw=0.9, linestyle='--')
    ax.set_xticks(enso_anual_df['anio'])
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Anomalía térmica oceánica (°C)')
    ax.set_xlabel('Año del mes central')
    ax.set_title(
        'Evolución Descriptiva de ONI e ICEN Medios Anuales, 2005-2025',
        loc='left', pad=12,
    )
    ax.legend(loc='upper left')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    guardar(fig, "Fig49_ONI_ICEN_Anual")


    # =========================================================================
    # FIGURA 50: PRECIPITACIÓN MENSUAL MEDIA (2021-2025)
    # =========================================================================
    print("\n=== FIGURA 50: Precipitación Mensual Media 2021-2025 ===")

    df_precip_plot = df_panel[df_panel['anio_objetivo'].between(2021, 2025)].copy()
    df_precip_plot['fecha_fuente'] = df_precip_plot['objetivo_t1'] - pd.DateOffset(months=4)
    df_precip_plot['anio_fuente'] = df_precip_plot['fecha_fuente'].dt.year
    df_precip_plot['mes_fuente'] = df_precip_plot['fecha_fuente'].dt.month

    precip_mensual = df_precip_plot.groupby(['anio_fuente', 'mes_fuente'])['f1_precip'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(14, 7))

    colores_anios = {
        2020: '#8c564b', # Marrón (para el año fuente 2020)
        2021: '#1f77b4', # Azul
        2022: '#ff7f0e', # Naranja
        2023: '#2ca02c', # Verde
        2024: '#e377c2', # Rosa
        2025: '#9467bd'  # Morado
    }

    for anio in sorted(precip_mensual['anio_fuente'].unique()):
        sub = precip_mensual[precip_mensual['anio_fuente'] == anio].sort_values('mes_fuente')

        ax.plot(sub['mes_fuente'], sub['f1_precip'],
                marker='o', ms=5, lw=1.5,
                label=str(anio), color=colores_anios[anio])

        for _, row in sub.iterrows():
            ax.annotate(f"{row['f1_precip']:.1f}",
                        (row['mes_fuente'], row['f1_precip']),
                        xytext=(0, 8), textcoords='offset points',
                        ha='center', fontsize=7.5,
                        color=colores_anios[anio], fontweight='bold')

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MESES_NOMBRE)
    ax.set_xlim(0.5, 12.5)

    max_val = precip_mensual['f1_precip'].max()
    ax.set_ylim(0, max_val * 1.15)

    ax.set_ylabel('Precipitación mensual (mm)')
    ax.set_xlabel('Mes climático fuente (t−4)')

    ax.set_title('Precipitación mensual media de las jurisdicciones entre 2021 y 2025',
                 loc='left', pad=12)

    ax.legend(title='Año del mes fuente', loc='upper right', framealpha=0.9)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    nota_fig(fig, 'Debido al horizonte t+1 y a la regla ERA5-Land m+3, el valor mostrado para un mes objetivo corresponde '
                  'al mes climático fuente situado cuatro meses antes.')

    plt.tight_layout()
    guardar(fig, "Fig50_Precipitacion_Mensual_2021_2025")

    # =========================================================================
    # FIGURA 51: TEMPERATURA MENSUAL MEDIA (2021-2025)
    # =========================================================================
    print("\n=== FIGURA 51: Temperatura Mensual Media 2021-2025 ===")

    df_temp_plot = df_panel[df_panel['anio_objetivo'].between(2021, 2025)].copy()
    df_temp_plot['fecha_fuente'] = df_temp_plot['objetivo_t1'] - pd.DateOffset(months=4)
    df_temp_plot['anio_fuente'] = df_temp_plot['fecha_fuente'].dt.year
    df_temp_plot['mes_fuente'] = df_temp_plot['fecha_fuente'].dt.month

    temp_mensual = df_temp_plot.groupby(['anio_fuente', 'mes_fuente'])['f1_temp'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(14, 7))

    # CORRECCIÓN: Agregamos el año 2020 al diccionario
    colores_anios = {
        2020: '#8c564b', # Marrón (para el año fuente 2020)
        2021: '#1f77b4', # Azul
        2022: '#ff7f0e', # Naranja
        2023: '#2ca02c', # Verde
        2024: '#e377c2', # Rosa
        2025: '#9467bd'  # Morado
    }

    for anio in sorted(temp_mensual['anio_fuente'].unique()):
        sub = temp_mensual[temp_mensual['anio_fuente'] == anio].sort_values('mes_fuente')

        ax.plot(sub['mes_fuente'], sub['f1_temp'],
                marker='o', ms=5, lw=1.5,
                label=str(anio), color=colores_anios[anio])

        for _, row in sub.iterrows():
            ax.annotate(f"{row['f1_temp']:.1f}",
                        (row['mes_fuente'], row['f1_temp']),
                        xytext=(0, 8), textcoords='offset points',
                        ha='center', fontsize=7.5,
                        color=colores_anios[anio], fontweight='bold')

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MESES_NOMBRE)
    ax.set_xlim(0.5, 12.5)

    min_val = temp_mensual['f1_temp'].min()
    max_val = temp_mensual['f1_temp'].max()
    rango = max_val - min_val
    ax.set_ylim(min_val - rango * 0.15, max_val + rango * 0.15)

    ax.set_ylabel('Temperatura (°C)')
    ax.set_xlabel('Mes climático fuente (t−4)')

    ax.set_title('Temperatura mensual media de las jurisdicciones entre 2021 y 2025',
                 loc='left', pad=12)

    ax.legend(title='Año del mes fuente', loc='upper right', framealpha=0.9)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    nota_fig(fig, 'Debido al horizonte t+1 y a la regla ERA5-Land m+3, el valor mostrado para un mes objetivo corresponde '
                  'al mes climático fuente situado cuatro meses antes.')

    plt.tight_layout()
    guardar(fig, "Fig51_Temperatura_Mensual_2021_2025")

    # =========================================================================
    # TABLA 37: BRIER SCORE MACRO POR AÑO DE PRUEBA
    # =========================================================================
    print("\n=== TABLA 37: Brier Score por Año de Prueba ===")
    df_preds_anual = df_preds.copy()
    df_preds_anual['anio'] = pd.to_datetime(
        df_preds_anual['objetivo_t1']).dt.year

    tabla37 = []
    for alg in algos:
        for fam in fams:
            col_p = f'prob_{alg}_{fam}'
            for anio in [2023, 2024, 2025]:
                sub = df_preds_anual[df_preds_anual['anio'] == anio]
                brier = (((sub[col_p] - sub['Y_caida']) ** 2)
                         .groupby(sub['jurisdiccion']).mean().mean())
                tabla37.append({
                    'Algoritmo': alg, 'Familia_Informacion': fam,
                    'Anio': anio, 'Brier_Macro': brier,
                })

    tabla37 = pd.DataFrame(tabla37)
    tabla37_pivot = tabla37.pivot_table(
        index=['Algoritmo', 'Familia_Informacion'],
        columns='Anio', values='Brier_Macro',
    )
    tabla37_pivot.to_csv(FIG_DIR / "Tabla37_Brier_Por_Anio.csv")
    display(tabla37_pivot.round(5))

    # -------------------------------------------------------------------------
    # TABLA 38: Variables con mayor importancia en las configuraciones
    # -------------------------------------------------------------------------
    print("\n=== TABLA 38: Variables con mayor importancia en las configuraciones ===")
    import joblib
    t38_rows = []
    modelos_dir = OUTPUT_DIR / 'Modelos_Reajuste'
    # Iteramos sobre los modelos de EN y RF para extraer importancias
    for alg in ['EN', 'RF', 'XGB']:
        for fam in ['F0', 'F1', 'F2']:
            key = f'{alg}_{fam}'
            path_mod = modelos_dir / f'{key}.joblib'
            if path_mod.exists():
                modelo = joblib.load(path_mod)
                columnas = PREPROCESADORES[key]['columnas']

                if hasattr(modelo, 'coef_'):
                    importancias = np.abs(modelo.coef_[0])
                elif hasattr(modelo, 'feature_importances_'):
                    importancias = modelo.feature_importances_
                else:
                    continue

                # Obtener las 3 variables con mayor magnitud
                top3_idx = np.argsort(importancias)[-3:][::-1]
                for idx in top3_idx:
                    t38_rows.append({
                        'Modelo': alg,
                        'Familia': fam,
                        'Tres variables con mayor magnitud o importancia': columnas[idx],
                        'Valor': importancias[idx]
                    })

    tabla38 = pd.DataFrame(t38_rows)
    tabla38.to_csv(FIG_DIR / "Tabla38_Importancia_Variables.csv", index=False)
    display(tabla38.round(6))

    # -------------------------------------------------------------------------
    # TABLA 39: Máximos anuales de frecuencia mensual
    # -------------------------------------------------------------------------
    print("\n=== TABLA 39: Máximos anuales de frecuencia mensual ===")
    # Usamos df_panel (que incluye prueba) para calcular la frecuencia mensual
    freq_mensual = df_panel.groupby(['anio_objetivo', 'mes_objetivo']).agg(
        n_caidas=('Y_caida', 'sum'),
        n_total=('Y_caida', 'size')
    ).reset_index()
    freq_mensual['porcentaje'] = 100 * freq_mensual['n_caidas'] / freq_mensual['n_total']

    t39_rows = []
    for anio, grupo in freq_mensual.groupby('anio_objetivo'):
        if anio < 2021: continue # La imagen muestra solo 2021 en adelante
        max_pct = grupo['porcentaje'].max()
        # Encontrar todos los meses que alcanzan el máximo
        meses_max = grupo[grupo['porcentaje'] == max_pct]['mes_objetivo'].tolist()
        n_jurisdicciones = int(grupo[grupo['porcentaje'] == max_pct]['n_caidas'].iloc[0])

        # Formatear meses
        nombres_meses = [MESES_NOMBRE[m-1] for m in meses_max]
        meses_str = ", ".join(nombres_meses)

        t39_rows.append({
            'Año': anio,
            'Meses del máximo': meses_str,
            'Jurisdicciones': f"{n_jurisdicciones} de 26",
            'Frecuencia': f"{max_pct:.2f} %"
        })

    tabla39 = pd.DataFrame(t39_rows)
    tabla39.to_csv(FIG_DIR / "Tabla39_Maximos_Anuales_Frecuencia.csv", index=False)
    display(tabla39)

    # -------------------------------------------------------------------------
    # TABLA 40: Extremos de precipitación mensual por año (2021-2025)
    # -------------------------------------------------------------------------
    print("\n=== TABLA 40: Extremos de precipitación mensual por año (2021-2025) ===")

    # Aplicamos el desfase de 4 meses para obtener la fecha fuente
    df_precip_ext = df_panel[df_panel['anio_objetivo'].between(2021, 2025)].copy()
    df_precip_ext['fecha_fuente'] = df_precip_ext['objetivo_t1'] - pd.DateOffset(months=4)
    df_precip_ext['anio_fuente'] = df_precip_ext['fecha_fuente'].dt.year
    df_precip_ext['mes_fuente'] = df_precip_ext['fecha_fuente'].dt.month

    # Agrupamos por año fuente y mes fuente
    precip_anual = df_precip_ext.groupby(['anio_fuente', 'mes_fuente'])['f1_precip'].mean().reset_index()

    t40_rows = []
    for anio, grupo in precip_anual.groupby('anio_fuente'):
        max_row = grupo.loc[grupo['f1_precip'].idxmax()]
        min_row = grupo.loc[grupo['f1_precip'].idxmin()]
        t40_rows.append({
            'Año (fuente)': anio,
            'Mes máximo': MESES_NOMBRE[int(max_row['mes_fuente'])-1],
            'Máximo mm': round(max_row['f1_precip'], 2),
            'Mes mínimo': MESES_NOMBRE[int(min_row['mes_fuente'])-1],
            'Mínimo mm': round(min_row['f1_precip'], 2)
        })

    tabla40 = pd.DataFrame(t40_rows)
    tabla40.to_csv(FIG_DIR / "Tabla40_Extremos_Precipitacion.csv", index=False)
    display(tabla40)

    # -------------------------------------------------------------------------
    # TABLA 41: Perfil mensual de precipitación y temperatura entre 2005 y 2025
    # -------------------------------------------------------------------------
    print("\n=== TABLA 41: Perfil mensual de precipitación y temperatura ===")
    # Calculamos la media y desviación estándar entre años para cada mes calendario
    perf_mensual = df_panel.groupby('mes_objetivo').agg(
        precip_media=('f1_precip', 'mean'),
        precip_std=('f1_precip', 'std'), # Desviación estándar entre todas las observaciones
        temp_media=('f1_temp', 'mean'),
        temp_std=('f1_temp', 'std')
    ).reset_index()

    # Nota: La desviación estándar "entre años" se puede interpretar como la std de las medias anuales
    # Para mayor precisión, calculamos las medias anuales por mes y luego la std de esas medias
    medias_anuales = df_panel.groupby(['anio_objetivo', 'mes_objetivo']).agg(
        precip_anual=('f1_precip', 'mean'),
        temp_anual=('f1_temp', 'mean')
    ).reset_index()

    perf_mensual_exacto = medias_anuales.groupby('mes_objetivo').agg(
        precip_media=('precip_anual', 'mean'),
        precip_std=('precip_anual', 'std'),
        temp_media=('temp_anual', 'mean'),
        temp_std=('temp_anual', 'std')
    ).reset_index()

    perf_mensual_exacto['Mes'] = perf_mensual_exacto['mes_objetivo'].apply(lambda x: MESES_NOMBRE[int(x)-1])

    tabla41 = perf_mensual_exacto[['Mes', 'precip_media', 'precip_std', 'temp_media', 'temp_std']].copy()
    tabla41.columns = ['Mes', 'Precipitación media mm', 'DesvEst entre años mm', 'Temperatura media °C', 'DesvEst entre años °C']

    tabla41.to_csv(FIG_DIR / "Tabla41_Perfil_Mensual.csv", index=False)
    display(tabla41.round(2))

    # -------------------------------------------------------------------------
    # TABLA 42: Métricas de las 12 configuraciones en la prueba final
    # -------------------------------------------------------------------------
    print("\n=== TABLA 42: Métricas de las 12 configuraciones en la prueba final ===")
    from sklearn.metrics import average_precision_score, roc_auc_score

    t42_rows = []
    for alg in algos:
        for fam in fams:
            col_p = f'prob_{alg}_{fam}'
            # Calcular métricas
            y_true = df_preds['Y_caida']
            y_prob = df_preds[col_p]

            # Brier Macro
            brier = brier_por_config(df_preds, col_p)

            # AP y ROC-AUC (usando todas las predicciones juntas, o macro si se prefiere)
            # La imagen muestra valores globales, así que usamos el pooled
            ap = average_precision_score(y_true, y_prob)
            roc = roc_auc_score(y_true, y_prob)

            t42_rows.append({
                'Algoritmo': alg,
                'Familia': fam,
                'Brier macro': brier,
                'AP': ap,
                'ROC-AUC': roc
            })

    tabla42 = pd.DataFrame(t42_rows)
    tabla42.to_csv(FIG_DIR / "Tabla42_Metricas_Prueba.csv", index=False)
    display(tabla42.round(6))

    # =========================================================================
    # TABLA 43: VARIACIÓN DEL BRIER SCORE POR FAMILIA
    # =========================================================================
    print("\n=== TABLA 43: Delta Brier por Familia ===")
    tabla43 = df_contrastes[
        (df_contrastes['Escenario'] == 'Panel 26 Jurisdicciones') &
        (df_contrastes['Familia_Algoritmica'] != 'RESUMEN MULTIMODELO')
    ].copy()
    tabla43_pivot = tabla43.pivot_table(
        index='Familia_Algoritmica', columns='Contraste',
        values='Delta_Brier',
    )
    tabla43_pivot.to_csv(FIG_DIR / "Tabla43_Delta_Brier_Por_Familia.csv")
    display(tabla43_pivot.round(6))

    # =========================================================================
    # TABLA 44: SENSIBILIDAD SIN LIMA METROPOLITANA
    # =========================================================================
    print("\n=== TABLA 44: Sensibilidad sin Lima Metropolitana ===")
    sens = df_contrastes[
        df_contrastes['Contraste'] == 'Aporte Total (F2-F0)'
    ].copy()
    pivot_sens = sens.pivot_table(
        index='Familia_Algoritmica', columns='Escenario',
        values='Delta_Brier',
    )
    pivot_sens['Cambio'] = (
        pivot_sens['Sensibilidad 25 Jurisdicciones'] -
        pivot_sens['Panel 26 Jurisdicciones']
    )
    pivot_sens['Signo_Mantenido'] = (
        np.sign(pivot_sens['Sensibilidad 25 Jurisdicciones']) ==
        np.sign(pivot_sens['Panel 26 Jurisdicciones'])
    )
    pivot_sens.to_csv(FIG_DIR / "Tabla44_Sensibilidad.csv")
    display(pivot_sens.round(6))

    # =========================================================================
    # TABLA 45: EVALUACIÓN DE HIPÓTESIS
    # =========================================================================
    print("\n=== TABLA 45: Evaluación de Hipótesis ===")
    display(df_hipotesis)

    log("Tablas y figuras de resultados completadas",
        celda="Celda11_Tablas_Resultados")

    return {
        'tabla36': tabla36,
        'tabla37': tabla37_pivot.reset_index(),
        'tabla43': tabla43_pivot.reset_index(),
        'tabla44': pivot_sens.reset_index(),
        'tabla45': df_hipotesis,
    }


# ==============================================================================
# EJECUCIÓN + REGISTRO DE HASHES
# ==============================================================================
tablas_resultados = generar_tablas_y_figuras(
    df_panel, panel_base, df_preds, df_res_oe2,
    df_contrastes, df_hipotesis, df_oni, df_icen,
)

for nombre, df_out in tablas_resultados.items():
    registrar_hash(f"Tabla_{nombre}", df=df_out)
for f in sorted(FIG_DIR.glob("Fig*.png")):
    registrar_hash(f"Grafico_{f.stem}", path=f)

# ==============================================================================
# RESUMEN FINAL
# ==============================================================================
print(f"\n{'='*70}")
print(f"PASO 11 COMPLETADO — TABLAS Y FIGURAS GENERADAS")
print(f"{'='*70}")
print(f"\nDirectorio: {FIG_DIR}")
print(f"\nFiguras PNG (9):")
for f in sorted(FIG_DIR.glob("Fig*.png")):
    print(f"  - {f.name}  ({f.stat().st_size // 1024} KB)")
print(f"\nTablas CSV (5):")
for f in sorted(FIG_DIR.glob("Tabla*.csv")):
    print(f"  - {f.name}")


[2026-10-30T03:39:01.733923+00:00] [INFO] [Celda11_Tablas_Resultados] INICIO generar_tablas_y_figuras
[2026-10-30T03:39:01.734941+00:00] [INFO] [Celda11_Tablas_Resultados] Iniciando tablas y figuras de resultados (OE2–OE4)

=== TABLA 35: Hiperparámetros finales seleccionados ===


,Familia,Conjunto,Candidato,Hiperparámetros finales,Brier validación
0,EN,F0,EN_F0_0005,C=0.01; l1_ratio=1.0,0.208492
1,EN,F1,EN_F1_0154,C=0.01; l1_ratio=1.0,0.208492
2,EN,F2,EN_F2_0303,C=0.01; l1_ratio=1.0,0.207372
3,LSTM,F0,LSTM_F0_0137,units=32; dense=16; dropout=0.2; lr=0.001; epo...,0.187742
4,LSTM,F1,LSTM_F1_0286,units=32; dense=16; dropout=0.2; lr=0.001; epo...,0.185737
5,LSTM,F2,LSTM_F2_0435,units=32; dense=16; dropout=0.2; lr=0.001; epo...,0.184851
6,RF,F0,RF_F0_0046,n_estimators=500; max_depth=16; min_samples_le...,0.202383
7,RF,F1,RF_F1_0201,n_estimators=500; max_depth=None; min_samples_...,0.201417
8,RF,F2,RF_F2_0368,n_estimators=1000; max_depth=None; min_samples...,0.200619
9,XGB,F0,XGB_F0_0072,n_estimators=300; max_depth=2; learning_rate=0...,0.205800



=== TABLA 36: Brier Score por Configuración y Bloque ===


,Algoritmo,Familia_Informacion,Brier_Entrenamiento,Modelo_Seleccion,Modelo_Prueba,Brier_Validacion,Brier_Prueba,Diferencia_Prueba_Menos_Validacion
0,EN,F0,0.195874,EN_F0_seleccion,EN_F0_reajuste,0.208492,0.201115,-0.007377
1,EN,F1,0.195874,EN_F1_seleccion,EN_F1_reajuste,0.208492,0.203069,-0.005423
2,EN,F2,0.195812,EN_F2_seleccion,EN_F2_reajuste,0.207372,0.197776,-0.009596
3,LSTM,F0,0.204774,LSTM_F0_seleccion,LSTM_F0_reajuste,0.187742,0.220928,0.033186
4,LSTM,F1,0.194265,LSTM_F1_seleccion,LSTM_F1_reajuste,0.185737,0.216514,0.030777
5,LSTM,F2,0.190604,LSTM_F2_seleccion,LSTM_F2_reajuste,0.184851,0.217186,0.032335
6,RF,F0,0.121185,RF_F0_seleccion,RF_F0_reajuste,0.202383,0.197661,-0.004722
7,RF,F1,0.120433,RF_F1_seleccion,RF_F1_reajuste,0.201417,0.202673,0.001255
8,RF,F2,0.119378,RF_F2_seleccion,RF_F2_reajuste,0.200619,0.205697,0.005078
9,XGB,F0,0.166062,XGB_F0_seleccion,XGB_F0_reajuste,0.205800,0.218415,0.012615



=== FIGURA 41: Brier Score Macro en Prueba ===
  ✓ Fig41_Brier_12_Configuraciones.png

=== FIGURA 42: Delta Brier por Familia y Comparación ===
  ✓ Fig42_Delta_Brier_Familia.png

=== FIGURA 43: Contrastes Multimodelo e IC95% ===
  ✓ Fig43_Contrastes_Multimodelo_IC95.png

=== FIGURA 44: Sensibilidad sin Lima Metropolitana ===
  ✓ Fig44_Sensibilidad_Sin_Lima.png

=== FIGURA 45: Recaudación Real Anual (2005-2025) ===
  ✓ Fig45_Recaudacion_Anual.png

=== FIGURA 46: Frecuencia Anual de Caídas ===
  ✓ Fig46_Caidas_Anuales.png

=== FIGURA 47: Comportamiento Mensual 2021-2025 ===
  ✓ Fig47_Caidas_Mensuales_2021_2025.png

=== FIGURA 48: Frecuencia de Caídas por Jurisdicción ===
  ✓ Fig48_Caidas_Jurisdiccion.png

=== FIGURA 49: ONI e ICEN Anual ===
  ✓ Fig49_ONI_ICEN_Anual.png

=== FIGURA 50: Precipitación Mensual Media 2021-2025 ===
  ✓ Fig50_Precipitacion_Mensual_2021_2025.png

=== FIGURA 10: Temperatura Mensual Media 2021-2025 ===
  ✓ Fig51_Temperatura_Mensual_2021_2025.png

=== TABLA 37: Br

...


=== TABLA 44: Sensibilidad sin Lima Metropolitana ===


Escenario,Panel 26 Jurisdicciones,Sensibilidad 25 Jurisdicciones,Cambio,Signo_Mantenido
Familia_Algoritmica,,,,
EN,-0.003339,-0.002924,0.000415,True
LSTM,-0.003742,-0.002570,0.001173,True
RESUMEN MULTIMODELO,-0.005451,-0.004909,0.000542,True
RF,0.008036,0.007951,-0.000085,True
XGB,-0.022757,-0.022091,0.000666,True



=== TABLA 45: Evaluación de Hipótesis ===


,Hipotesis,Delta_Brier,IC95,Mejora_Puntual,Veredicto
0,H1 (Total),-0.005451,"[-0.0289, 0.0100]",True,EVIDENCIA INSUFICIENTE
1,HE1 (Territorial),-0.000590,"[-0.0093, 0.0068]",True,EVIDENCIA INSUFICIENTE
2,HE2 (ENOS),-0.004861,"[-0.0211, 0.0063]",True,EVIDENCIA INSUFICIENTE
3,HE3 (Robustez),-0.004909,"Alg. Neg: 3/4 | IC Sin Lima: [-0.0275, 0.0099]",NaN,CUMPLE


[2026-10-30T03:39:15.406235+00:00] [INFO] [Celda11_Tablas_Resultados] Tablas y figuras de resultados completadas
[2026-10-30T03:39:15.415654+00:00] [INFO] [FLAG] FLAG creado: Celda11_Tablas_Resultados.OK.flag
[2026-10-30T03:39:15.415774+00:00] [INFO] [Celda11_Tablas_Resultados] FIN generar_tablas_y_figuras — 13.68s
[2026-10-30T03:39:15.417379+00:00] [INFO] [HASH] HASH registrado: Tabla_tabla36 = 83127025385014db...
[2026-10-30T03:39:15.423421+00:00] [INFO] [HASH] HASH registrado: Tabla_tabla37 = 3c7e936679d14736...
[2026-10-30T03:39:15.426565+00:00] [INFO] [HASH] HASH registrado: Tabla_tabla43 = e90e5c494856eb7a...
[2026-10-30T03:39:15.429370+00:00] [INFO] [HASH] HASH registrado: Tabla_tabla44 = b789c3e6393ede4a...
[2026-10-30T03:39:15.432106+00:00] [INFO] [HASH] HASH registrado: Tabla_tabla45 = 7384f0d1438fd8ad...
[2026-10-30T03:39:15.436277+00:00] [INFO] [HASH] HASH registrado: Grafico_Fig41_Brier_12_Configuraciones = 5a5d1a01b38c96eb...
[2026-10-30T03:39:15.439944+00:00] [INFO] [HAS

## Paso 12 — Empaquetado y descarga del proyecto

Cierra la corrida y prepara la entrega final:

1. **Empaqueta todo** (`resultados_tesis/` + `00_AUDITORIA/<RUN_ID>/` + notebook) en un único ZIP.
2. **Crea el flag** `Celda12_Empaquetado.OK.flag` con el hash SHA-256 y el tamaño del ZIP.
3. **Escribe el manifiesto global** (`manifiesto.json`) que incluye:
   - Entorno (versiones de paquetes).
   - Todos los hashes de artefactos.
   - Todos los flags (incluido el del paso 12).
   - Errores registrados (si hubo).
   - Resumen de hipótesis, contrastes y tablas generadas.
   - Hash y tamaño del ZIP final.
4. **Descarga el ZIP** a la máquina local del analista.

Este paso garantiza que toda la corrida sea **reproducible y auditable**: cualquier tercero puede reconstruir el análisis y verificar cada hash.

**Control V8:** Manifiesto y estado PREPARADO antes del ZIP; comprobación y hash final fuera del ZIP. Se exige copia del notebook realmente ejecutado.


**Compatibilidad Colab:** si no hay copia del notebook en la ruta configurada, se intenta recuperar el documento real de la interfaz. Si la interfaz no lo permite, se solicita subir su descarga `.ipynb`.


In [15]:
# ==============================================================================
# 12: EMPAQUETADO Y DESCARGA (MANIFIESTO + ZIP + DESCARGA)
# ==============================================================================
# Cierra la corrida:
#   1. Empaqueta TODO (resultados + auditoría + notebook) en un ZIP
#   2. Crea el flag Celda12_Empaquetado.OK con hash y tamaño del ZIP
#   3. Escribe el manifiesto global (entorno, hashes, flags, errores, log)
#   4. Descarga el ZIP a la máquina local
#
# El flag se crea ANTES de escribir el manifiesto para que este último
# lo incluya en manifiesto["flags"]. El ZIP se construye antes del flag
# para poder hashearlo en la metadata.

import shutil
import zipfile

try:
    # -------------------------------------------------------------------------
    # 1. Construir el ZIP con TODO el contenido de OUTPUT_DIR
    # -------------------------------------------------------------------------
    zip_name = f"/content/Resultados_Tesis_Final_{RUN_ID}"
    shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)

    # Añadir el notebook (si existe en /content) al ZIP
    with zipfile.ZipFile(f"{zip_name}.zip", 'a') as zf:
        for extra in Path("/content").glob("*.ipynb"):
            zf.write(extra, extra.name)

    zip_sha = sha256_file(f"{zip_name}.zip")
    zip_bytes = Path(f"{zip_name}.zip").stat().st_size

    # -------------------------------------------------------------------------
    # 2. Crear el flag del paso 12 ANTES del manifiesto
    #    (así el manifiesto lo incluirá en manifiesto["flags"])
    # -------------------------------------------------------------------------
    crear_flag("Celda12_Empaquetado", "OK",
               {"zip": f"{zip_name}.zip",
                "sha256_zip": zip_sha,
                "bytes_zip": zip_bytes})

    # -------------------------------------------------------------------------
    # 3. Escribir el manifiesto global
    # -------------------------------------------------------------------------
    manifiesto = escribir_manifiesto(secciones_extra={
        "hipotesis": df_hipotesis.to_dict(orient="records"),
        "contrastes_resumen": df_contrastes[
            df_contrastes['Familia_Algoritmica'] == 'RESUMEN MULTIMODELO'
        ].to_dict(orient="records"),
        "tablas_generadas": list(tablas_resultados.keys()),
        "zip_final": {
            "path": f"{zip_name}.zip",
            "sha256": zip_sha,
            "bytes": zip_bytes,
        },
    })

    # -------------------------------------------------------------------------
    # 4. Descargar el ZIP
    # -------------------------------------------------------------------------
    print(f"\n{'='*70}")
    print(f"PASO 12 COMPLETADO — EMPAQUETADO Y DESCARGA")
    print(f"{'='*70}")
    print(f"ZIP         : {zip_name}.zip")
    print(f"SHA-256 ZIP : {zip_sha[:32]}...")
    print(f"Bytes ZIP   : {zip_bytes:,}")
    print(f"Manifiesto  : {MANIFEST_FILE}")
    print(f"Auditoría   : {AUDIT_DIR}")
    print(f"\nRUN_ID      : {RUN_ID}")
    print(f"\n>>> Descargando ZIP... <<<")

    from google.colab import files
    files.download(f"{zip_name}.zip")

except Exception as e:
    registrar_error("Celda12_Empaquetado", e)
    raise

[2026-10-30T03:39:57.789954+00:00] [INFO] [FLAG] FLAG creado: Celda12_Empaquetado.OK.flag

PASO 12 COMPLETADO — EMPAQUETADO Y DESCARGA
ZIP         : /content/Resultados_Tesis_Final_20261030T002900_112193Z.zip
SHA-256 ZIP : effdbc36fc9ffdfb6fda71a1c9fa2cc4...
Bytes ZIP   : 54,116,136
Manifiesto  : /content/resultados_tesis/20261030T002900_112193Z/00_AUDITORIA/20261030T002900_112193Z/manifiesto.json
Auditoría   : /content/resultados_tesis/20261030T002900_112193Z/00_AUDITORIA/20261030T002900_112193Z

RUN_ID      : 20261030T002900_112193Z

>>> Descargando ZIP... <<<


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>